<div dir="rtl" align="right">

# Notebook 16 — Hybrid Retrieval

این Notebook مسئول ترکیب Dense Retrieval فریز‌شده NB14 و Sparse Retrieval فریز‌شده NB15 است.

هدف، انتخاب یک روش Hybrid Retrieval قابل دفاع، قابل بازسازی و نسخه‌دار است که هم شباهت معنایی سؤال و هم تطبیق واژگان، نام‌ها، اعداد و اصطلاحات دقیق را پوشش دهد.

ورودی‌های اصلی این Notebook عبارت‌اند از:

- Dense Index Release و Handoff فریز‌شده NB14؛
- Sparse Index Release و Handoff فریز‌شده NB15؛
- Canonical Chunk Release شامل ۳۴۸۸ Chunk؛
- Golden Dataset V2 شامل ۳۸۱۳ Query؛
- Gold Mapping و Split Contract بازسازی‌شده و منطبق با NB15؛
- Embedding Model فریز‌شده `intfloat/multilingual-e5-small`.

این Notebook موارد زیر را اجرا خواهد کرد:

1. اعتبارسنجی کامل Artifactها و Hashهای NB14 و NB15؛
2. بازسازی Development/Test Split بدون نشت؛
3. اجرای Dense Baseline؛
4. اجرای Sparse Baseline؛
5. آزمایش Reciprocal Rank Fusion یا RRF؛
6. آزمایش Weighted Fusion؛
7. آزمایش Candidate Depth و Top-k؛
8. Metadata Filtering؛
9. Source Diversity Control؛
10. Slice Metrics و Error Analysis؛
11. Ablation میان Dense، Sparse و Hybrid؛
12. انتخاب پیکربندی نهایی با ADR؛
13. ارزیابی مستقل روی Test Split قفل‌شده؛
14. ساخت Release و Handoff نسخه‌دار برای NB17.

Reranking، Query Rewrite، Context Building و Generation خارج از Scope این Notebook هستند و در Notebookهای بعدی اجرا خواهند شد.

هیچ‌یک از Dense Index یا Sparse Indexهای فریز‌شده در این Notebook تغییر نخواهند کرد.

</div>

<div dir="rtl" align="right">

## قرارداد اجرا و برنامه مرحله‌ای NB16

اجرای NB16 با سیاست Artifact-first و بدون وابستگی پنهان به Kernel نوت‌بوک‌های قبلی انجام می‌شود.

### قرارداد ورودی

ورودی‌های Canonical باید پیش از شروع آزمایش‌ها Load و Validate شوند:

- `artifacts/phase2/dense_index/dense_index_release_v1.json`
- `artifacts/phase2/dense_index/nb16_dense_index_handoff_v1.json`
- `artifacts/phase2/sparse_index/sparse_index_release_v1.json`
- `artifacts/phase2/sparse_index/nb16_sparse_index_handoff_v1.json`
- `data/phase2/index/sparse_index_v1/sparse_index_manifest.json`
- `data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl`
- `artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl`

تمام File SHAها، Logical Index SHAها، Configuration SHAها، Record Countها و Freeze Statusها باید پیش از مصرف بررسی شوند.

### سیاست جلوگیری از نشت

همان Split قطعی NB15 باید با Policy، Seed و Hash یکسان بازسازی شود:

- Development: انتخاب Candidate Depth، RRF و Weighted Fusion؛
- Test: قفل تا زمان فریزشدن پیکربندی Hybrid و ADR؛
- تغییر پیکربندی براساس نتیجه Test ممنوع است.

### Experiment Matrix

آزمایش‌ها حداقل شامل این خانواده‌ها خواهند بود:

1. Dense-only؛
2. Sparse-only؛
3. RRF با پارامترهای ازپیش‌تعریف‌شده؛
4. Weighted Fusion با نرمال‌سازی امتیاز صریح؛
5. Candidate Depthهای مختلف؛
6. Top-kهای مختلف؛
7. Metadata Filtering؛
8. Source Diversity Control؛
9. Ablation و Slice Analysis.

### معیارهای ارزیابی

- Hit@5 و Hit@10
- Recall@5 و Recall@10
- Precision@5 و Precision@10
- MRR
- nDCG@5 و nDCG@10
- Source coverage و Source diversity
- p50 و p95 Latency محلی
- Failure taxonomy
- بهبود Hybrid نسبت به بهترین Baseline

### قانون انتخاب

روش نهایی فقط در صورتی پذیرفته می‌شود که Experiment Matrix و ADR نشان دهند افزایش پیچیدگی Hybrid نسبت به Baselineها بهبود قابل دفاع ایجاد کرده است.

اگر Hybrid بهبود معناداری نداشته باشد، پیچیدگی اضافی پذیرفته نمی‌شود و این نتیجه باید صریحاً در ADR ثبت شود.

### قرارداد خروجی

خروجی نهایی باید شامل موارد زیر باشد:

- Hybrid configuration freeze؛
- Experiment matrix؛
- Ablation report؛
- Independent Test evaluation؛
- Error analysis؛
- Hybrid manifest و release؛
- Handoff نسخه‌دار برای NB17؛
- Final Health Check؛
- توضیح نهایی درباره تحقق یا عدم تحقق هدف Notebook.

تمام APIهای پولی غیرفعال هستند و Query Embeddingها باید با مدل محلی فریز‌شده و Cache نسخه‌دار تولید شوند.

</div>

In [1]:
# Code Cell 01 — Imports and runtime bootstrap only

import ast
import collections
import copy
import dataclasses
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import statistics
import sys
import time
import unicodedata

from pathlib import Path

import numpy as np
import pandas as pd
import scipy

from qdrant_client import QdrantClient
from qdrant_client import models as qdrant_models
from sentence_transformers import SentenceTransformer

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Code Cell 02 — Project-root discovery and read-only preflight

EXPECTED_PYTHON_MAJOR = 3
EXPECTED_PYTHON_MINOR = 11

DENSE_RELEASE_RELATIVE_PATH = Path(
    "artifacts/phase2/dense_index/"
    "dense_index_release_v1.json"
)
DENSE_HANDOFF_RELATIVE_PATH = Path(
    "artifacts/phase2/dense_index/"
    "nb16_dense_index_handoff_v1.json"
)
SPARSE_RELEASE_RELATIVE_PATH = Path(
    "artifacts/phase2/sparse_index/"
    "sparse_index_release_v1.json"
)
SPARSE_HANDOFF_RELATIVE_PATH = Path(
    "artifacts/phase2/sparse_index/"
    "nb16_sparse_index_handoff_v1.json"
)
SPARSE_MANIFEST_RELATIVE_PATH = Path(
    "data/phase2/index/sparse_index_v1/"
    "sparse_index_manifest.json"
)
CHUNK_RELATIVE_PATH = Path(
    "data/phase2/chunks/nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)
GOLDEN_RELATIVE_PATH = Path(
    "artifacts/phase2/golden_dataset_v2/"
    "golden_dataset_v2.jsonl"
)
NB16_RELATIVE_PATH = Path(
    "notebooks/phase2/"
    "16_hybrid_retrieval.ipynb"
)

EXPECTED_DENSE_RELEASE_SHA256 = (
    "706c020caa2bda77c767fda8cd1b5749"
    "8818afdbdbbbdd2a72f898b55ca6ca50"
)
EXPECTED_SPARSE_RELEASE_SHA256 = (
    "879f52850a71629f02437c1ab1173e3f"
    "9e6c55ec5441ad4649784be63935c51b"
)
EXPECTED_SPARSE_HANDOFF_SHA256 = (
    "e4070684ba1662de35e2fbdf2737560a"
    "ee1b7c67a694e6dab3bd98be2f578439"
)
EXPECTED_SPARSE_MANIFEST_SHA256 = (
    "ce8192a64e555b00c2708ecde3885da8"
    "70bb2e6725640c6fd194f885300e8fac"
)
EXPECTED_CHUNK_SHA256 = (
    "3891c4ecad9332ba753cd605bbccd99da"
    "9d063f64934a30ee39e495349cffb50"
)
EXPECTED_GOLDEN_SHA256 = (
    "b9ca18e710686ddca48a25c637840b716"
    "adb306717cafd8ec7c96ce9e182e6f2"
)

current_path = Path.cwd().resolve()

root_candidates = []

for base_path in (
    current_path,
    *current_path.parents,
):
    root_candidates.append(base_path)
    root_candidates.append(
        base_path / "RAG_finance"
    )

PROJECT_ROOT = None

for candidate_root in root_candidates:
    if (
        (
            candidate_root
            / DENSE_RELEASE_RELATIVE_PATH
        ).is_file()
        and (
            candidate_root
            / SPARSE_RELEASE_RELATIVE_PATH
        ).is_file()
        and (
            candidate_root
            / CHUNK_RELATIVE_PATH
        ).is_file()
        and (
            candidate_root
            / GOLDEN_RELATIVE_PATH
        ).is_file()
    ):
        PROJECT_ROOT = candidate_root.resolve()
        break

assert PROJECT_ROOT is not None, (
    "Canonical RAG_finance project root "
    "could not be discovered."
)

DENSE_RELEASE_PATH = (
    PROJECT_ROOT
    / DENSE_RELEASE_RELATIVE_PATH
)
DENSE_HANDOFF_PATH = (
    PROJECT_ROOT
    / DENSE_HANDOFF_RELATIVE_PATH
)
SPARSE_RELEASE_PATH = (
    PROJECT_ROOT
    / SPARSE_RELEASE_RELATIVE_PATH
)
SPARSE_HANDOFF_PATH = (
    PROJECT_ROOT
    / SPARSE_HANDOFF_RELATIVE_PATH
)
SPARSE_MANIFEST_PATH = (
    PROJECT_ROOT
    / SPARSE_MANIFEST_RELATIVE_PATH
)
CHUNK_PATH = (
    PROJECT_ROOT
    / CHUNK_RELATIVE_PATH
)
GOLDEN_PATH = (
    PROJECT_ROOT
    / GOLDEN_RELATIVE_PATH
)
NB16_PATH = (
    PROJECT_ROOT
    / NB16_RELATIVE_PATH
)

required_paths = {
    "dense_release": DENSE_RELEASE_PATH,
    "dense_handoff": DENSE_HANDOFF_PATH,
    "sparse_release": SPARSE_RELEASE_PATH,
    "sparse_handoff": SPARSE_HANDOFF_PATH,
    "sparse_manifest": SPARSE_MANIFEST_PATH,
    "canonical_chunks": CHUNK_PATH,
    "golden_dataset": GOLDEN_PATH,
    "nb16_notebook": NB16_PATH,
}

missing_required_paths = {
    artifact_name: str(artifact_path)
    for artifact_name, artifact_path
    in required_paths.items()
    if not artifact_path.is_file()
}

assert not missing_required_paths, (
    "Missing required NB16 inputs: "
    f"{missing_required_paths}"
)


def preflight_sha256_file(file_path):
    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as file_handle:
        for block in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


preflight_hashes = {
    "dense_release": (
        preflight_sha256_file(
            DENSE_RELEASE_PATH
        )
    ),
    "sparse_release": (
        preflight_sha256_file(
            SPARSE_RELEASE_PATH
        )
    ),
    "sparse_handoff": (
        preflight_sha256_file(
            SPARSE_HANDOFF_PATH
        )
    ),
    "sparse_manifest": (
        preflight_sha256_file(
            SPARSE_MANIFEST_PATH
        )
    ),
    "canonical_chunks": (
        preflight_sha256_file(
            CHUNK_PATH
        )
    ),
    "golden_dataset": (
        preflight_sha256_file(
            GOLDEN_PATH
        )
    ),
}

expected_preflight_hashes = {
    "dense_release": (
        EXPECTED_DENSE_RELEASE_SHA256
    ),
    "sparse_release": (
        EXPECTED_SPARSE_RELEASE_SHA256
    ),
    "sparse_handoff": (
        EXPECTED_SPARSE_HANDOFF_SHA256
    ),
    "sparse_manifest": (
        EXPECTED_SPARSE_MANIFEST_SHA256
    ),
    "canonical_chunks": (
        EXPECTED_CHUNK_SHA256
    ),
    "golden_dataset": (
        EXPECTED_GOLDEN_SHA256
    ),
}

hash_mismatches = {
    artifact_name: {
        "expected": (
            expected_preflight_hashes[
                artifact_name
            ]
        ),
        "actual": (
            preflight_hashes[
                artifact_name
            ]
        ),
    }
    for artifact_name
    in expected_preflight_hashes
    if (
        preflight_hashes[artifact_name]
        != expected_preflight_hashes[
            artifact_name
        ]
    )
}

assert not hash_mismatches, (
    "NB16 preflight hash mismatch: "
    f"{hash_mismatches}"
)

assert sys.version_info.major == (
    EXPECTED_PYTHON_MAJOR
)
assert sys.version_info.minor == (
    EXPECTED_PYTHON_MINOR
)

package_versions = {
    package_name: (
        importlib.metadata.version(
            package_name
        )
    )
    for package_name in (
        "numpy",
        "pandas",
        "scipy",
        "qdrant-client",
        "sentence-transformers",
    )
}

nb16_preflight = {
    "notebook": (
        "16_hybrid_retrieval.ipynb"
    ),
    "project_root": str(PROJECT_ROOT),
    "python": {
        "version": platform.python_version(),
        "implementation": (
            platform.python_implementation()
        ),
        "platform": platform.platform(),
    },
    "package_versions": package_versions,
    "required_input_count": len(
        required_paths
    ),
    "missing_required_paths": (
        missing_required_paths
    ),
    "validated_hashes": (
        preflight_hashes
    ),
    "repository_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "preflight_status": "PASS",
}

print(
    json.dumps(
        nb16_preflight,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "notebook": "16_hybrid_retrieval.ipynb",
  "project_root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance",
  "python": {
    "version": "3.11.3",
    "implementation": "CPython",
    "platform": "macOS-12.7.6-x86_64-i386-64bit"
  },
  "package_versions": {
    "numpy": "1.24.3",
    "pandas": "1.5.3",
    "scipy": "1.13.1",
    "qdrant-client": "1.18.0",
    "sentence-transformers": "5.0.0"
  },
  "required_input_count": 8,
  "missing_required_paths": {},
  "validated_hashes": {
    "dense_release": "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
    "sparse_release": "879f52850a71629f02437c1ab1173e3f9e6c55ec5441ad4649784be63935c51b",
    "sparse_handoff": "e4070684ba1662de35e2fbdf2737560aee1b7c67a694e6dab3bd98be2f578439",
    "sparse_manifest": "ce8192a64e555b00c2708ecde3885da870bb2e6725640c6fd194f885300e8fac",
    "canonical_chunks": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    "golden_dataset": "b9ca18e7

<div dir="rtl" align="right">

## بارگذاری و اعتبارسنجی قراردادهای Dense، Sparse و Golden Dataset

در این مرحله تمام قراردادهای نسخه‌دار NB14 و NB15 به‌صورت Read-only بارگذاری می‌شوند.

کنترل‌ها شامل موارد زیر هستند:

1. وضعیت Frozen هر دو Release؛
2. وضعیت آمادگی Handoffها برای NB16؛
3. برابری SHAهای Manifest، Release و فایل‌های مصرفی؛
4. برابری تعداد ۳۴۸۸ Document در Dense و Sparse؛
5. برابری Logical Index Hashها با مقادیر فریز‌شده؛
6. صحت Configuration SHA مربوط به Sparse Retrieval؛
7. اعتبار Canonical Chunk File و Golden Dataset V2؛
8. یکتایی Chunk ID و Candidate ID؛
9. کنترل پوشش سه نوع منبع Website، Book و Transcript؛
10. اثبات اینکه هیچ Index یا Artifact موجودی در این مرحله تغییر نمی‌کند.

خروجی این سل، قرارداد ورودی رسمی NB16 خواهد بود. تا زمانی که این قرارداد `PASS` نشود، اجرای Dense، Sparse یا Fusion مجاز نیست.

</div>

In [3]:
# Code Cell 03 — Load and validate NB16 canonical input contracts

EXPECTED_DENSE_LOGICAL_INDEX_SHA256 = (
    "2fc5e50ebbd1e1e66d59c703eb48f136"
    "7b2d3deadfed62efb9d5a5e6959f7ef4"
)
EXPECTED_SPARSE_LOGICAL_INDEX_SHA256 = (
    "ce09bb4749da03c9dfa1a849de18fc8c"
    "490ce6b09eb34a30a96b9fedc478a8eb"
)
EXPECTED_SPARSE_CONFIGURATION_SHA256 = (
    "96d66676d20bacd764ec9a163995dd482"
    "66c90c2acea1c8a5f77d95d9efb6ecc"
)
EXPECTED_DOCUMENT_COUNT = 3488
EXPECTED_GOLDEN_QUERY_COUNT = 3813

dense_release = json.loads(
    DENSE_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

dense_handoff = json.loads(
    DENSE_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

sparse_release = json.loads(
    SPARSE_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

sparse_handoff = json.loads(
    SPARSE_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

sparse_manifest = json.loads(
    SPARSE_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


def load_jsonl_records(file_path):
    records = []

    with Path(file_path).open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        for line_number, line in enumerate(
            file_handle,
            start=1,
        ):
            stripped_line = line.strip()

            assert stripped_line, (
                f"Empty JSONL line at "
                f"{file_path}:{line_number}"
            )

            records.append(
                json.loads(stripped_line)
            )

    return records


def contract_status(
    contract,
):
    for status_key in (
        "status",
        "handoff_status",
        "final_status",
        "manifest_status",
    ):
        if status_key in contract:
            return contract[status_key]

    return None


def resolve_project_artifact(
    path_value,
):
    artifact_path = Path(path_value)

    assert not artifact_path.is_absolute(), (
        "Artifact contract contains a "
        f"non-portable absolute path: {path_value}"
    )

    resolved_path = (
        PROJECT_ROOT / artifact_path
    ).resolve()

    assert (
        PROJECT_ROOT == resolved_path
        or PROJECT_ROOT
        in resolved_path.parents
    ), (
        "Artifact path escapes project root: "
        f"{path_value}"
    )

    return resolved_path


dense_release_status = dense_release.get(
    "status"
)
dense_release_final_status = (
    dense_release.get("final_status")
)

dense_handoff_status = (
    dense_handoff.get(
        "status",
        dense_handoff.get(
            "handoff_status"
        ),
    )
)

sparse_release_status = sparse_release.get(
    "status"
)
sparse_release_final_status = (
    sparse_release.get("final_status")
)

sparse_handoff_status = (
    sparse_handoff.get(
        "handoff_status",
        sparse_handoff.get("status"),
    )
)

assert dense_release_status == (
    "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
)
assert dense_release_final_status == (
    "PASS_COMPLETE_FROZEN"
)
assert dense_handoff_status == (
    "READY_FOR_NB16"
)

assert sparse_release_status == (
    "FROZEN_FOR_HYBRID_RETRIEVAL"
)
assert sparse_release_final_status == (
    "PASS_COMPLETE_FROZEN"
)
assert sparse_handoff_status == (
    "READY_FOR_NB16"
)

assert (
    dense_release["point_count"]
    == EXPECTED_DOCUMENT_COUNT
)
assert (
    sparse_release["document_count"]
    == EXPECTED_DOCUMENT_COUNT
)
assert (
    sparse_manifest[
        "index_statistics"
    ]["document_count"]
    == EXPECTED_DOCUMENT_COUNT
)

assert (
    dense_release[
        "logical_index_sha256"
    ]
    == EXPECTED_DENSE_LOGICAL_INDEX_SHA256
)
assert (
    sparse_release[
        "logical_index_sha256"
    ]
    == EXPECTED_SPARSE_LOGICAL_INDEX_SHA256
)
assert (
    sparse_manifest[
        "logical_index_sha256"
    ]
    == EXPECTED_SPARSE_LOGICAL_INDEX_SHA256
)

assert (
    sparse_release[
        "configuration_sha256"
    ]
    == EXPECTED_SPARSE_CONFIGURATION_SHA256
)
assert (
    sparse_handoff[
        "configuration_sha256"
    ]
    == EXPECTED_SPARSE_CONFIGURATION_SHA256
)
assert (
    sparse_manifest[
        "configuration"
    ]["configuration_sha256"]
    == EXPECTED_SPARSE_CONFIGURATION_SHA256
)

assert (
    sparse_release[
        "manifest"
    ]["sha256"]
    == EXPECTED_SPARSE_MANIFEST_SHA256
)
assert (
    sparse_handoff[
        "manifest"
    ]["sha256"]
    == EXPECTED_SPARSE_MANIFEST_SHA256
)
assert (
    sparse_handoff[
        "sparse_release"
    ]["sha256"]
    == EXPECTED_SPARSE_RELEASE_SHA256
)

assert (
    sparse_handoff[
        "dense_pairing_contract"
    ]["dense_release_sha256"]
    == EXPECTED_DENSE_RELEASE_SHA256
)
assert (
    sparse_handoff[
        "dense_pairing_contract"
    ]["dense_logical_index_sha256"]
    == EXPECTED_DENSE_LOGICAL_INDEX_SHA256
)
assert (
    sparse_handoff[
        "dense_pairing_contract"
    ]["shared_document_count"]
    == EXPECTED_DOCUMENT_COUNT
)

sparse_documents_path = (
    resolve_project_artifact(
        sparse_handoff[
            "loader_contract"
        ]["documents_path"]
    )
)
sparse_postings_path = (
    resolve_project_artifact(
        sparse_handoff[
            "loader_contract"
        ]["postings_path"]
    )
)
sparse_configuration_path = (
    resolve_project_artifact(
        sparse_handoff[
            "loader_contract"
        ]["configuration_path"]
    )
)

assert sparse_documents_path.is_file()
assert sparse_postings_path.is_file()
assert sparse_configuration_path.is_file()

assert (
    preflight_sha256_file(
        sparse_documents_path
    )
    == sparse_handoff[
        "loader_contract"
    ]["documents_sha256"]
)
assert (
    preflight_sha256_file(
        sparse_postings_path
    )
    == sparse_handoff[
        "loader_contract"
    ]["postings_sha256"]
)
assert (
    preflight_sha256_file(
        sparse_configuration_path
    )
    == sparse_handoff[
        "loader_contract"
    ]["configuration_file_sha256"]
)

assert (
    sparse_handoff[
        "loader_contract"
    ]["analyzer_id"]
    == "fa_join_zwnj_stopwords_v1"
)
assert (
    sparse_handoff[
        "loader_contract"
    ]["bm25_implementation"]
    == "local_robertson_bm25_v1"
)
assert float(
    sparse_handoff[
        "loader_contract"
    ]["k1"]
) == 1.2
assert float(
    sparse_handoff[
        "loader_contract"
    ]["b"]
) == 0.3

chunk_records = load_jsonl_records(
    CHUNK_PATH
)
golden_records = load_jsonl_records(
    GOLDEN_PATH
)

assert len(
    chunk_records
) == EXPECTED_DOCUMENT_COUNT
assert len(
    golden_records
) == EXPECTED_GOLDEN_QUERY_COUNT

required_chunk_fields = {
    "chunk_id",
    "content",
    "source_type",
    "source_id",
    "member_locators",
}

required_golden_fields = {
    "candidate_id",
    "claim_id",
    "evaluation_content",
    "evidence_locators",
    "source_types",
}

assert all(
    required_chunk_fields.issubset(
        record
    )
    for record in chunk_records
)
assert all(
    required_golden_fields.issubset(
        record
    )
    for record in golden_records
)

chunk_ids = [
    record["chunk_id"]
    for record in chunk_records
]
golden_candidate_ids = [
    record["candidate_id"]
    for record in golden_records
]

assert len(chunk_ids) == len(
    set(chunk_ids)
)
assert len(
    golden_candidate_ids
) == len(
    set(golden_candidate_ids)
)

assert all(
    isinstance(
        record["content"],
        str,
    )
    and record["content"].strip()
    for record in chunk_records
)

assert all(
    isinstance(
        record["evaluation_content"],
        dict,
    )
    and isinstance(
        record[
            "evaluation_content"
        ].get("question"),
        str,
    )
    and record[
        "evaluation_content"
    ]["question"].strip()
    for record in golden_records
)

chunk_source_type_counts = dict(
    sorted(
        collections.Counter(
            record["source_type"]
            for record in chunk_records
        ).items()
    )
)

golden_source_type_counts = (
    collections.Counter()
)

for record in golden_records:
    for source_type in set(
        record["source_types"]
    ):
        golden_source_type_counts[
            source_type
        ] += 1

golden_source_type_counts = dict(
    sorted(
        golden_source_type_counts.items()
    )
)

assert set(
    chunk_source_type_counts
) == {
    "book",
    "transcript",
    "website",
}
assert set(
    golden_source_type_counts
) == {
    "book",
    "transcript",
    "website",
}

dense_handoff_sha256 = (
    preflight_sha256_file(
        DENSE_HANDOFF_PATH
    )
)

nb16_canonical_input_contract = {
    "artifact_type": (
        "NB16_CANONICAL_INPUT_CONTRACT"
    ),
    "schema_version": "1.0",
    "dense": {
        "release_path": str(
            DENSE_RELEASE_RELATIVE_PATH
        ),
        "release_sha256": (
            EXPECTED_DENSE_RELEASE_SHA256
        ),
        "handoff_path": str(
            DENSE_HANDOFF_RELATIVE_PATH
        ),
        "handoff_sha256": (
            dense_handoff_sha256
        ),
        "release_status": (
            dense_release_status
        ),
        "handoff_status": (
            dense_handoff_status
        ),
        "logical_index_sha256": (
            EXPECTED_DENSE_LOGICAL_INDEX_SHA256
        ),
        "document_count": (
            dense_release["point_count"]
        ),
        "mutation_allowed": False,
    },
    "sparse": {
        "release_path": str(
            SPARSE_RELEASE_RELATIVE_PATH
        ),
        "release_sha256": (
            EXPECTED_SPARSE_RELEASE_SHA256
        ),
        "handoff_path": str(
            SPARSE_HANDOFF_RELATIVE_PATH
        ),
        "handoff_sha256": (
            EXPECTED_SPARSE_HANDOFF_SHA256
        ),
        "manifest_path": str(
            SPARSE_MANIFEST_RELATIVE_PATH
        ),
        "manifest_sha256": (
            EXPECTED_SPARSE_MANIFEST_SHA256
        ),
        "release_status": (
            sparse_release_status
        ),
        "handoff_status": (
            sparse_handoff_status
        ),
        "logical_index_sha256": (
            EXPECTED_SPARSE_LOGICAL_INDEX_SHA256
        ),
        "configuration_sha256": (
            EXPECTED_SPARSE_CONFIGURATION_SHA256
        ),
        "document_count": (
            sparse_release[
                "document_count"
            ]
        ),
        "mutation_allowed": False,
    },
    "canonical_corpus": {
        "path": str(
            CHUNK_RELATIVE_PATH
        ),
        "sha256": EXPECTED_CHUNK_SHA256,
        "record_count": len(
            chunk_records
        ),
        "source_type_counts": (
            chunk_source_type_counts
        ),
    },
    "golden_dataset": {
        "path": str(
            GOLDEN_RELATIVE_PATH
        ),
        "sha256": EXPECTED_GOLDEN_SHA256,
        "record_count": len(
            golden_records
        ),
        "source_type_counts": (
            golden_source_type_counts
        ),
        "freeze_status": "FROZEN",
    },
    "shared_document_identity": {
        "document_count": (
            EXPECTED_DOCUMENT_COUNT
        ),
        "chunk_id_uniqueness": "PASS",
        "dense_sparse_count_parity": (
            "PASS"
        ),
    },
    "repository_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "input_contract_status": "PASS",
}

print(
    json.dumps(
        nb16_canonical_input_contract,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "artifact_type": "NB16_CANONICAL_INPUT_CONTRACT",
  "schema_version": "1.0",
  "dense": {
    "release_path": "artifacts/phase2/dense_index/dense_index_release_v1.json",
    "release_sha256": "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
    "handoff_path": "artifacts/phase2/dense_index/nb16_dense_index_handoff_v1.json",
    "handoff_sha256": "da7f812a79a21b1876c74668a548badb0f271b2d18ac2f1b2cf9663f789f301f",
    "release_status": "FROZEN_FOR_DOWNSTREAM_RETRIEVAL",
    "handoff_status": "READY_FOR_NB16",
    "logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
    "document_count": 3488,
    "mutation_allowed": false
  },
  "sparse": {
    "release_path": "artifacts/phase2/sparse_index/sparse_index_release_v1.json",
    "release_sha256": "879f52850a71629f02437c1ab1173e3f9e6c55ec5441ad4649784be63935c51b",
    "handoff_path": "artifacts/phase2/sparse_index/nb16_sparse_index_handoff_v1.json",
    "handoff_sha256": "e4070684b

<div dir="rtl" align="right">

## بازسازی Gold Mapping برای ارزیابی Retrieval

Golden Dataset به‌جای اشاره مستقیم به Chunk ID، Evidence Locatorهای منبع را نگهداری می‌کند. برای ارزیابی Dense، Sparse و Hybrid باید این Locatorها به Chunkهای Canonical متصل شوند.

روش Mapping همان قرارداد تثبیت‌شده در NB15 است:

1. Canonical Serialization برای Locatorهای ساخت‌یافته؛
2. استخراج Locatorهای عضو هر Chunk؛
3. ساخت Reverse Index از Locator به Chunk ID؛
4. اتصال هر Evidence Locator به تمام Chunkهای منطبق؛
5. ساخت مجموعه Relevant Chunkها برای هر Query؛
6. کنترل کامل پوشش ۳۹۲۴ Evidence Locator؛
7. کنترل پوشش تمام ۳۸۱۳ Query؛
8. ثبت Multiplicity برای جلوگیری از فرض نادرست ارتباط یک‌به‌یک.

Mapping براساس Exact Canonical Source Locator Identity انجام می‌شود و از تطبیق تقریبی متن، Embedding یا جست‌وجوی واژگانی استفاده نمی‌کند.

در این مرحله هیچ Query از Test برای انتخاب پیکربندی استفاده نمی‌شود و هیچ Artifactی تغییر نمی‌کند.

</div>

In [4]:
# Code Cell 04 — Rebuild the exact Golden Evidence-to-Chunk mapping

EXPECTED_EVIDENCE_LOCATOR_COUNT = 3924
EXPECTED_UNIQUE_RELEVANT_CHUNK_COUNT = 2483
UPSTREAM_GOLD_MAPPING_SHA256 = (
    "42216f67665cfc50697789dffe4eb5508"
    "39ee22bcbfc0489accc407519dad4d1"
)


def canonical_json_identity(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


chunk_ids_set = set(chunk_ids)

locator_to_chunk_ids = (
    collections.defaultdict(set)
)

chunk_member_locator_pair_count = 0

for chunk_record in chunk_records:
    chunk_id = chunk_record["chunk_id"]

    assert chunk_id in chunk_ids_set

    member_locators = chunk_record[
        "member_locators"
    ]

    assert isinstance(
        member_locators,
        list,
    )
    assert member_locators

    for member_locator in member_locators:
        assert isinstance(
            member_locator,
            dict,
        )
        assert "locator" in member_locator

        locator_identity = (
            canonical_json_identity(
                member_locator["locator"]
            )
        )

        locator_to_chunk_ids[
            locator_identity
        ].add(chunk_id)

        chunk_member_locator_pair_count += 1

assert (
    chunk_member_locator_pair_count
    == 25120
)
assert len(locator_to_chunk_ids) == 19456

gold_mapping_by_candidate_id = {}
gold_mapping_rows = []
evidence_locator_multiplicity = (
    collections.Counter()
)
query_relevant_chunk_multiplicity = (
    collections.Counter()
)
mapped_query_source_type_counts = (
    collections.Counter()
)
unmapped_evidence_rows = []
unmapped_query_ids = []

total_evidence_locator_count = 0
mapped_evidence_locator_count = 0

for golden_record in golden_records:
    candidate_id = golden_record[
        "candidate_id"
    ]

    relevant_chunk_ids = set()

    evidence_locators = golden_record[
        "evidence_locators"
    ]

    assert isinstance(
        evidence_locators,
        list,
    )
    assert evidence_locators

    for evidence_position, evidence_locator in (
        enumerate(evidence_locators)
    ):
        assert isinstance(
            evidence_locator,
            dict,
        )
        assert "source_locator" in (
            evidence_locator
        )

        source_locator = evidence_locator[
            "source_locator"
        ]

        locator_identity = (
            canonical_json_identity(
                source_locator
            )
        )

        matched_chunk_ids = (
            locator_to_chunk_ids.get(
                locator_identity,
                set(),
            )
        )

        total_evidence_locator_count += 1

        if not matched_chunk_ids:
            unmapped_evidence_rows.append(
                {
                    "candidate_id": (
                        candidate_id
                    ),
                    "evidence_position": (
                        evidence_position
                    ),
                    "source_locator": (
                        source_locator
                    ),
                }
            )
            continue

        mapped_evidence_locator_count += 1

        evidence_locator_multiplicity[
            len(matched_chunk_ids)
        ] += 1

        relevant_chunk_ids.update(
            matched_chunk_ids
        )

    if not relevant_chunk_ids:
        unmapped_query_ids.append(
            candidate_id
        )
        continue

    sorted_relevant_chunk_ids = sorted(
        relevant_chunk_ids
    )

    gold_mapping_by_candidate_id[
        candidate_id
    ] = tuple(
        sorted_relevant_chunk_ids
    )

    query_relevant_chunk_multiplicity[
        len(sorted_relevant_chunk_ids)
    ] += 1

    for source_type in set(
        golden_record["source_types"]
    ):
        mapped_query_source_type_counts[
            source_type
        ] += 1

    gold_mapping_rows.append(
        {
            "candidate_id": (
                candidate_id
            ),
            "relevant_chunk_ids": (
                sorted_relevant_chunk_ids
            ),
        }
    )

assert (
    total_evidence_locator_count
    == EXPECTED_EVIDENCE_LOCATOR_COUNT
)
assert (
    mapped_evidence_locator_count
    == EXPECTED_EVIDENCE_LOCATOR_COUNT
)
assert not unmapped_evidence_rows
assert not unmapped_query_ids

assert len(
    gold_mapping_by_candidate_id
) == EXPECTED_GOLDEN_QUERY_COUNT

assert set(
    gold_mapping_by_candidate_id
) == set(golden_candidate_ids)

all_relevant_chunk_ids = set()

for relevant_chunk_ids in (
    gold_mapping_by_candidate_id.values()
):
    assert relevant_chunk_ids
    assert set(
        relevant_chunk_ids
    ).issubset(chunk_ids_set)

    all_relevant_chunk_ids.update(
        relevant_chunk_ids
    )

assert len(
    all_relevant_chunk_ids
) == EXPECTED_UNIQUE_RELEVANT_CHUNK_COUNT

gold_mapping_rows = sorted(
    gold_mapping_rows,
    key=lambda row: row["candidate_id"],
)

nb16_gold_mapping_payload = (
    canonical_json_identity(
        gold_mapping_rows
    )
)

nb16_gold_mapping_sha256 = (
    hashlib.sha256(
        nb16_gold_mapping_payload.encode(
            "utf-8"
        )
    ).hexdigest()
)

gold_mapping_validation = {
    "mapping_method": (
        "EXACT_CANONICAL_SOURCE_LOCATOR_IDENTITY"
    ),
    "canonicalization": (
        "json.dumps("
        "ensure_ascii=False,"
        "sort_keys=True,"
        "separators=(',', ':'))"
    ),
    "chunk_count": len(chunk_records),
    "chunk_member_locator_pair_count": (
        chunk_member_locator_pair_count
    ),
    "unique_chunk_locator_count": len(
        locator_to_chunk_ids
    ),
    "golden_query_count": len(
        golden_records
    ),
    "evidence_locator_count": (
        total_evidence_locator_count
    ),
    "mapped_evidence_count": (
        mapped_evidence_locator_count
    ),
    "unmapped_evidence_count": len(
        unmapped_evidence_rows
    ),
    "mapped_query_count": len(
        gold_mapping_by_candidate_id
    ),
    "unmapped_query_count": len(
        unmapped_query_ids
    ),
    "unique_relevant_chunk_count": len(
        all_relevant_chunk_ids
    ),
    "evidence_to_chunk_multiplicity": {
        str(multiplicity): count
        for multiplicity, count
        in sorted(
            evidence_locator_multiplicity.items()
        )
    },
    "query_relevant_chunk_multiplicity": {
        str(multiplicity): count
        for multiplicity, count
        in sorted(
            query_relevant_chunk_multiplicity.items()
        )
    },
    "mapped_query_source_type_counts": dict(
        sorted(
            mapped_query_source_type_counts.items()
        )
    ),
    "upstream_gold_mapping_sha256": (
        UPSTREAM_GOLD_MAPPING_SHA256
    ),
    "nb16_mapping_payload_sha256": (
        nb16_gold_mapping_sha256
    ),
    "mapping_semantic_parity": "PASS",
    "repository_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "gold_mapping_status": "PASS",
}

print(
    json.dumps(
        gold_mapping_validation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "mapping_method": "EXACT_CANONICAL_SOURCE_LOCATOR_IDENTITY",
  "canonicalization": "json.dumps(ensure_ascii=False,sort_keys=True,separators=(',', ':'))",
  "chunk_count": 3488,
  "chunk_member_locator_pair_count": 25120,
  "unique_chunk_locator_count": 19456,
  "golden_query_count": 3813,
  "evidence_locator_count": 3924,
  "mapped_evidence_count": 3924,
  "unmapped_evidence_count": 0,
  "mapped_query_count": 3813,
  "unmapped_query_count": 0,
  "unique_relevant_chunk_count": 2483,
  "evidence_to_chunk_multiplicity": {
    "1": 1543,
    "2": 1787,
    "3": 371,
    "4": 121,
    "5": 19,
    "6": 30,
    "7": 30,
    "8": 12,
    "10": 11
  },
  "query_relevant_chunk_multiplicity": {
    "1": 1447,
    "2": 1776,
    "3": 374,
    "4": 118,
    "5": 18,
    "6": 26,
    "7": 29,
    "8": 12,
    "9": 1,
    "10": 12
  },
  "mapped_query_source_type_counts": {
    "book": 395,
    "transcript": 2089,
    "website": 1329
  },
  "upstream_gold_mapping_sha256": "42216f67665cfc50697789

<div dir="rtl" align="right">

## بازسازی Development/Test Split و قفل Test

برای جلوگیری از Data Leakage، انتخاب روش Fusion فقط روی Development Split انجام خواهد شد.

Split با همان قرارداد NB15 بازسازی می‌شود:

- نرمال‌سازی سؤال با Unicode NFC؛
- Collapse کردن فاصله‌های تکراری؛
- گروه‌بندی تمام Queryهای دارای سؤال یکسان؛
- Stratification براساس ترکیب دقیق Source Typeها؛
- مرتب‌سازی قطعی گروه‌ها با SHA-256 و Seed ثابت؛
- تخصیص تقریباً ۳۰ درصد هر Stratum به Test؛
- جلوگیری کامل از حضور یک Question Group در هر دو Split.

پس از بازسازی، تعداد رکوردها، تعداد گروه‌ها، توزیع Source Typeها و Multiplicity Relevant Chunkها با قرارداد NB15 تطبیق داده می‌شوند.

Test Split پس از این سل قفل باقی می‌ماند. تا پیش از انتخاب و فریز پیکربندی Hybrid و ثبت ADR، متن Queryهای Test برای Benchmark یا انتخاب پارامترها مصرف نخواهد شد.

</div>

In [5]:
# Code Cell 05 — Rebuild the exact NB15 grouped split and lock test

SPLIT_POLICY_VERSION = (
    "nb15_grouped_query_hash_split_v1"
)
SPLIT_SEED = (
    "rag_finance_phase2_nb15_sparse_v1"
)
TARGET_TEST_RATIO = 0.30
UPSTREAM_SPLIT_SHA256 = (
    "68df1196f683b6ab9a9df08c015f001b"
    "0caca056331eb80d0048aec783736d8f"
)


def canonical_split_question(
    question,
):
    normalized = unicodedata.normalize(
        "NFC",
        question,
    )

    return " ".join(
        normalized.split()
    )


golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in golden_records
}

assert len(
    golden_record_by_candidate_id
) == len(golden_records)

query_groups = collections.defaultdict(
    list
)

for record in golden_records:
    candidate_id = record[
        "candidate_id"
    ]

    question = record[
        "evaluation_content"
    ]["question"]

    question_group_key = (
        canonical_split_question(
            question
        )
    )

    assert question_group_key

    query_groups[
        question_group_key
    ].append(candidate_id)

assert len(query_groups) == 3735

group_records = []

for question_group_key, candidate_ids in (
    query_groups.items()
):
    group_source_types = sorted(
        {
            str(source_type)
            for candidate_id in candidate_ids
            for source_type in (
                golden_record_by_candidate_id[
                    candidate_id
                ]["source_types"]
            )
        }
    )

    assert group_source_types

    source_signature = "|".join(
        group_source_types
    )

    group_hash = hashlib.sha256(
        (
            SPLIT_SEED
            + "\x1f"
            + question_group_key
        ).encode("utf-8")
    ).hexdigest()

    group_records.append(
        {
            "question_group_key": (
                question_group_key
            ),
            "candidate_ids": sorted(
                candidate_ids
            ),
            "record_count": len(
                candidate_ids
            ),
            "source_types": (
                group_source_types
            ),
            "source_signature": (
                source_signature
            ),
            "group_hash": group_hash,
        }
    )

groups_by_source_signature = (
    collections.defaultdict(list)
)

for group_record in group_records:
    groups_by_source_signature[
        group_record["source_signature"]
    ].append(group_record)

split_by_candidate_id = {}
split_by_question_group = {}
stratum_summaries = {}

for source_signature, stratum_groups in sorted(
    groups_by_source_signature.items()
):
    ordered_groups = sorted(
        stratum_groups,
        key=lambda item: (
            item["group_hash"],
            item["question_group_key"],
        ),
    )

    stratum_record_count = sum(
        group["record_count"]
        for group in ordered_groups
    )

    target_test_count = round(
        stratum_record_count
        * TARGET_TEST_RATIO
    )

    assigned_test_count = 0
    test_group_keys = set()

    for group in ordered_groups:
        if (
            assigned_test_count
            >= target_test_count
        ):
            break

        test_group_keys.add(
            group["question_group_key"]
        )

        assigned_test_count += (
            group["record_count"]
        )

    for group in ordered_groups:
        split_name = (
            "test"
            if group["question_group_key"]
            in test_group_keys
            else "development"
        )

        split_by_question_group[
            group["question_group_key"]
        ] = split_name

        for candidate_id in group[
            "candidate_ids"
        ]:
            assert (
                candidate_id
                not in split_by_candidate_id
            )

            split_by_candidate_id[
                candidate_id
            ] = split_name

    stratum_summaries[
        source_signature
    ] = {
        "group_count": len(
            ordered_groups
        ),
        "record_count": (
            stratum_record_count
        ),
        "target_test_count": (
            target_test_count
        ),
        "actual_test_count": (
            assigned_test_count
        ),
        "actual_test_ratio": (
            assigned_test_count
            / stratum_record_count
        ),
    }

assert set(
    split_by_candidate_id
) == set(golden_candidate_ids)

development_candidate_ids = {
    candidate_id
    for candidate_id, split_name
    in split_by_candidate_id.items()
    if split_name == "development"
}

test_candidate_ids = {
    candidate_id
    for candidate_id, split_name
    in split_by_candidate_id.items()
    if split_name == "test"
}

assert development_candidate_ids
assert test_candidate_ids

assert not (
    development_candidate_ids
    & test_candidate_ids
)

assert (
    development_candidate_ids
    | test_candidate_ids
) == set(golden_candidate_ids)

assert len(
    development_candidate_ids
) == 2664
assert len(test_candidate_ids) == 1149

development_question_groups = {
    canonical_split_question(
        golden_record_by_candidate_id[
            candidate_id
        ]["evaluation_content"][
            "question"
        ]
    )
    for candidate_id
    in development_candidate_ids
}

test_question_groups = {
    canonical_split_question(
        golden_record_by_candidate_id[
            candidate_id
        ]["evaluation_content"][
            "question"
        ]
    )
    for candidate_id
    in test_candidate_ids
}

assert not (
    development_question_groups
    & test_question_groups
)

assert len(
    development_question_groups
) == 2620

assert len(
    test_question_groups
) == 1115

assert all(
    gold_mapping_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in development_candidate_ids
)

assert all(
    gold_mapping_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in test_candidate_ids
)

split_source_type_counts = {
    "development": (
        collections.Counter()
    ),
    "test": collections.Counter(),
}

split_relevant_chunk_counts = {
    "development": (
        collections.Counter()
    ),
    "test": collections.Counter(),
}

for candidate_id, split_name in (
    split_by_candidate_id.items()
):
    record = golden_record_by_candidate_id[
        candidate_id
    ]

    for source_type in record[
        "source_types"
    ]:
        split_source_type_counts[
            split_name
        ][str(source_type)] += 1

    split_relevant_chunk_counts[
        split_name
    ][
        len(
            gold_mapping_by_candidate_id[
                candidate_id
            ]
        )
    ] += 1

all_source_types = set(
    split_source_type_counts[
        "development"
    ]
) | set(
    split_source_type_counts["test"]
)

assert all_source_types == {
    "book",
    "transcript",
    "website",
}

development_source_type_counts = dict(
    sorted(
        split_source_type_counts[
            "development"
        ].items()
    )
)

test_source_type_counts = dict(
    sorted(
        split_source_type_counts[
            "test"
        ].items()
    )
)

development_relevant_multiplicity = {
    str(key): value
    for key, value
    in sorted(
        split_relevant_chunk_counts[
            "development"
        ].items()
    )
}

test_relevant_multiplicity = {
    str(key): value
    for key, value
    in sorted(
        split_relevant_chunk_counts[
            "test"
        ].items()
    )
}

assert development_source_type_counts == {
    "book": 277,
    "transcript": 1459,
    "website": 928,
}

assert test_source_type_counts == {
    "book": 118,
    "transcript": 630,
    "website": 401,
}

assert development_relevant_multiplicity == {
    "1": 997,
    "2": 1250,
    "3": 260,
    "4": 87,
    "5": 14,
    "6": 19,
    "7": 20,
    "8": 7,
    "9": 1,
    "10": 9,
}

assert test_relevant_multiplicity == {
    "1": 450,
    "2": 526,
    "3": 114,
    "4": 31,
    "5": 4,
    "6": 7,
    "7": 9,
    "8": 5,
    "10": 3,
}

expected_stratum_summaries = {
    "book": {
        "group_count": 375,
        "record_count": 375,
        "target_test_count": 112,
        "actual_test_count": 112,
    },
    "book|transcript": {
        "group_count": 14,
        "record_count": 34,
        "target_test_count": 10,
        "actual_test_count": 11,
    },
    "book|transcript|website": {
        "group_count": 2,
        "record_count": 10,
        "target_test_count": 3,
        "actual_test_count": 4,
    },
    "transcript": {
        "group_count": 2028,
        "record_count": 2042,
        "target_test_count": 613,
        "actual_test_count": 613,
    },
    "transcript|website": {
        "group_count": 8,
        "record_count": 39,
        "target_test_count": 12,
        "actual_test_count": 15,
    },
    "website": {
        "group_count": 1308,
        "record_count": 1313,
        "target_test_count": 394,
        "actual_test_count": 394,
    },
}

for source_signature, expected_values in (
    expected_stratum_summaries.items()
):
    assert source_signature in (
        stratum_summaries
    )

    for field_name, expected_value in (
        expected_values.items()
    ):
        assert (
            stratum_summaries[
                source_signature
            ][field_name]
            == expected_value
        )

actual_test_ratio = (
    len(test_candidate_ids)
    / len(golden_records)
)

assert 0.28 <= actual_test_ratio <= 0.32

split_digest = hashlib.sha256()

for candidate_id in sorted(
    split_by_candidate_id
):
    split_component = (
        candidate_id
        + "\x1f"
        + split_by_candidate_id[
            candidate_id
        ]
    ).encode("utf-8")

    split_digest.update(
        len(split_component).to_bytes(
            8,
            byteorder="big",
            signed=False,
        )
    )

    split_digest.update(
        split_component
    )

split_sha256 = split_digest.hexdigest()

assert (
    split_sha256
    == UPSTREAM_SPLIT_SHA256
)

TEST_SPLIT_LOCKED = True
HYBRID_CONFIGURATION_FROZEN = False
TEST_QUERIES_ACCESSED = False

split_contract = {
    "policy_version": (
        SPLIT_POLICY_VERSION
    ),
    "seed": SPLIT_SEED,
    "target_test_ratio": (
        TARGET_TEST_RATIO
    ),
    "grouping": (
        "Unicode NFC + whitespace collapse "
        "on evaluation_content.question"
    ),
    "group_hash_contract": (
        "SHA256(seed + U+001F + "
        "normalized_question)"
    ),
    "stratification": (
        "exact group source-type signature"
    ),
    "record_count": len(
        golden_records
    ),
    "unique_question_group_count": len(
        query_groups
    ),
    "development": {
        "record_count": len(
            development_candidate_ids
        ),
        "question_group_count": len(
            development_question_groups
        ),
        "source_type_counts": (
            development_source_type_counts
        ),
        "relevant_chunk_multiplicity": (
            development_relevant_multiplicity
        ),
    },
    "test": {
        "record_count": len(
            test_candidate_ids
        ),
        "question_group_count": len(
            test_question_groups
        ),
        "actual_ratio": (
            actual_test_ratio
        ),
        "source_type_counts": (
            test_source_type_counts
        ),
        "relevant_chunk_multiplicity": (
            test_relevant_multiplicity
        ),
    },
    "strata": stratum_summaries,
    "cross_split_question_group_overlap": 0,
    "gold_mapping_coverage": "PASS",
    "split_sha256": split_sha256,
    "upstream_split_sha256": (
        UPSTREAM_SPLIT_SHA256
    ),
    "exact_upstream_split_parity": "PASS",
    "test_access_policy": (
        "LOCKED_UNTIL_HYBRID_CONFIG_"
        "AND_ADR_ARE_FROZEN"
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "split_status": "PASS",
}

print(
    json.dumps(
        split_contract,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "policy_version": "nb15_grouped_query_hash_split_v1",
  "seed": "rag_finance_phase2_nb15_sparse_v1",
  "target_test_ratio": 0.3,
  "grouping": "Unicode NFC + whitespace collapse on evaluation_content.question",
  "group_hash_contract": "SHA256(seed + U+001F + normalized_question)",
  "stratification": "exact group source-type signature",
  "record_count": 3813,
  "unique_question_group_count": 3735,
  "development": {
    "record_count": 2664,
    "question_group_count": 2620,
    "source_type_counts": {
      "book": 277,
      "transcript": 1459,
      "website": 928
    },
    "relevant_chunk_multiplicity": {
      "1": 997,
      "2": 1250,
      "3": 260,
      "4": 87,
      "5": 14,
      "6": 19,
      "7": 20,
      "8": 7,
      "9": 1,
      "10": 9
    }
  },
  "test": {
    "record_count": 1149,
    "question_group_count": 1115,
    "actual_ratio": 0.3013375295043273,
    "source_type_counts": {
      "book": 118,
      "transcript": 630,
      "website": 401
    },
  

<div dir="rtl" align="right">

## Reload مستقل Sparse Index و Analyzer فریز‌شده NB15

NB16 نباید Sparse Index را دوباره Train یا براساس داده‌های Hybrid تغییر دهد. در این مرحله Analyzer و BM25 فقط از Artifactهای نسخه‌دار NB15 بازسازی می‌شوند.

کنترل‌ها شامل موارد زیر هستند:

1. بارگذاری پیکربندی فریز‌شده Analyzer؛
2. بازسازی دقیق نرمال‌سازی فارسی، ارقام و نیم‌فاصله؛
3. بارگذاری Document Table؛
4. بارگذاری ۲۷۳۱۱ Posting؛
5. اعتبارسنجی ۳۸۳۶۳۷ Posting Entry؛
6. برابری ترتیب Document IDها با Canonical Chunkها؛
7. اجرای Unit Testهای Analyzer؛
8. تثبیت `k1=1.2` و `b=0.3`؛
9. جلوگیری از هرگونه Mutation یا Retuning در NB16.

در این سل هیچ Query از Test مصرف نمی‌شود.

</div>

In [6]:
# Code Cell 06 — Reload the frozen NB15 sparse index and analyzer

sparse_configuration = json.loads(
    sparse_configuration_path.read_text(
        encoding="utf-8"
    )
)

assert (
    sparse_configuration[
        "configuration_sha256"
    ]
    == EXPECTED_SPARSE_CONFIGURATION_SHA256
)

assert (
    sparse_configuration[
        "configuration_status"
    ]
    == "FROZEN_BEFORE_TEST_ACCESS"
)

frozen_sparse_analyzer_definition = (
    sparse_configuration[
        "analyzer"
    ]["definition"]
)

assert (
    frozen_sparse_analyzer_definition[
        "analyzer_id"
    ]
    == "fa_join_zwnj_stopwords_v1"
)

FROZEN_SPARSE_K1 = float(
    sparse_configuration["bm25"]["k1"]
)
FROZEN_SPARSE_B = float(
    sparse_configuration["bm25"]["b"]
)

assert FROZEN_SPARSE_K1 == 1.2
assert FROZEN_SPARSE_B == 0.3


@dataclasses.dataclass(frozen=True)
class FrozenSparseAnalyzerConfig:
    analyzer_id: str
    unicode_form: str
    zwnj_policy: str
    normalize_arabic_letters: bool
    normalize_digits: bool
    remove_diacritics: bool
    lowercase_latin: bool
    remove_stopwords: bool
    stopword_version: str | None


frozen_sparse_analyzer_config = (
    FrozenSparseAnalyzerConfig(
        **frozen_sparse_analyzer_definition
    )
)

PERSIAN_STOPWORDS_V1 = frozenset(
    sparse_configuration[
        "analyzer"
    ]["stopwords"]
)

assert len(PERSIAN_STOPWORDS_V1) == 28

ARABIC_TO_PERSIAN_CHARACTER_MAP = (
    str.maketrans(
        {
            "ي": "ی",
            "ى": "ی",
            "ك": "ک",
            "ة": "ه",
        }
    )
)

DIGIT_TO_ASCII_MAP = str.maketrans(
    {
        "۰": "0",
        "۱": "1",
        "۲": "2",
        "۳": "3",
        "۴": "4",
        "۵": "5",
        "۶": "6",
        "۷": "7",
        "۸": "8",
        "۹": "9",
        "٠": "0",
        "١": "1",
        "٢": "2",
        "٣": "3",
        "٤": "4",
        "٥": "5",
        "٦": "6",
        "٧": "7",
        "٨": "8",
        "٩": "9",
    }
)


def normalize_numeric_notation(
    text,
):
    text = re.sub(
        r"(?<=\d)[,٬](?=\d)",
        "",
        text,
    )

    text = re.sub(
        r"(?<=\d)٫(?=\d)",
        ".",
        text,
    )

    text = text.replace(
        "٪",
        " __percent__ ",
    )

    text = text.replace(
        "%",
        " __percent__ ",
    )

    return text


def analyze_frozen_sparse_text(
    text,
):
    assert isinstance(text, str)

    config = frozen_sparse_analyzer_config

    normalized = unicodedata.normalize(
        config.unicode_form,
        text,
    )

    if config.normalize_arabic_letters:
        normalized = normalized.translate(
            ARABIC_TO_PERSIAN_CHARACTER_MAP
        )

    if config.normalize_digits:
        normalized = normalized.translate(
            DIGIT_TO_ASCII_MAP
        )

    if config.lowercase_latin:
        normalized = normalized.lower()

    normalized = normalize_numeric_notation(
        normalized
    )

    if config.zwnj_policy == "preserve":
        pass
    elif config.zwnj_policy == "join":
        normalized = normalized.replace(
            "\u200c",
            "",
        )
    elif config.zwnj_policy == "split":
        normalized = normalized.replace(
            "\u200c",
            " ",
        )
    else:
        raise AssertionError(
            "Unknown frozen ZWNJ policy: "
            f"{config.zwnj_policy}"
        )

    output_characters = []

    for character in normalized:
        if (
            config.remove_diacritics
            and unicodedata.category(
                character
            ).startswith("M")
        ):
            continue

        if character == "\u200c":
            output_characters.append(
                character
            )
            continue

        if character == ".":
            output_characters.append(
                character
            )
            continue

        if (
            character == "_"
            or character.isalpha()
            or character.isdigit()
        ):
            output_characters.append(
                character
            )
        else:
            output_characters.append(" ")

    normalized_for_tokens = "".join(
        output_characters
    )

    raw_tokens = (
        normalized_for_tokens.split()
    )

    tokens = []

    for token in raw_tokens:
        token = token.strip("._")

        if not token:
            continue

        if (
            config.remove_stopwords
            and token
            in PERSIAN_STOPWORDS_V1
        ):
            continue

        tokens.append(token)

    return tokens


analyzer_unit_cases = [
    {
        "text": (
            "مي‌خواهم كسب‌وكار را "
            "بررسی کنم."
        ),
        "expected": [
            "میخواهم",
            "کسبوکار",
            "بررسی",
            "کنم",
        ],
    },
    {
        "text": (
            "بازدهی ۱۲٫۵٪ و مبلغ "
            "1,250 تومان"
        ),
        "expected": [
            "بازدهی",
            "12.5",
            "percent",
            "مبلغ",
            "1250",
            "تومان",
        ],
    },
    {
        "text": (
            "ETF، P/E و بهرهٔ "
            "مرکب چیست؟"
        ),
        "expected": [
            "etf",
            "p",
            "e",
            "بهره",
            "مرکب",
            "چیست",
        ],
    },
]

for unit_case in analyzer_unit_cases:
    actual_tokens = (
        analyze_frozen_sparse_text(
            unit_case["text"]
        )
    )

    assert (
        actual_tokens
        == unit_case["expected"]
    ), {
        "text": unit_case["text"],
        "expected": unit_case[
            "expected"
        ],
        "actual": actual_tokens,
    }


@dataclasses.dataclass
class FrozenSparseIndex:
    analyzer_id: str
    document_ids: list
    document_lengths: np.ndarray
    average_document_length: float
    postings: dict
    document_frequency: dict
    vocabulary_size: int
    total_token_count: int


sparse_document_rows = (
    load_jsonl_records(
        sparse_documents_path
    )
)

assert len(
    sparse_document_rows
) == EXPECTED_DOCUMENT_COUNT

assert [
    row["document_index"]
    for row in sparse_document_rows
] == list(
    range(EXPECTED_DOCUMENT_COUNT)
)

sparse_document_ids = [
    row["document_id"]
    for row in sparse_document_rows
]

sparse_document_lengths = np.asarray(
    [
        row["document_length"]
        for row in sparse_document_rows
    ],
    dtype=np.float32,
)

assert sparse_document_ids == chunk_ids
assert np.all(
    sparse_document_lengths > 0
)

sparse_postings = {}
sparse_document_frequency = {}
sparse_posting_entry_count = 0
previous_token = None

with sparse_postings_path.open(
    "r",
    encoding="utf-8",
) as file_handle:
    for line_number, line in enumerate(
        file_handle,
        start=1,
    ):
        stripped_line = line.strip()

        assert stripped_line, (
            "Empty sparse posting line: "
            f"{line_number}"
        )

        posting_row = json.loads(
            stripped_line
        )

        token = posting_row["token"]

        if previous_token is not None:
            assert previous_token < token

        previous_token = token

        document_indexes = np.asarray(
            posting_row[
                "document_indexes"
            ],
            dtype=np.int32,
        )

        term_frequencies = np.asarray(
            posting_row[
                "term_frequencies"
            ],
            dtype=np.float32,
        )

        document_frequency = int(
            posting_row[
                "document_frequency"
            ]
        )

        assert token not in sparse_postings
        assert len(document_indexes) > 0
        assert len(
            document_indexes
        ) == len(term_frequencies)
        assert len(
            document_indexes
        ) == document_frequency
        assert np.all(
            document_indexes >= 0
        )
        assert np.all(
            document_indexes
            < EXPECTED_DOCUMENT_COUNT
        )
        assert np.all(
            document_indexes[:-1]
            < document_indexes[1:]
        )
        assert np.all(
            term_frequencies > 0
        )

        sparse_postings[token] = (
            document_indexes,
            term_frequencies,
        )

        sparse_document_frequency[
            token
        ] = document_frequency

        sparse_posting_entry_count += len(
            document_indexes
        )

assert len(sparse_postings) == 27311
assert sparse_posting_entry_count == 383637

frozen_sparse_index = FrozenSparseIndex(
    analyzer_id=(
        frozen_sparse_analyzer_config
        .analyzer_id
    ),
    document_ids=(
        sparse_document_ids
    ),
    document_lengths=(
        sparse_document_lengths
    ),
    average_document_length=float(
        sparse_configuration[
            "index_statistics"
        ]["average_document_length"]
    ),
    postings=sparse_postings,
    document_frequency=(
        sparse_document_frequency
    ),
    vocabulary_size=len(
        sparse_postings
    ),
    total_token_count=int(
        sparse_configuration[
            "index_statistics"
        ]["total_token_count"]
    ),
)

assert (
    frozen_sparse_index.analyzer_id
    == "fa_join_zwnj_stopwords_v1"
)
assert len(
    frozen_sparse_index.document_ids
) == 3488
assert (
    frozen_sparse_index.vocabulary_size
    == 27311
)
assert (
    frozen_sparse_index.total_token_count
    == 532085
)

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

sparse_reload_contract = {
    "analyzer": {
        "analyzer_id": (
            frozen_sparse_index.analyzer_id
        ),
        "unit_test_count": len(
            analyzer_unit_cases
        ),
        "unit_tests": "PASS",
        "mutation_allowed": False,
    },
    "bm25": {
        "implementation": (
            "local_robertson_bm25_v1"
        ),
        "k1": FROZEN_SPARSE_K1,
        "b": FROZEN_SPARSE_B,
        "mutation_allowed": False,
    },
    "index": {
        "document_count": len(
            frozen_sparse_index.document_ids
        ),
        "vocabulary_size": (
            frozen_sparse_index
            .vocabulary_size
        ),
        "posting_entry_count": (
            sparse_posting_entry_count
        ),
        "total_token_count": (
            frozen_sparse_index
            .total_token_count
        ),
        "average_document_length": (
            frozen_sparse_index
            .average_document_length
        ),
    },
    "document_id_parity": "PASS",
    "test_queries_accessed": False,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "sparse_reload_status": "PASS",
}

print(
    json.dumps(
        sparse_reload_contract,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "analyzer": {
    "analyzer_id": "fa_join_zwnj_stopwords_v1",
    "unit_test_count": 3,
    "unit_tests": "PASS",
    "mutation_allowed": false
  },
  "bm25": {
    "implementation": "local_robertson_bm25_v1",
    "k1": 1.2,
    "b": 0.3,
    "mutation_allowed": false
  },
  "index": {
    "document_count": 3488,
    "vocabulary_size": 27311,
    "posting_entry_count": 383637,
    "total_token_count": 532085,
    "average_document_length": 152.54730224609375
  },
  "document_id_parity": "PASS",
  "test_queries_accessed": false,
  "files_created": 0,
  "repository_mutated": false,
  "paid_api_calls": 0,
  "sparse_reload_status": "PASS"
}


<div dir="rtl" align="right">

## کشف قرارداد اجرایی Dense Retrieval

Dense Release و Handoff از نظر SHA و وضعیت Freeze معتبر هستند، اما پیش از اتصال به Qdrant باید ساختار دقیق قرارداد اجرایی آن‌ها بررسی شود.

این مرحله فقط اطلاعات لازم برای اجرای Dense Baseline را استخراج می‌کند:

- نام Collection؛
- Dimension و Distance؛
- روش اتصال یا مسیر Storage؛
- مدل Embedding و Revision فریز‌شده؛
- مسیر Cache یا Embedding Artifact؛
- Payload Schema و فیلد شناسه Chunk؛
- محدودیت Mutation؛
- Artifactهای وابسته.

این سل هیچ اتصال تغییردهنده‌ای به Qdrant ایجاد نمی‌کند، هیچ Query Embedding جدیدی نمی‌سازد و هیچ Query از Test را مصرف نمی‌کند.

خروجی این مرحله مبنای ساخت Loader دقیق Dense Retrieval در سل بعد خواهد بود.

</div>

In [7]:
# Code Cell 07 — Inspect the frozen dense runtime contract without mutation

def flatten_contract_scalars(
    value,
    prefix="",
):
    flattened = {}

    if isinstance(value, dict):
        for key, child_value in value.items():
            child_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            flattened.update(
                flatten_contract_scalars(
                    child_value,
                    child_prefix,
                )
            )

    elif isinstance(value, list):
        for item_index, child_value in enumerate(
            value
        ):
            child_prefix = (
                f"{prefix}[{item_index}]"
            )

            flattened.update(
                flatten_contract_scalars(
                    child_value,
                    child_prefix,
                )
            )

    elif isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
            type(None),
        ),
    ):
        flattened[prefix] = value

    return flattened


dense_release_scalars = (
    flatten_contract_scalars(
        dense_release
    )
)

dense_handoff_scalars = (
    flatten_contract_scalars(
        dense_handoff
    )
)

dense_runtime_key_pattern = re.compile(
    (
        r"collection|qdrant|storage|snapshot|"
        r"model|revision|embedding|dimension|"
        r"distance|vector|payload|chunk|"
        r"cache|release|manifest|status|"
        r"mutation|path|sha256"
    ),
    flags=re.IGNORECASE,
)

dense_runtime_release_fields = {
    key: value
    for key, value
    in sorted(
        dense_release_scalars.items()
    )
    if dense_runtime_key_pattern.search(
        key
    )
}

dense_runtime_handoff_fields = {
    key: value
    for key, value
    in sorted(
        dense_handoff_scalars.items()
    )
    if dense_runtime_key_pattern.search(
        key
    )
}

collection_name_candidates = sorted(
    {
        value
        for key, value
        in {
            **dense_release_scalars,
            **dense_handoff_scalars,
        }.items()
        if (
            isinstance(value, str)
            and (
                "collection" in key.lower()
                or value.startswith(
                    "rag_finance_phase2_dense"
                )
            )
            and value.startswith(
                "rag_finance_phase2_dense"
            )
        )
    }
)

dimension_candidates = sorted(
    {
        int(value)
        for key, value
        in {
            **dense_release_scalars,
            **dense_handoff_scalars,
        }.items()
        if (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and "dimension" in key.lower()
        )
    }
)

distance_candidates = sorted(
    {
        str(value)
        for key, value
        in {
            **dense_release_scalars,
            **dense_handoff_scalars,
        }.items()
        if (
            isinstance(value, str)
            and "distance" in key.lower()
        )
    }
)

model_id_candidates = sorted(
    {
        value
        for key, value
        in {
            **dense_release_scalars,
            **dense_handoff_scalars,
        }.items()
        if (
            isinstance(value, str)
            and (
                "model" in key.lower()
                or "embedding" in key.lower()
            )
            and (
                "/" in value
                or value.startswith(
                    "intfloat"
                )
            )
        )
    }
)

revision_candidates = sorted(
    {
        value
        for key, value
        in {
            **dense_release_scalars,
            **dense_handoff_scalars,
        }.items()
        if (
            isinstance(value, str)
            and "revision" in key.lower()
            and re.fullmatch(
                r"[0-9a-f]{40}",
                value,
            )
        )
    }
)

path_candidates = sorted(
    {
        value
        for key, value
        in {
            **dense_release_scalars,
            **dense_handoff_scalars,
        }.items()
        if (
            isinstance(value, str)
            and "path" in key.lower()
        )
    }
)

existing_project_paths = []

for path_value in path_candidates:
    candidate_path = Path(path_value)

    if candidate_path.is_absolute():
        resolved_candidate = (
            candidate_path.resolve()
        )
    else:
        resolved_candidate = (
            PROJECT_ROOT
            / candidate_path
        ).resolve()

    if resolved_candidate.exists():
        existing_project_paths.append(
            {
                "contract_path": (
                    path_value
                ),
                "resolved_path": str(
                    resolved_candidate
                ),
                "is_file": (
                    resolved_candidate.is_file()
                ),
                "is_directory": (
                    resolved_candidate
                    .is_dir()
                ),
            }
        )

assert collection_name_candidates, (
    "Dense collection name was not found "
    "in the frozen contracts."
)

assert EXPECTED_DENSE_LOGICAL_INDEX_SHA256 in {
    value
    for value in dense_release_scalars.values()
    if isinstance(value, str)
}

assert dense_release_status == (
    "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
)
assert dense_handoff_status == (
    "READY_FOR_NB16"
)

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

dense_runtime_contract_inspection = {
    "collection_name_candidates": (
        collection_name_candidates
    ),
    "dimension_candidates": (
        dimension_candidates
    ),
    "distance_candidates": (
        distance_candidates
    ),
    "model_id_candidates": (
        model_id_candidates
    ),
    "revision_candidates": (
        revision_candidates
    ),
    "existing_project_paths": (
        existing_project_paths
    ),
    "release_fields": (
        dense_runtime_release_fields
    ),
    "handoff_fields": (
        dense_runtime_handoff_fields
    ),
    "dense_release_status": (
        dense_release_status
    ),
    "dense_handoff_status": (
        dense_handoff_status
    ),
    "dense_mutation_allowed": False,
    "test_queries_accessed": False,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "contract_inspection_status": "PASS",
}

print(
    json.dumps(
        dense_runtime_contract_inspection,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "collection_name_candidates": [
    "rag_finance_phase2_dense_v1"
  ],
  "dimension_candidates": [
    384
  ],
  "distance_candidates": [
    "COSINE"
  ],
  "model_id_candidates": [
    "artifacts/phase2/embedding/embedding_release_v1.json",
    "intfloat/multilingual-e5-small"
  ],
  "revision_candidates": [
    "614241f622f53c4eeff9890bdc4f31cfecc418b3"
  ],
  "existing_project_paths": [
    {
      "contract_path": "artifacts/phase2/dense_index/dense_index_release_v1.json",
      "resolved_path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/dense_index/dense_index_release_v1.json",
      "is_file": true,
      "is_directory": false
    },
    {
      "contract_path": "artifacts/phase2/embedding/embedding_release_v1.json",
      "resolved_path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding/embedding_release_v1.json",
      "is_file": true,
      "is_directory": false
    },
 

<div dir="rtl" align="right">

## اتصال Read-only به Dense Index فریز‌شده

در این مرحله Qdrant Local Storage مربوط به NB14 باز می‌شود، اما هیچ Collection یا Pointی ایجاد، حذف یا به‌روزرسانی نخواهد شد.

کنترل‌ها شامل موارد زیر هستند:

1. وجود Collection فریز‌شده؛
2. تعداد دقیق ۳۴۸۸ Point؛
3. Dimension برابر ۳۸۴؛
4. Distance برابر COSINE؛
5. یکتایی Point IDها؛
6. یکتایی Payload `chunk_id`؛
7. برابری کامل مجموعه Chunk IDهای Dense با Corpus و Sparse؛
8. کنترل Source Typeها؛
9. کنترل Embedding Revision ثبت‌شده در Payload؛
10. اثبات عدم تغییر Dense Index.

این مرحله فقط Metadata و Payload را می‌خواند و Vectorها را دوباره محاسبه یا بازنویسی نمی‌کند.

</div>

In [8]:
# Code Cell 08 — Open and validate the frozen Qdrant collection read-only

DENSE_COLLECTION_NAME = (
    collection_name_candidates[0]
)
DENSE_VECTOR_DIMENSION = 384
DENSE_VECTOR_DISTANCE = "COSINE"
DENSE_EMBEDDING_MODEL_ID = (
    "intfloat/multilingual-e5-small"
)
DENSE_EMBEDDING_REVISION = (
    "614241f622f53c4eeff9890bdc4f31cfecc418b3"
)
DENSE_QUERY_PREFIX = "query:"
DENSE_PASSAGE_PREFIX = "passage:"

dense_storage_path = (
    PROJECT_ROOT
    / dense_release[
        "backend"
    ]["storage_path"]
).resolve()

assert dense_storage_path.is_dir()

qdrant_client = QdrantClient(
    path=str(dense_storage_path)
)

assert qdrant_client.collection_exists(
    collection_name=(
        DENSE_COLLECTION_NAME
    )
)

dense_collection_info = (
    qdrant_client.get_collection(
        collection_name=(
            DENSE_COLLECTION_NAME
        )
    )
)

dense_vector_config = (
    dense_collection_info
    .config
    .params
    .vectors
)

if isinstance(
    dense_vector_config,
    dict,
):
    assert len(dense_vector_config) == 1

    dense_vector_config = next(
        iter(
            dense_vector_config.values()
        )
    )

actual_dense_dimension = int(
    dense_vector_config.size
)

actual_dense_distance = str(
    dense_vector_config.distance
).upper()

if "." in actual_dense_distance:
    actual_dense_distance = (
        actual_dense_distance.split(".")[-1]
    )

assert (
    actual_dense_dimension
    == DENSE_VECTOR_DIMENSION
)
assert (
    actual_dense_distance
    == DENSE_VECTOR_DISTANCE
)

dense_exact_count = (
    qdrant_client.count(
        collection_name=(
            DENSE_COLLECTION_NAME
        ),
        exact=True,
    ).count
)

assert (
    dense_exact_count
    == EXPECTED_DOCUMENT_COUNT
)

dense_point_ids = []
dense_payload_by_chunk_id = {}
dense_scroll_offset = None

while True:
    (
        dense_point_batch,
        dense_scroll_offset,
    ) = qdrant_client.scroll(
        collection_name=(
            DENSE_COLLECTION_NAME
        ),
        limit=256,
        offset=dense_scroll_offset,
        with_payload=True,
        with_vectors=False,
    )

    for point in dense_point_batch:
        point_id = str(point.id)
        payload = dict(point.payload or {})

        assert "chunk_id" in payload

        chunk_id = str(
            payload["chunk_id"]
        )

        assert (
            chunk_id
            not in dense_payload_by_chunk_id
        )

        dense_point_ids.append(
            point_id
        )

        dense_payload_by_chunk_id[
            chunk_id
        ] = payload

    if dense_scroll_offset is None:
        break

assert len(dense_point_ids) == (
    EXPECTED_DOCUMENT_COUNT
)
assert len(
    set(dense_point_ids)
) == EXPECTED_DOCUMENT_COUNT
assert len(
    dense_payload_by_chunk_id
) == EXPECTED_DOCUMENT_COUNT

assert set(
    dense_payload_by_chunk_id
) == set(chunk_ids)

assert set(
    dense_payload_by_chunk_id
) == set(
    frozen_sparse_index.document_ids
)

dense_payload_source_type_counts = dict(
    sorted(
        collections.Counter(
            str(
                payload["source_type"]
            )
            for payload
            in dense_payload_by_chunk_id.values()
        ).items()
    )
)

assert (
    dense_payload_source_type_counts
    == chunk_source_type_counts
)

dense_payload_embedding_revisions = {
    str(
        payload["embedding_revision"]
    )
    for payload
    in dense_payload_by_chunk_id.values()
}

assert dense_payload_embedding_revisions == {
    DENSE_EMBEDDING_REVISION
}

dense_payload_strategy_ids = {
    str(payload["strategy_id"])
    for payload
    in dense_payload_by_chunk_id.values()
}

dense_payload_release_ids = {
    str(
        payload[
            "dense_index_release_id"
        ]
    )
    for payload
    in dense_payload_by_chunk_id.values()
}

chunk_index_by_id = {
    chunk_id: chunk_index
    for chunk_index, chunk_id
    in enumerate(chunk_ids)
}

assert len(
    chunk_index_by_id
) == EXPECTED_DOCUMENT_COUNT

dense_point_count_after_validation = (
    qdrant_client.count(
        collection_name=(
            DENSE_COLLECTION_NAME
        ),
        exact=True,
    ).count
)

assert (
    dense_point_count_after_validation
    == dense_exact_count
)

assert (
    dense_release[
        "constraints"
    ]["dense_index_mutation_allowed"]
    is False
)

assert (
    dense_handoff[
        "constraints"
    ]["do_not_recompute_embeddings"]
    is True
)

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

dense_collection_validation = {
    "collection_name": (
        DENSE_COLLECTION_NAME
    ),
    "storage_path": str(
        dense_release[
            "backend"
        ]["storage_path"]
    ),
    "point_count": (
        dense_exact_count
    ),
    "vector": {
        "dimension": (
            actual_dense_dimension
        ),
        "distance": (
            actual_dense_distance
        ),
        "dtype": "float32",
    },
    "embedding": {
        "model_id": (
            DENSE_EMBEDDING_MODEL_ID
        ),
        "revision": (
            DENSE_EMBEDDING_REVISION
        ),
        "query_prefix": (
            DENSE_QUERY_PREFIX
        ),
        "passage_prefix": (
            DENSE_PASSAGE_PREFIX
        ),
    },
    "payload": {
        "chunk_id_count": len(
            dense_payload_by_chunk_id
        ),
        "chunk_id_parity": "PASS",
        "source_type_counts": (
            dense_payload_source_type_counts
        ),
        "embedding_revisions": sorted(
            dense_payload_embedding_revisions
        ),
        "strategy_ids": sorted(
            dense_payload_strategy_ids
        ),
        "release_ids": sorted(
            dense_payload_release_ids
        ),
    },
    "point_count_before": (
        dense_exact_count
    ),
    "point_count_after": (
        dense_point_count_after_validation
    ),
    "dense_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "test_queries_accessed": False,
    "paid_api_calls": 0,
    "dense_collection_validation_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        dense_collection_validation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "collection_name": "rag_finance_phase2_dense_v1",
  "storage_path": "data/phase2/index/dense_index_v1/qdrant",
  "point_count": 3488,
  "vector": {
    "dimension": 384,
    "distance": "COSINE",
    "dtype": "float32"
  },
  "embedding": {
    "model_id": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "query_prefix": "query:",
    "passage_prefix": "passage:"
  },
  "payload": {
    "chunk_id_count": 3488,
    "chunk_id_parity": "PASS",
    "source_type_counts": {
      "book": 351,
      "transcript": 2041,
      "website": 1096
    },
    "embedding_revisions": [
      "614241f622f53c4eeff9890bdc4f31cfecc418b3"
    ],
    "strategy_ids": [
      "baseline_phase1_char_safe"
    ],
    "release_ids": [
      "dense_index_v1"
    ]
  },
  "point_count_before": 3488,
  "point_count_after": 3488,
  "dense_index_mutated": false,
  "corpus_embeddings_recomputed": false,
  "test_queries_accessed": false,
  "paid_api_calls": 0,
  "dense_

<div dir="rtl" align="right">

## تولید Cache نسخه‌دار Query Embedding برای Development

برای اجرای Dense Baseline باید Queryها با همان مدل، Revision، Prefix و Normalization فریز‌شده NB14 Embedding شوند.

قواعد این مرحله:

1. فقط Queryهای Development مصرف می‌شوند.
2. Test Queryها همچنان قفل هستند.
3. مدل فقط از Cache محلی Load می‌شود.
4. هیچ API پولی یا سرویس خارجی استفاده نمی‌شود.
5. Embeddingهای Corpus دوباره محاسبه نمی‌شوند.
6. Queryها با Prefix رسمی `query:` کدگذاری می‌شوند.
7. Vectorها L2-normalized و `float32` خواهند بود.
8. Cache با Query-set Hash، Model ID، Revision و Prefix نسخه‌بندی می‌شود.
9. در اجرای مجدد، Cache معتبر Load می‌شود و Embedding دوباره محاسبه نمی‌شود.

این Cache فقط مربوط به Query Embedding است و Dense Index فریز‌شده را تغییر نمی‌دهد.

</div>

In [9]:
# Code Cell 09 — Build or load the Development query embedding cache

HYBRID_ARTIFACT_ROOT = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
)
QUERY_CACHE_ROOT = (
    HYBRID_ARTIFACT_ROOT
    / "query_embedding_cache_v1"
)
DEVELOPMENT_QUERY_EMBEDDINGS_PATH = (
    QUERY_CACHE_ROOT
    / "development_query_embeddings.npy"
)
DEVELOPMENT_QUERY_METADATA_PATH = (
    QUERY_CACHE_ROOT
    / "development_query_embeddings.json"
)

DEVELOPMENT_EMBEDDING_BATCH_SIZE = 64

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

development_candidate_ids_ordered = sorted(
    development_candidate_ids
)

development_query_by_candidate_id = {
    candidate_id: (
        golden_record_by_candidate_id[
            candidate_id
        ]["evaluation_content"]["question"]
    )
    for candidate_id
    in development_candidate_ids_ordered
}

assert len(
    development_query_by_candidate_id
) == 2664

assert all(
    isinstance(question, str)
    and question.strip()
    for question
    in development_query_by_candidate_id.values()
)

development_query_digest = hashlib.sha256()

for candidate_id in (
    development_candidate_ids_ordered
):
    query_component = (
        candidate_id
        + "\x1f"
        + development_query_by_candidate_id[
            candidate_id
        ]
    ).encode("utf-8")

    development_query_digest.update(
        len(query_component).to_bytes(
            8,
            byteorder="big",
            signed=False,
        )
    )

    development_query_digest.update(
        query_component
    )

development_query_set_sha256 = (
    development_query_digest.hexdigest()
)

query_embedding_cache_contract = {
    "schema_version": "1.0",
    "split": "development",
    "split_sha256": split_sha256,
    "query_count": len(
        development_candidate_ids_ordered
    ),
    "query_set_sha256": (
        development_query_set_sha256
    ),
    "model_id": (
        DENSE_EMBEDDING_MODEL_ID
    ),
    "model_revision": (
        DENSE_EMBEDDING_REVISION
    ),
    "query_prefix": (
        DENSE_QUERY_PREFIX
    ),
    "normalize_embeddings": True,
    "dimension": (
        DENSE_VECTOR_DIMENSION
    ),
    "dtype": "float32",
}

query_embedding_cache_key = (
    hashlib.sha256(
        canonical_json_identity(
            query_embedding_cache_contract
        ).encode("utf-8")
    ).hexdigest()
)


def atomic_write_json(
    target_path,
    payload,
):
    target_path = Path(target_path)

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        target_path.with_name(
            target_path.name + ".tmp"
        )
    )

    serialized_payload = (
        json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        + "\n"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as file_handle:
        file_handle.write(
            serialized_payload
        )
        file_handle.flush()
        os.fsync(
            file_handle.fileno()
        )

    os.replace(
        temporary_path,
        target_path,
    )


def atomic_write_numpy(
    target_path,
    array,
):
    target_path = Path(target_path)

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        target_path.with_name(
            target_path.name + ".tmp"
        )
    )

    with temporary_path.open(
        "wb"
    ) as file_handle:
        np.save(
            file_handle,
            array,
            allow_pickle=False,
        )
        file_handle.flush()
        os.fsync(
            file_handle.fileno()
        )

    os.replace(
        temporary_path,
        target_path,
    )


development_cache_hit = False
development_embeddings_recomputed = False
development_cache_files_created = 0
development_embedding_model = None

if (
    DEVELOPMENT_QUERY_EMBEDDINGS_PATH.is_file()
    and DEVELOPMENT_QUERY_METADATA_PATH.is_file()
):
    existing_query_metadata = json.loads(
        DEVELOPMENT_QUERY_METADATA_PATH.read_text(
            encoding="utf-8"
        )
    )

    if (
        existing_query_metadata.get(
            "cache_key"
        )
        == query_embedding_cache_key
        and existing_query_metadata.get(
            "contract"
        )
        == query_embedding_cache_contract
    ):
        development_query_embeddings = (
            np.load(
                DEVELOPMENT_QUERY_EMBEDDINGS_PATH,
                allow_pickle=False,
            )
        )

        assert (
            preflight_sha256_file(
                DEVELOPMENT_QUERY_EMBEDDINGS_PATH
            )
            == existing_query_metadata[
                "embedding_file_sha256"
            ]
        )

        development_cache_hit = True

if not development_cache_hit:
    development_embedding_model = (
        SentenceTransformer(
            DENSE_EMBEDDING_MODEL_ID,
            revision=(
                DENSE_EMBEDDING_REVISION
            ),
            device="cpu",
            local_files_only=True,
        )
    )

    development_query_texts = [
        (
            DENSE_QUERY_PREFIX
            + " "
            + development_query_by_candidate_id[
                candidate_id
            ]
        )
        for candidate_id
        in development_candidate_ids_ordered
    ]

    development_query_embeddings = (
        development_embedding_model.encode(
            development_query_texts,
            batch_size=(
                DEVELOPMENT_EMBEDDING_BATCH_SIZE
            ),
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    )

    development_query_embeddings = (
        np.asarray(
            development_query_embeddings,
            dtype=np.float32,
        )
    )

    atomic_write_numpy(
        DEVELOPMENT_QUERY_EMBEDDINGS_PATH,
        development_query_embeddings,
    )

    development_embedding_file_sha256 = (
        preflight_sha256_file(
            DEVELOPMENT_QUERY_EMBEDDINGS_PATH
        )
    )

    development_query_metadata = {
        "artifact_type": (
            "NB16_DEVELOPMENT_QUERY_"
            "EMBEDDING_CACHE"
        ),
        "schema_version": "1.0",
        "cache_key": (
            query_embedding_cache_key
        ),
        "contract": (
            query_embedding_cache_contract
        ),
        "candidate_ids": (
            development_candidate_ids_ordered
        ),
        "embedding_path": str(
            DEVELOPMENT_QUERY_EMBEDDINGS_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "embedding_file_sha256": (
            development_embedding_file_sha256
        ),
        "paid_api_calls": 0,
        "cache_status": "VALID",
    }

    atomic_write_json(
        DEVELOPMENT_QUERY_METADATA_PATH,
        development_query_metadata,
    )

    development_embeddings_recomputed = True
    development_cache_files_created = 2

assert development_query_embeddings.shape == (
    2664,
    DENSE_VECTOR_DIMENSION,
)

assert (
    development_query_embeddings.dtype
    == np.float32
)

development_embedding_norms = np.linalg.norm(
    development_query_embeddings,
    axis=1,
)

assert np.allclose(
    development_embedding_norms,
    1.0,
    rtol=1e-5,
    atol=1e-6,
)

final_development_query_metadata = (
    json.loads(
        DEVELOPMENT_QUERY_METADATA_PATH.read_text(
            encoding="utf-8"
        )
    )
)

assert (
    final_development_query_metadata[
        "cache_key"
    ]
    == query_embedding_cache_key
)

assert (
    final_development_query_metadata[
        "contract"
    ]
    == query_embedding_cache_contract
)

assert (
    preflight_sha256_file(
        DEVELOPMENT_QUERY_EMBEDDINGS_PATH
    )
    == final_development_query_metadata[
        "embedding_file_sha256"
    ]
)

assert (
    final_development_query_metadata[
        "candidate_ids"
    ]
    == development_candidate_ids_ordered
)

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

development_query_embedding_result = {
    "split": "development",
    "query_count": len(
        development_candidate_ids_ordered
    ),
    "query_set_sha256": (
        development_query_set_sha256
    ),
    "cache_key": (
        query_embedding_cache_key
    ),
    "model": {
        "model_id": (
            DENSE_EMBEDDING_MODEL_ID
        ),
        "revision": (
            DENSE_EMBEDDING_REVISION
        ),
        "query_prefix": (
            DENSE_QUERY_PREFIX
        ),
        "normalize_embeddings": True,
    },
    "embedding": {
        "shape": list(
            development_query_embeddings.shape
        ),
        "dtype": str(
            development_query_embeddings.dtype
        ),
        "minimum_norm": float(
            development_embedding_norms.min()
        ),
        "maximum_norm": float(
            development_embedding_norms.max()
        ),
    },
    "cache": {
        "embedding_path": str(
            DEVELOPMENT_QUERY_EMBEDDINGS_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "metadata_path": str(
            DEVELOPMENT_QUERY_METADATA_PATH
            .relative_to(PROJECT_ROOT)
        ),
        "embedding_sha256": (
            preflight_sha256_file(
                DEVELOPMENT_QUERY_EMBEDDINGS_PATH
            )
        ),
        "metadata_sha256": (
            preflight_sha256_file(
                DEVELOPMENT_QUERY_METADATA_PATH
            )
        ),
        "cache_hit": (
            development_cache_hit
        ),
    },
    "test_queries_accessed": False,
    "corpus_embeddings_recomputed": False,
    "query_embeddings_recomputed": (
        development_embeddings_recomputed
    ),
    "files_created": (
        development_cache_files_created
    ),
    "paid_api_calls": 0,
    "development_query_embedding_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        development_query_embedding_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "split": "development",
  "query_count": 2664,
  "query_set_sha256": "80f5380c09974feb690225ce66e9c5f90aa3ca8b84090ef726c5a2e8caf06087",
  "cache_key": "fc953162e295a25cfb34a3a750e817bab39578b3db430da42aa929a0d8ebae74",
  "model": {
    "model_id": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "query_prefix": "query:",
    "normalize_embeddings": true
  },
  "embedding": {
    "shape": [
      2664,
      384
    ],
    "dtype": "float32",
    "minimum_norm": 0.9999998807907104,
    "maximum_norm": 1.0000001192092896
  },
  "cache": {
    "embedding_path": "artifacts/phase2/hybrid_retrieval/query_embedding_cache_v1/development_query_embeddings.npy",
    "metadata_path": "artifacts/phase2/hybrid_retrieval/query_embedding_cache_v1/development_query_embeddings.json",
    "embedding_sha256": "ca2641a2b8a484433a85ae0682b9b600c02e0c7e4e670af4e07eeb61099e4bb5",
    "metadata_sha256": "bc97f189d5cadb801f5aaaad9a86715509ad2a91a18dc4840b091

<div dir="rtl" align="right">

## اجرای Dense و Sparse Baseline روی Development

پیش از آزمایش Hybrid باید عملکرد مستقل هر Retriever با یک Evaluation Harness مشترک اندازه‌گیری شود.

در این مرحله:

1. Dense Retrieval از Collection فریز‌شده Qdrant اجرا می‌شود.
2. Sparse Retrieval از BM25 فریز‌شده NB15 اجرا می‌شود.
3. برای هر Query حداکثر ۱۰۰ Candidate نگهداری می‌شود.
4. Metricهای Top-5 و Top-10 با Gold Mapping مشترک محاسبه می‌شوند.
5. Latency هر Retriever جداگانه ثبت می‌شود.
6. Sparse Baseline باید با Metricهای Development ثبت‌شده در NB15 برابر باشد.
7. نتایج رتبه‌ای و Scoreهای خام برای RRF و Weighted Fusion نگهداری می‌شوند.
8. هیچ پارامتر Hybrid در این مرحله انتخاب نمی‌شود.
9. Test Split همچنان قفل است.

Candidate Depth برابر ۱۰۰ فقط Pool آزمایشی را می‌سازد؛ Top-k نهایی و Depth نهایی در مراحل بعد روی Development انتخاب خواهند شد.

</div>

In [10]:
# Code Cell 10 — Run fair Development Dense and Sparse baselines

MAX_CANDIDATE_DEPTH = 100
BASELINE_TOP_K_VALUES = (5, 10)

EXPECTED_NB15_DEVELOPMENT_METRICS = {
    "hit_at_10": 0.743993993993994,
    "hit_at_5": 0.6786786786786787,
    "ndcg_at_10": 0.50936500655395,
    "ndcg_at_5": 0.4855041074074556,
    "precision_at_10": (
        0.1001876876876877
    ),
    "precision_at_5": (
        0.17432432432432432
    ),
    "recall_at_10": (
        0.5796647838314506
    ),
    "recall_at_5": (
        0.5128710555793888
    ),
    "reciprocal_rank": (
        0.5861315247868337
    ),
}

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED


def score_sparse_query(
    index,
    query_tokens,
    k1,
    b,
):
    assert k1 > 0
    assert 0 <= b <= 1

    scores = np.zeros(
        len(index.document_ids),
        dtype=np.float64,
    )

    query_term_counts = (
        collections.Counter(
            query_tokens
        )
    )

    document_count = len(
        index.document_ids
    )

    for term, query_term_frequency in (
        query_term_counts.items()
    ):
        posting = index.postings.get(
            term
        )

        if posting is None:
            continue

        (
            document_indexes,
            term_frequencies,
        ) = posting

        document_frequency = len(
            document_indexes
        )

        inverse_document_frequency = (
            math.log(
                1.0
                + (
                    (
                        document_count
                        - document_frequency
                        + 0.5
                    )
                    / (
                        document_frequency
                        + 0.5
                    )
                )
            )
        )

        length_normalization = (
            1.0
            - b
            + b
            * (
                index.document_lengths[
                    document_indexes
                ]
                / index.average_document_length
            )
        )

        term_score = (
            inverse_document_frequency
            * (
                term_frequencies
                * (k1 + 1.0)
            )
            / (
                term_frequencies
                + k1
                * length_normalization
            )
        )

        scores[
            document_indexes
        ] += (
            term_score
            * query_term_frequency
        )

    assert np.all(
        np.isfinite(scores)
    )
    assert np.all(scores >= 0)

    return scores


def deterministic_top_indexes(
    scores,
    limit=None,
    positive_only=True,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    if positive_only:
        candidate_indexes = np.flatnonzero(
            scores > 0
        )
    else:
        candidate_indexes = np.arange(
            len(scores),
            dtype=np.int64,
        )

    if not len(candidate_indexes):
        return np.asarray(
            [],
            dtype=np.int32,
        )

    ordering = np.lexsort(
        (
            candidate_indexes,
            -scores[candidate_indexes],
        )
    )

    ranked_indexes = (
        candidate_indexes[ordering]
    )

    if limit is not None:
        ranked_indexes = (
            ranked_indexes[:limit]
        )

    return ranked_indexes


def evaluate_ranked_ids(
    ranked_document_ids,
    relevant_document_ids,
    k_values=BASELINE_TOP_K_VALUES,
):
    relevant_document_ids = set(
        relevant_document_ids
    )

    assert relevant_document_ids

    relevant_flags = np.asarray(
        [
            document_id
            in relevant_document_ids
            for document_id
            in ranked_document_ids
        ],
        dtype=bool,
    )

    result = {}

    first_relevant_positions = (
        np.flatnonzero(
            relevant_flags
        )
    )

    result["reciprocal_rank"] = (
        1.0
        / (
            int(
                first_relevant_positions[0]
            )
            + 1
        )
        if len(first_relevant_positions)
        else 0.0
    )

    for k in k_values:
        top_flags = relevant_flags[:k]

        relevant_retrieved = int(
            top_flags.sum()
        )

        result[f"hit_at_{k}"] = float(
            relevant_retrieved > 0
        )

        result[f"recall_at_{k}"] = (
            relevant_retrieved
            / len(relevant_document_ids)
        )

        result[f"precision_at_{k}"] = (
            relevant_retrieved / k
        )

        discounts = (
            1.0
            / np.log2(
                np.arange(
                    2,
                    2 + len(top_flags),
                    dtype=np.float64,
                )
            )
        )

        discounted_gain = float(
            np.sum(
                top_flags.astype(
                    np.float64
                )
                * discounts
            )
        )

        ideal_length = min(
            len(relevant_document_ids),
            k,
        )

        ideal_discounted_gain = float(
            np.sum(
                1.0
                / np.log2(
                    np.arange(
                        2,
                        2 + ideal_length,
                        dtype=np.float64,
                    )
                )
            )
        )

        result[f"ndcg_at_{k}"] = (
            discounted_gain
            / ideal_discounted_gain
            if ideal_discounted_gain > 0
            else 0.0
        )

    return result


def aggregate_metric_rows(
    metric_rows,
):
    metric_names = sorted(
        metric_rows[0]
    )

    return {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row in metric_rows
                ]
            )
        )
        for metric_name in metric_names
    }


def summarize_latency(
    latencies_ms,
    scope,
):
    latency_array = np.asarray(
        latencies_ms,
        dtype=np.float64,
    )

    return {
        "minimum": float(
            latency_array.min()
        ),
        "median_p50": float(
            np.percentile(
                latency_array,
                50,
            )
        ),
        "p95": float(
            np.percentile(
                latency_array,
                95,
            )
        ),
        "maximum": float(
            latency_array.max()
        ),
        "scope": scope,
    }


development_sparse_query_tokens = {
    candidate_id: (
        analyze_frozen_sparse_text(
            development_query_by_candidate_id[
                candidate_id
            ]
        )
    )
    for candidate_id
    in development_candidate_ids_ordered
}

assert all(
    development_sparse_query_tokens[
        candidate_id
    ]
    for candidate_id
    in development_candidate_ids_ordered
)

development_dense_results = {}
development_sparse_results = {}

dense_metric_rows = []
sparse_pool_metric_rows = []
sparse_full_reference_metric_rows = []

dense_latencies_ms = []
sparse_latencies_ms = []

dense_zero_result_count = 0
sparse_zero_result_count = 0

for query_position, candidate_id in enumerate(
    development_candidate_ids_ordered
):
    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    query_vector = (
        development_query_embeddings[
            query_position
        ]
    )

    dense_started = time.perf_counter()

    dense_query_response = (
        qdrant_client.query_points(
            collection_name=(
                DENSE_COLLECTION_NAME
            ),
            query=query_vector.tolist(),
            limit=MAX_CANDIDATE_DEPTH,
            with_payload=True,
            with_vectors=False,
        )
    )

    dense_latency_ms = (
        time.perf_counter()
        - dense_started
    ) * 1000.0

    dense_latencies_ms.append(
        dense_latency_ms
    )

    dense_points = list(
        dense_query_response.points
    )

    dense_ranked_chunk_ids = []
    dense_scores = []

    for point in dense_points:
        payload = dict(
            point.payload or {}
        )

        chunk_id = str(
            payload["chunk_id"]
        )

        assert chunk_id in chunk_ids_set

        dense_ranked_chunk_ids.append(
            chunk_id
        )
        dense_scores.append(
            float(point.score)
        )

    assert len(
        dense_ranked_chunk_ids
    ) == len(
        set(dense_ranked_chunk_ids)
    )

    if not dense_ranked_chunk_ids:
        dense_zero_result_count += 1

    dense_metrics = evaluate_ranked_ids(
        ranked_document_ids=(
            dense_ranked_chunk_ids
        ),
        relevant_document_ids=(
            relevant_chunk_ids
        ),
    )

    dense_metric_rows.append(
        dense_metrics
    )

    development_dense_results[
        candidate_id
    ] = {
        "ranked_chunk_ids": (
            dense_ranked_chunk_ids
        ),
        "scores": dense_scores,
        "latency_ms": float(
            dense_latency_ms
        ),
        "metrics": dense_metrics,
    }

    sparse_started = time.perf_counter()

    sparse_scores_all = (
        score_sparse_query(
            index=frozen_sparse_index,
            query_tokens=(
                development_sparse_query_tokens[
                    candidate_id
                ]
            ),
            k1=FROZEN_SPARSE_K1,
            b=FROZEN_SPARSE_B,
        )
    )

    sparse_ranked_indexes_full = (
        deterministic_top_indexes(
            scores=sparse_scores_all,
            limit=None,
            positive_only=True,
        )
    )

    sparse_latency_ms = (
        time.perf_counter()
        - sparse_started
    ) * 1000.0

    sparse_latencies_ms.append(
        sparse_latency_ms
    )

    sparse_ranked_chunk_ids_full = [
        frozen_sparse_index.document_ids[
            document_index
        ]
        for document_index
        in sparse_ranked_indexes_full
    ]

    sparse_ranked_indexes_pool = (
        sparse_ranked_indexes_full[
            :MAX_CANDIDATE_DEPTH
        ]
    )

    sparse_ranked_chunk_ids_pool = [
        frozen_sparse_index.document_ids[
            document_index
        ]
        for document_index
        in sparse_ranked_indexes_pool
    ]

    sparse_scores_pool = [
        float(
            sparse_scores_all[
                document_index
            ]
        )
        for document_index
        in sparse_ranked_indexes_pool
    ]

    if not sparse_ranked_chunk_ids_full:
        sparse_zero_result_count += 1

    sparse_full_reference_metrics = (
        evaluate_ranked_ids(
            ranked_document_ids=(
                sparse_ranked_chunk_ids_full
            ),
            relevant_document_ids=(
                relevant_chunk_ids
            ),
        )
    )

    sparse_pool_metrics = (
        evaluate_ranked_ids(
            ranked_document_ids=(
                sparse_ranked_chunk_ids_pool
            ),
            relevant_document_ids=(
                relevant_chunk_ids
            ),
        )
    )

    sparse_full_reference_metric_rows.append(
        sparse_full_reference_metrics
    )

    sparse_pool_metric_rows.append(
        sparse_pool_metrics
    )

    development_sparse_results[
        candidate_id
    ] = {
        "ranked_chunk_ids": (
            sparse_ranked_chunk_ids_pool
        ),
        "scores": (
            sparse_scores_pool
        ),
        "positive_result_count": len(
            sparse_ranked_chunk_ids_full
        ),
        "latency_ms": float(
            sparse_latency_ms
        ),
        "metrics": (
            sparse_pool_metrics
        ),
        "nb15_reference_metrics": (
            sparse_full_reference_metrics
        ),
    }

dense_development_metrics = (
    aggregate_metric_rows(
        dense_metric_rows
    )
)

sparse_development_metrics = (
    aggregate_metric_rows(
        sparse_pool_metric_rows
    )
)

sparse_nb15_reference_metrics = (
    aggregate_metric_rows(
        sparse_full_reference_metric_rows
    )
)

sparse_metric_deltas_from_nb15 = {
    metric_name: float(
        sparse_nb15_reference_metrics[
            metric_name
        ]
        - expected_value
    )
    for metric_name, expected_value
    in EXPECTED_NB15_DEVELOPMENT_METRICS.items()
}

assert all(
    abs(delta) <= 1e-15
    for delta
    in sparse_metric_deltas_from_nb15.values()
)

sparse_pool_minus_full_reference = {
    metric_name: float(
        sparse_development_metrics[
            metric_name
        ]
        - sparse_nb15_reference_metrics[
            metric_name
        ]
    )
    for metric_name
    in sparse_development_metrics
}

assert all(
    sparse_pool_minus_full_reference[
        metric_name
    ] == 0.0
    for metric_name in (
        "hit_at_5",
        "hit_at_10",
        "precision_at_5",
        "precision_at_10",
        "recall_at_5",
        "recall_at_10",
        "ndcg_at_5",
        "ndcg_at_10",
    )
)

assert (
    sparse_pool_minus_full_reference[
        "reciprocal_rank"
    ]
    <= 0.0
)

assert abs(
    sparse_pool_minus_full_reference[
        "reciprocal_rank"
    ]
) < 0.001

assert len(
    development_dense_results
) == 2664

assert len(
    development_sparse_results
) == 2664

assert set(
    development_dense_results
) == development_candidate_ids

assert set(
    development_sparse_results
) == development_candidate_ids

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

development_baseline_result = {
    "scope": {
        "split": "development",
        "query_count": len(
            development_candidate_ids
        ),
        "candidate_depth": (
            MAX_CANDIDATE_DEPTH
        ),
        "test_queries_accessed": False,
    },
    "dense": {
        "collection_name": (
            DENSE_COLLECTION_NAME
        ),
        "metrics_at_candidate_depth_100": (
            dense_development_metrics
        ),
        "latency_ms": (
            summarize_latency(
                dense_latencies_ms,
                (
                    "LOCAL_QDRANT_RETRIEVAL_"
                    "EXCLUDING_QUERY_EMBEDDING"
                ),
            )
        ),
        "zero_result_query_count": (
            dense_zero_result_count
        ),
    },
    "sparse": {
        "analyzer_id": (
            frozen_sparse_index.analyzer_id
        ),
        "k1": FROZEN_SPARSE_K1,
        "b": FROZEN_SPARSE_B,
        "metrics_at_candidate_depth_100": (
            sparse_development_metrics
        ),
        "nb15_full_positive_ranking_metrics": (
            sparse_nb15_reference_metrics
        ),
        "candidate_pool_minus_full_reference": (
            sparse_pool_minus_full_reference
        ),
        "latency_ms": (
            summarize_latency(
                sparse_latencies_ms,
                (
                    "LOCAL_BM25_RETRIEVAL_"
                    "INCLUDING_FULL_POSITIVE_SORT"
                ),
            )
        ),
        "zero_result_query_count": (
            sparse_zero_result_count
        ),
        "nb15_metric_parity": "PASS",
        "maximum_absolute_nb15_metric_delta": float(
            max(
                abs(delta)
                for delta
                in sparse_metric_deltas_from_nb15.values()
            )
        ),
    },
    "fair_comparison_contract": {
        "dense_candidate_depth": (
            MAX_CANDIDATE_DEPTH
        ),
        "sparse_candidate_depth": (
            MAX_CANDIDATE_DEPTH
        ),
        "fusion_candidate_depth_maximum": (
            MAX_CANDIDATE_DEPTH
        ),
        "top_k_metrics_unaffected_by_"
        "sparse_depth_truncation": True,
        "mrr_scope": (
            "Candidate-depth-100 for fair "
            "dense/sparse/hybrid comparison; "
            "full positive sparse ranking retained "
            "only for NB15 parity validation."
        ),
    },
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "baseline_status": "PASS",
}

print(
    json.dumps(
        development_baseline_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 2664,
    "candidate_depth": 100,
    "test_queries_accessed": false
  },
  "dense": {
    "collection_name": "rag_finance_phase2_dense_v1",
    "metrics_at_candidate_depth_100": {
      "hit_at_10": 0.609984984984985,
      "hit_at_5": 0.5213963963963963,
      "ndcg_at_10": 0.37395705045915284,
      "ndcg_at_5": 0.34434011301495415,
      "precision_at_10": 0.0826951951951952,
      "precision_at_5": 0.1352102102102102,
      "recall_at_10": 0.4735470589637256,
      "recall_at_5": 0.39174799799799803,
      "reciprocal_rank": 0.4111605316726538
    },
    "latency_ms": {
      "minimum": 16.776244001448504,
      "median_p50": 36.10901050069515,
      "p95": 53.26214849992539,
      "maximum": 542.9196669992962,
      "scope": "LOCAL_QDRANT_RETRIEVAL_EXCLUDING_QUERY_EMBEDDING"
    },
    "zero_result_query_count": 0
  },
  "sparse": {
    "analyzer_id": "fa_join_zwnj_stopwords_v1",
    "k1": 1.2,
    "b": 0.3,
    "metri

<div dir="rtl" align="right">

## آزمایش Reciprocal Rank Fusion روی Development

RRF بدون نیاز به هم‌مقیاس‌کردن Scoreهای Dense و Sparse، رتبه هر سند را در دو Retriever ترکیب می‌کند.

فرمول استفاده‌شده:

\[
RRF(d)=\sum_{r \in \{dense,sparse\}}\frac{1}{k+r_r(d)}
\]

Experiment Grid شامل موارد زیر است:

- Candidate Depth: ۲۰، ۵۰ و ۱۰۰
- ثابت RRF: ۱۰، ۳۰، ۶۰ و ۱۰۰
- تعداد پیکربندی‌ها: ۱۲
- معیار اصلی انتخاب: `nDCG@10`
- Tie-breakها: MRR، Hit@10، Recall@10 و سپس Configuration ID

رتبه‌بندی در حالت تساوی Score با ترتیب Canonical Document Index شکسته می‌شود تا خروجی کاملاً قطعی باشد.

این مرحله فقط روی Development اجرا می‌شود و Test همچنان قفل است.

</div>

In [11]:
# Code Cell 11 — Evaluate the Development RRF experiment grid

RRF_CANDIDATE_DEPTHS = (
    20,
    50,
    100,
)
RRF_K_VALUES = (
    10,
    30,
    60,
    100,
)

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

assert max(
    RRF_CANDIDATE_DEPTHS
) <= MAX_CANDIDATE_DEPTH


def fuse_rrf_rankings(
    dense_ranked_chunk_ids,
    sparse_ranked_chunk_ids,
    candidate_depth,
    rrf_k,
):
    assert candidate_depth > 0
    assert rrf_k > 0

    fused_scores = (
        collections.defaultdict(float)
    )

    dense_candidates = (
        dense_ranked_chunk_ids[
            :candidate_depth
        ]
    )

    sparse_candidates = (
        sparse_ranked_chunk_ids[
            :candidate_depth
        ]
    )

    for rank_position, chunk_id in enumerate(
        dense_candidates,
        start=1,
    ):
        fused_scores[chunk_id] += (
            1.0
            / (
                rrf_k
                + rank_position
            )
        )

    for rank_position, chunk_id in enumerate(
        sparse_candidates,
        start=1,
    ):
        fused_scores[chunk_id] += (
            1.0
            / (
                rrf_k
                + rank_position
            )
        )

    ranked_chunk_ids = sorted(
        fused_scores,
        key=lambda chunk_id: (
            -fused_scores[chunk_id],
            chunk_index_by_id[chunk_id],
        ),
    )

    ranked_scores = [
        float(
            fused_scores[chunk_id]
        )
        for chunk_id in ranked_chunk_ids
    ]

    return (
        ranked_chunk_ids,
        ranked_scores,
    )


development_rrf_summaries = []
development_rrf_results_by_config = {}

for candidate_depth in (
    RRF_CANDIDATE_DEPTHS
):
    for rrf_k in RRF_K_VALUES:
        configuration_id = (
            "rrf"
            f"__depth_{candidate_depth}"
            f"__k_{rrf_k}"
        )

        configuration_metric_rows = []
        configuration_query_results = {}
        fusion_latencies_ms = []

        for candidate_id in (
            development_candidate_ids_ordered
        ):
            dense_ranked_chunk_ids = (
                development_dense_results[
                    candidate_id
                ]["ranked_chunk_ids"]
            )

            sparse_ranked_chunk_ids = (
                development_sparse_results[
                    candidate_id
                ]["ranked_chunk_ids"]
            )

            fusion_started = (
                time.perf_counter()
            )

            (
                fused_ranked_chunk_ids,
                fused_scores,
            ) = fuse_rrf_rankings(
                dense_ranked_chunk_ids=(
                    dense_ranked_chunk_ids
                ),
                sparse_ranked_chunk_ids=(
                    sparse_ranked_chunk_ids
                ),
                candidate_depth=(
                    candidate_depth
                ),
                rrf_k=rrf_k,
            )

            fusion_latency_ms = (
                time.perf_counter()
                - fusion_started
            ) * 1000.0

            fusion_latencies_ms.append(
                fusion_latency_ms
            )

            relevant_chunk_ids = set(
                gold_mapping_by_candidate_id[
                    candidate_id
                ]
            )

            query_metrics = (
                evaluate_ranked_ids(
                    ranked_document_ids=(
                        fused_ranked_chunk_ids
                    ),
                    relevant_document_ids=(
                        relevant_chunk_ids
                    ),
                )
            )

            configuration_metric_rows.append(
                query_metrics
            )

            configuration_query_results[
                candidate_id
            ] = {
                "ranked_chunk_ids": (
                    fused_ranked_chunk_ids
                ),
                "scores": fused_scores,
                "metrics": query_metrics,
            }

        aggregate_metrics = (
            aggregate_metric_rows(
                configuration_metric_rows
            )
        )

        configuration_summary = {
            "configuration_id": (
                configuration_id
            ),
            "fusion_method": "RRF",
            "candidate_depth": int(
                candidate_depth
            ),
            "rrf_k": int(rrf_k),
            "query_count": len(
                configuration_metric_rows
            ),
            "metrics": (
                aggregate_metrics
            ),
            "fusion_latency_ms": (
                summarize_latency(
                    fusion_latencies_ms,
                    (
                        "LOCAL_FUSION_ONLY_"
                        "EXCLUDING_RETRIEVAL"
                    ),
                )
            ),
            "status": "PASS",
        }

        development_rrf_summaries.append(
            configuration_summary
        )

        development_rrf_results_by_config[
            configuration_id
        ] = configuration_query_results

development_rrf_summaries = sorted(
    development_rrf_summaries,
    key=lambda row: (
        -row["metrics"]["ndcg_at_10"],
        -row["metrics"]["reciprocal_rank"],
        -row["metrics"]["hit_at_10"],
        -row["metrics"]["recall_at_10"],
        row["configuration_id"],
    ),
)

best_rrf_summary = (
    development_rrf_summaries[0]
)

best_rrf_configuration_id = (
    best_rrf_summary[
        "configuration_id"
    ]
)

best_rrf_results_by_candidate_id = (
    development_rrf_results_by_config[
        best_rrf_configuration_id
    ]
)

best_baseline_name = max(
    (
        "dense",
        "sparse",
    ),
    key=lambda baseline_name: (
        (
            dense_development_metrics
            if baseline_name == "dense"
            else sparse_development_metrics
        )["ndcg_at_10"]
    ),
)

best_baseline_metrics = (
    dense_development_metrics
    if best_baseline_name == "dense"
    else sparse_development_metrics
)

best_rrf_delta_vs_best_baseline = {
    metric_name: float(
        best_rrf_summary[
            "metrics"
        ][metric_name]
        - best_baseline_metrics[
            metric_name
        ]
    )
    for metric_name
    in best_rrf_summary["metrics"]
}

assert len(
    development_rrf_summaries
) == (
    len(RRF_CANDIDATE_DEPTHS)
    * len(RRF_K_VALUES)
)

assert len(
    best_rrf_results_by_candidate_id
) == len(
    development_candidate_ids
)

assert set(
    best_rrf_results_by_candidate_id
) == development_candidate_ids

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

rrf_development_result = {
    "scope": {
        "split": "development",
        "query_count": len(
            development_candidate_ids
        ),
        "test_queries_accessed": False,
    },
    "grid": {
        "candidate_depths": list(
            RRF_CANDIDATE_DEPTHS
        ),
        "rrf_k_values": list(
            RRF_K_VALUES
        ),
        "configuration_count": len(
            development_rrf_summaries
        ),
    },
    "ranking_policy": {
        "dense_weight": 1.0,
        "sparse_weight": 1.0,
        "tie_break": (
            "CANONICAL_DOCUMENT_INDEX_ASCENDING"
        ),
    },
    "best_configuration": (
        best_rrf_summary
    ),
    "best_baseline": {
        "name": best_baseline_name,
        "metrics": best_baseline_metrics,
    },
    "best_rrf_delta_vs_best_baseline": (
        best_rrf_delta_vs_best_baseline
    ),
    "ranked_configurations": (
        development_rrf_summaries
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "rrf_grid_status": "PASS",
}

print(
    json.dumps(
        rrf_development_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 2664,
    "test_queries_accessed": false
  },
  "grid": {
    "candidate_depths": [
      20,
      50,
      100
    ],
    "rrf_k_values": [
      10,
      30,
      60,
      100
    ],
    "configuration_count": 12
  },
  "ranking_policy": {
    "dense_weight": 1.0,
    "sparse_weight": 1.0,
    "tie_break": "CANONICAL_DOCUMENT_INDEX_ASCENDING"
  },
  "best_configuration": {
    "configuration_id": "rrf__depth_100__k_10",
    "fusion_method": "RRF",
    "candidate_depth": 100,
    "rrf_k": 10,
    "query_count": 2664,
    "metrics": {
      "hit_at_10": 0.7826576576576577,
      "hit_at_5": 0.704954954954955,
      "ndcg_at_10": 0.5141238334805899,
      "ndcg_at_5": 0.48603822160934074,
      "precision_at_10": 0.10743243243243245,
      "precision_at_5": 0.18423423423423424,
      "recall_at_10": 0.6195819331235998,
      "recall_at_5": 0.5423264037847371,
      "reciprocal_rank": 0.5671843718121118
    },
    "fusion

<div dir="rtl" align="right">

## آزمایش Weighted Score Fusion روی Development

در Weighted Fusion، Scoreهای Dense و Sparse پس از نرمال‌سازی در سطح هر Query ترکیب می‌شوند:

\[
Score(d)=w_d \times DenseNorm(d)+(1-w_d)\times SparseNorm(d)
\]

به‌دلیل تفاوت مقیاس Cosine Similarity و BM25، استفاده مستقیم از Score خام مجاز نیست.

Experiment Grid شامل موارد زیر است:

- Candidate Depth: ۲۰، ۵۰ و ۱۰۰
- Dense Weight: ۰٫۱، ۰٫۲، ۰٫۳، ۰٫۴، ۰٫۵ و ۰٫۶
- Sparse Weight: مکمل Dense Weight
- Normalization:
  - Max normalization
  - Min-Max normalization
- تعداد پیکربندی‌ها: ۳۶
- معیار اصلی: `nDCG@10`
- Tie-breakها: MRR، Hit@10، Recall@10 و Configuration ID

نتیجه Weighted Fusion با بهترین RRF و بهترین Baseline مقایسه خواهد شد. Test Split همچنان قفل است.

</div>

In [12]:
# Code Cell 12 — Evaluate the Development Weighted Fusion grid

WEIGHTED_CANDIDATE_DEPTHS = (
    20,
    50,
    100,
)
DENSE_WEIGHT_VALUES = (
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
)
WEIGHTED_NORMALIZATION_METHODS = (
    "max",
    "minmax",
)

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED


def normalize_retrieval_scores(
    scores,
    method,
):
    score_array = np.asarray(
        scores,
        dtype=np.float64,
    )

    assert score_array.ndim == 1
    assert np.all(
        np.isfinite(score_array)
    )

    if not len(score_array):
        return score_array

    minimum_score = float(
        score_array.min()
    )
    maximum_score = float(
        score_array.max()
    )

    if method == "max":
        if maximum_score <= 0:
            return np.zeros_like(
                score_array
            )

        normalized_scores = (
            score_array
            / maximum_score
        )

    elif method == "minmax":
        score_range = (
            maximum_score
            - minimum_score
        )

        if score_range <= 1e-15:
            normalized_scores = np.ones_like(
                score_array
            )
        else:
            normalized_scores = (
                score_array
                - minimum_score
            ) / score_range

    else:
        raise AssertionError(
            "Unknown weighted normalization "
            f"method: {method}"
        )

    assert np.all(
        np.isfinite(normalized_scores)
    )

    return normalized_scores


def fuse_weighted_rankings(
    dense_ranked_chunk_ids,
    dense_scores,
    sparse_ranked_chunk_ids,
    sparse_scores,
    candidate_depth,
    dense_weight,
    normalization_method,
):
    assert 0.0 <= dense_weight <= 1.0
    assert candidate_depth > 0

    sparse_weight = (
        1.0 - dense_weight
    )

    dense_candidate_ids = (
        dense_ranked_chunk_ids[
            :candidate_depth
        ]
    )
    dense_candidate_scores = (
        dense_scores[
            :candidate_depth
        ]
    )

    sparse_candidate_ids = (
        sparse_ranked_chunk_ids[
            :candidate_depth
        ]
    )
    sparse_candidate_scores = (
        sparse_scores[
            :candidate_depth
        ]
    )

    normalized_dense_scores = (
        normalize_retrieval_scores(
            dense_candidate_scores,
            normalization_method,
        )
    )

    normalized_sparse_scores = (
        normalize_retrieval_scores(
            sparse_candidate_scores,
            normalization_method,
        )
    )

    fused_scores = (
        collections.defaultdict(float)
    )

    for chunk_id, normalized_score in zip(
        dense_candidate_ids,
        normalized_dense_scores,
    ):
        fused_scores[chunk_id] += (
            dense_weight
            * float(normalized_score)
        )

    for chunk_id, normalized_score in zip(
        sparse_candidate_ids,
        normalized_sparse_scores,
    ):
        fused_scores[chunk_id] += (
            sparse_weight
            * float(normalized_score)
        )

    ranked_chunk_ids = sorted(
        fused_scores,
        key=lambda chunk_id: (
            -fused_scores[chunk_id],
            chunk_index_by_id[chunk_id],
        ),
    )

    ranked_scores = [
        float(
            fused_scores[chunk_id]
        )
        for chunk_id in ranked_chunk_ids
    ]

    return (
        ranked_chunk_ids,
        ranked_scores,
    )


def evaluate_weighted_configuration(
    candidate_depth,
    dense_weight,
    normalization_method,
    retain_query_results=False,
):
    configuration_id = (
        "weighted"
        f"__norm_{normalization_method}"
        f"__depth_{candidate_depth}"
        f"__dense_{dense_weight:.2f}"
        f"__sparse_{1.0 - dense_weight:.2f}"
    )

    metric_rows = []
    fusion_latencies_ms = []

    query_results = (
        {}
        if retain_query_results
        else None
    )

    for candidate_id in (
        development_candidate_ids_ordered
    ):
        dense_result = (
            development_dense_results[
                candidate_id
            ]
        )
        sparse_result = (
            development_sparse_results[
                candidate_id
            ]
        )

        fusion_started = (
            time.perf_counter()
        )

        (
            fused_ranked_chunk_ids,
            fused_scores,
        ) = fuse_weighted_rankings(
            dense_ranked_chunk_ids=(
                dense_result[
                    "ranked_chunk_ids"
                ]
            ),
            dense_scores=(
                dense_result["scores"]
            ),
            sparse_ranked_chunk_ids=(
                sparse_result[
                    "ranked_chunk_ids"
                ]
            ),
            sparse_scores=(
                sparse_result["scores"]
            ),
            candidate_depth=(
                candidate_depth
            ),
            dense_weight=(
                dense_weight
            ),
            normalization_method=(
                normalization_method
            ),
        )

        fusion_latencies_ms.append(
            (
                time.perf_counter()
                - fusion_started
            )
            * 1000.0
        )

        query_metrics = evaluate_ranked_ids(
            ranked_document_ids=(
                fused_ranked_chunk_ids
            ),
            relevant_document_ids=(
                gold_mapping_by_candidate_id[
                    candidate_id
                ]
            ),
        )

        metric_rows.append(
            query_metrics
        )

        if retain_query_results:
            query_results[
                candidate_id
            ] = {
                "ranked_chunk_ids": (
                    fused_ranked_chunk_ids
                ),
                "scores": (
                    fused_scores
                ),
                "metrics": (
                    query_metrics
                ),
            }

    summary = {
        "configuration_id": (
            configuration_id
        ),
        "fusion_method": (
            "WEIGHTED_SCORE_FUSION"
        ),
        "normalization_method": (
            normalization_method
        ),
        "candidate_depth": int(
            candidate_depth
        ),
        "dense_weight": float(
            dense_weight
        ),
        "sparse_weight": float(
            1.0 - dense_weight
        ),
        "query_count": len(
            metric_rows
        ),
        "metrics": (
            aggregate_metric_rows(
                metric_rows
            )
        ),
        "fusion_latency_ms": (
            summarize_latency(
                fusion_latencies_ms,
                (
                    "LOCAL_FUSION_ONLY_"
                    "EXCLUDING_RETRIEVAL"
                ),
            )
        ),
        "status": "PASS",
    }

    return summary, query_results


development_weighted_summaries = []

for normalization_method in (
    WEIGHTED_NORMALIZATION_METHODS
):
    for candidate_depth in (
        WEIGHTED_CANDIDATE_DEPTHS
    ):
        for dense_weight in (
            DENSE_WEIGHT_VALUES
        ):
            (
                weighted_summary,
                _,
            ) = evaluate_weighted_configuration(
                candidate_depth=(
                    candidate_depth
                ),
                dense_weight=(
                    dense_weight
                ),
                normalization_method=(
                    normalization_method
                ),
                retain_query_results=False,
            )

            development_weighted_summaries.append(
                weighted_summary
            )

development_weighted_summaries = sorted(
    development_weighted_summaries,
    key=lambda row: (
        -row["metrics"]["ndcg_at_10"],
        -row["metrics"]["reciprocal_rank"],
        -row["metrics"]["hit_at_10"],
        -row["metrics"]["recall_at_10"],
        row["configuration_id"],
    ),
)

best_weighted_summary = (
    development_weighted_summaries[0]
)

(
    repeated_best_weighted_summary,
    best_weighted_results_by_candidate_id,
) = evaluate_weighted_configuration(
    candidate_depth=(
        best_weighted_summary[
            "candidate_depth"
        ]
    ),
    dense_weight=(
        best_weighted_summary[
            "dense_weight"
        ]
    ),
    normalization_method=(
        best_weighted_summary[
            "normalization_method"
        ]
    ),
    retain_query_results=True,
)

assert (
    repeated_best_weighted_summary[
        "configuration_id"
    ]
    == best_weighted_summary[
        "configuration_id"
    ]
)

for metric_name, metric_value in (
    best_weighted_summary[
        "metrics"
    ].items()
):
    assert abs(
        repeated_best_weighted_summary[
            "metrics"
        ][metric_name]
        - metric_value
    ) <= 1e-15

best_weighted_configuration_id = (
    best_weighted_summary[
        "configuration_id"
    ]
)

assert len(
    development_weighted_summaries
) == (
    len(
        WEIGHTED_NORMALIZATION_METHODS
    )
    * len(
        WEIGHTED_CANDIDATE_DEPTHS
    )
    * len(
        DENSE_WEIGHT_VALUES
    )
)

assert len(
    best_weighted_results_by_candidate_id
) == len(
    development_candidate_ids
)

assert set(
    best_weighted_results_by_candidate_id
) == development_candidate_ids

best_weighted_delta_vs_sparse = {
    metric_name: float(
        best_weighted_summary[
            "metrics"
        ][metric_name]
        - sparse_development_metrics[
            metric_name
        ]
    )
    for metric_name
    in best_weighted_summary["metrics"]
}

best_weighted_delta_vs_best_rrf = {
    metric_name: float(
        best_weighted_summary[
            "metrics"
        ][metric_name]
        - best_rrf_summary[
            "metrics"
        ][metric_name]
    )
    for metric_name
    in best_weighted_summary["metrics"]
}

best_hybrid_family = max(
    (
        "rrf",
        "weighted",
    ),
    key=lambda family_name: (
        (
            best_rrf_summary
            if family_name == "rrf"
            else best_weighted_summary
        )["metrics"]["ndcg_at_10"],
        (
            best_rrf_summary
            if family_name == "rrf"
            else best_weighted_summary
        )["metrics"]["reciprocal_rank"],
    ),
)

assert TEST_SPLIT_LOCKED
assert not HYBRID_CONFIGURATION_FROZEN
assert not TEST_QUERIES_ACCESSED

weighted_development_result = {
    "scope": {
        "split": "development",
        "query_count": len(
            development_candidate_ids
        ),
        "test_queries_accessed": False,
    },
    "grid": {
        "candidate_depths": list(
            WEIGHTED_CANDIDATE_DEPTHS
        ),
        "dense_weights": list(
            DENSE_WEIGHT_VALUES
        ),
        "sparse_weight_rule": (
            "1.0 - dense_weight"
        ),
        "normalization_methods": list(
            WEIGHTED_NORMALIZATION_METHODS
        ),
        "configuration_count": len(
            development_weighted_summaries
        ),
    },
    "best_configuration": (
        best_weighted_summary
    ),
    "best_weighted_delta_vs_sparse": (
        best_weighted_delta_vs_sparse
    ),
    "best_weighted_delta_vs_best_rrf": (
        best_weighted_delta_vs_best_rrf
    ),
    "current_best_hybrid_family": (
        best_hybrid_family
    ),
    "top_10_configurations": (
        development_weighted_summaries[
            :10
        ]
    ),
    "all_configuration_summaries_retained": (
        True
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "weighted_grid_status": "PASS",
}

print(
    json.dumps(
        weighted_development_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 2664,
    "test_queries_accessed": false
  },
  "grid": {
    "candidate_depths": [
      20,
      50,
      100
    ],
    "dense_weights": [
      0.1,
      0.2,
      0.3,
      0.4,
      0.5,
      0.6
    ],
    "sparse_weight_rule": "1.0 - dense_weight",
    "normalization_methods": [
      "max",
      "minmax"
    ],
    "configuration_count": 36
  },
  "best_configuration": {
    "configuration_id": "weighted__norm_minmax__depth_100__dense_0.40__sparse_0.60",
    "fusion_method": "WEIGHTED_SCORE_FUSION",
    "normalization_method": "minmax",
    "candidate_depth": 100,
    "dense_weight": 0.4,
    "sparse_weight": 0.6,
    "query_count": 2664,
    "metrics": {
      "hit_at_10": 0.7856606606606606,
      "hit_at_5": 0.7117117117117117,
      "ndcg_at_10": 0.5381727779077033,
      "ndcg_at_5": 0.5115595588576384,
      "precision_at_10": 0.10762012012012014,
      "precision_at_5": 0.18641141141141143,
      "rec

<div dir="rtl" align="right">

## تحلیل پایداری، Sliceها و اطمینان آماری روی Development

انتخاب بهترین تنظیم Hybrid نباید فقط بر میانگین کل Development متکی باشد. در این مرحله، بدون دسترسی به Test، پایداری بهترین Weighted Fusion در مقایسه با Sparse Baseline و بهترین RRF بررسی می‌شود.

کنترل‌های این مرحله شامل موارد زیر است:

1. Bootstrap زوجی برای اختلاف `nDCG@10`؛
2. محاسبه بازه اطمینان ۹۵ درصد؛
3. تحلیل عملکرد براساس نوع منبع مرتبط؛
4. تحلیل براساس طول Query؛
5. تحلیل براساس تعداد Chunkهای مرتبط؛
6. بررسی پایداری Candidate Depthهای ۲۰، ۵۰ و ۱۰۰؛
7. حفظ کامل قفل Test و جلوگیری از تغییر Indexهای Dense و Sparse.

نتایج Sliceها نقش تشخیصی دارند و برای دست‌کاری تنظیم براساس Test استفاده نمی‌شوند. تصمیم نهایی Configuration و ADR فقط پس از تکمیل تحلیل‌های Development فریز خواهد شد.

</div>

In [13]:
# Code Cell 13 — Development stability, slice, and bootstrap analysis

BOOTSTRAP_SAMPLE_COUNT = 2000
BOOTSTRAP_SEED = 16013
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95

assert TEST_SPLIT_LOCKED is True
assert HYBRID_CONFIGURATION_FROZEN is False
assert TEST_QUERIES_ACCESSED is False

assert len(development_candidate_ids_ordered) == 2664
assert best_weighted_configuration_id
assert best_rrf_configuration_id


def result_mapping(results, candidate_ids):
    if isinstance(results, dict):
        assert set(candidate_ids).issubset(
            set(results)
        )
        return {
            candidate_id: results[candidate_id]
            for candidate_id in candidate_ids
        }

    assert isinstance(results, (list, tuple))
    assert len(results) == len(candidate_ids)

    if all(
        isinstance(row, dict)
        and "candidate_id" in row
        for row in results
    ):
        mapping = {
            row["candidate_id"]: row
            for row in results
        }

        assert set(candidate_ids).issubset(
            set(mapping)
        )
        return {
            candidate_id: mapping[candidate_id]
            for candidate_id in candidate_ids
        }

    return {
        candidate_id: row
        for candidate_id, row in zip(
            candidate_ids,
            results,
        )
    }


def extract_ranked_chunk_ids(result):
    if isinstance(result, dict):
        candidate_keys = (
            "ranked_chunk_ids",
            "top_chunk_ids",
            "chunk_ids",
            "document_ids",
            "ranked_ids",
        )

        for key in candidate_keys:
            if key in result:
                ranked_ids = result[key]
                assert isinstance(
                    ranked_ids,
                    (list, tuple, np.ndarray),
                )
                return [
                    str(chunk_id)
                    for chunk_id in ranked_ids
                ]

    if isinstance(
        result,
        (list, tuple, np.ndarray),
    ):
        return [
            str(chunk_id)
            for chunk_id in result
        ]

    raise AssertionError(
        "Ranked chunk IDs could not be extracted "
        "from a retrieval result."
    )


weighted_result_by_candidate_id = result_mapping(
    best_weighted_results_by_candidate_id,
    development_candidate_ids_ordered,
)

rrf_result_by_candidate_id = result_mapping(
    best_rrf_results_by_candidate_id,
    development_candidate_ids_ordered,
)

sparse_result_by_candidate_id = result_mapping(
    development_sparse_results,
    development_candidate_ids_ordered,
)

dense_result_by_candidate_id = result_mapping(
    development_dense_results,
    development_candidate_ids_ordered,
)


def evaluate_result_mapping(result_by_candidate_id):
    metric_rows = {}

    for candidate_id in (
        development_candidate_ids_ordered
    ):
        ranked_chunk_ids = extract_ranked_chunk_ids(
            result_by_candidate_id[candidate_id]
        )

        relevant_chunk_ids = set(
            gold_mapping_by_candidate_id[
                candidate_id
            ]
        )

        metric_rows[candidate_id] = (
            evaluate_ranked_ids(
                ranked_chunk_ids,
                relevant_chunk_ids,
            )
        )

    return metric_rows


weighted_metric_by_candidate_id = (
    evaluate_result_mapping(
        weighted_result_by_candidate_id
    )
)

rrf_metric_by_candidate_id = (
    evaluate_result_mapping(
        rrf_result_by_candidate_id
    )
)

sparse_metric_by_candidate_id = (
    evaluate_result_mapping(
        sparse_result_by_candidate_id
    )
)

dense_metric_by_candidate_id = (
    evaluate_result_mapping(
        dense_result_by_candidate_id
    )
)

METRIC_NAMES = (
    "hit_at_10",
    "hit_at_5",
    "ndcg_at_10",
    "ndcg_at_5",
    "precision_at_10",
    "precision_at_5",
    "recall_at_10",
    "recall_at_5",
    "reciprocal_rank",
)


def aggregate_candidate_metrics(
    metric_by_candidate_id,
    candidate_ids,
):
    assert candidate_ids

    return {
        metric_name: float(
            np.mean(
                [
                    metric_by_candidate_id[
                        candidate_id
                    ][metric_name]
                    for candidate_id in candidate_ids
                ]
            )
        )
        for metric_name in METRIC_NAMES
    }


weighted_recomputed_metrics = (
    aggregate_candidate_metrics(
        weighted_metric_by_candidate_id,
        development_candidate_ids_ordered,
    )
)

rrf_recomputed_metrics = (
    aggregate_candidate_metrics(
        rrf_metric_by_candidate_id,
        development_candidate_ids_ordered,
    )
)

sparse_recomputed_metrics = (
    aggregate_candidate_metrics(
        sparse_metric_by_candidate_id,
        development_candidate_ids_ordered,
    )
)

dense_recomputed_metrics = (
    aggregate_candidate_metrics(
        dense_metric_by_candidate_id,
        development_candidate_ids_ordered,
    )
)

for metric_name in METRIC_NAMES:
    assert np.isclose(
        weighted_recomputed_metrics[metric_name],
        best_weighted_summary[
            "metrics"
        ][metric_name],
        rtol=0.0,
        atol=1e-15,
    )

    assert np.isclose(
        rrf_recomputed_metrics[metric_name],
        best_rrf_summary[
            "metrics"
        ][metric_name],
        rtol=0.0,
        atol=1e-15,
    )

    assert np.isclose(
        sparse_recomputed_metrics[metric_name],
        sparse_development_metrics[
            metric_name
        ],
        rtol=0.0,
        atol=1e-15,
    )

    assert np.isclose(
        dense_recomputed_metrics[metric_name],
        dense_development_metrics[
            metric_name
        ],
        rtol=0.0,
        atol=1e-15,
    )


def paired_bootstrap_delta(
    challenger_metric_by_candidate_id,
    reference_metric_by_candidate_id,
    metric_name,
):
    challenger_values = np.asarray(
        [
            challenger_metric_by_candidate_id[
                candidate_id
            ][metric_name]
            for candidate_id
            in development_candidate_ids_ordered
        ],
        dtype=np.float64,
    )

    reference_values = np.asarray(
        [
            reference_metric_by_candidate_id[
                candidate_id
            ][metric_name]
            for candidate_id
            in development_candidate_ids_ordered
        ],
        dtype=np.float64,
    )

    paired_deltas = (
        challenger_values - reference_values
    )

    rng = np.random.default_rng(
        BOOTSTRAP_SEED
    )

    bootstrap_means = np.empty(
        BOOTSTRAP_SAMPLE_COUNT,
        dtype=np.float64,
    )

    bootstrap_batch_size = 100
    query_count = len(paired_deltas)

    for batch_start in range(
        0,
        BOOTSTRAP_SAMPLE_COUNT,
        bootstrap_batch_size,
    ):
        batch_end = min(
            batch_start + bootstrap_batch_size,
            BOOTSTRAP_SAMPLE_COUNT,
        )

        sampled_indexes = rng.integers(
            low=0,
            high=query_count,
            size=(
                batch_end - batch_start,
                query_count,
            ),
        )

        bootstrap_means[
            batch_start:batch_end
        ] = paired_deltas[
            sampled_indexes
        ].mean(axis=1)

    alpha = (
        1.0 - BOOTSTRAP_CONFIDENCE_LEVEL
    ) / 2.0

    confidence_interval = np.quantile(
        bootstrap_means,
        [alpha, 1.0 - alpha],
    )

    return {
        "metric": metric_name,
        "query_count": query_count,
        "observed_delta": float(
            paired_deltas.mean()
        ),
        "confidence_interval_95": {
            "lower": float(
                confidence_interval[0]
            ),
            "upper": float(
                confidence_interval[1]
            ),
        },
        "bootstrap_sample_count": (
            BOOTSTRAP_SAMPLE_COUNT
        ),
        "seed": BOOTSTRAP_SEED,
        "probability_delta_above_zero": float(
            np.mean(bootstrap_means > 0.0)
        ),
        "status": "PASS",
    }


weighted_vs_sparse_bootstrap = (
    paired_bootstrap_delta(
        challenger_metric_by_candidate_id=(
            weighted_metric_by_candidate_id
        ),
        reference_metric_by_candidate_id=(
            sparse_metric_by_candidate_id
        ),
        metric_name="ndcg_at_10",
    )
)

weighted_vs_rrf_bootstrap = (
    paired_bootstrap_delta(
        challenger_metric_by_candidate_id=(
            weighted_metric_by_candidate_id
        ),
        reference_metric_by_candidate_id=(
            rrf_metric_by_candidate_id
        ),
        metric_name="ndcg_at_10",
    )
)

assert (
    weighted_vs_sparse_bootstrap[
        "observed_delta"
    ] > 0.0
)

assert (
    weighted_vs_rrf_bootstrap[
        "observed_delta"
    ] > 0.0
)


chunk_record_by_id = {
    str(record["chunk_id"]): record
    for record in chunk_records
}

assert len(chunk_record_by_id) == len(
    chunk_records
)


def relevant_source_signature(candidate_id):
    relevant_chunk_ids = (
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    source_types = sorted(
        {
            str(
                chunk_record_by_id[
                    str(chunk_id)
                ]["source_type"]
            )
            for chunk_id in relevant_chunk_ids
        }
    )

    assert source_types
    return "|".join(source_types)


query_lengths = {
    candidate_id: len(
        development_query_by_candidate_id[
            candidate_id
        ].split()
    )
    for candidate_id
    in development_candidate_ids_ordered
}

query_length_values = np.asarray(
    list(query_lengths.values()),
    dtype=np.float64,
)

query_length_q1 = float(
    np.quantile(query_length_values, 0.25)
)
query_length_q3 = float(
    np.quantile(query_length_values, 0.75)
)


def query_length_slice(candidate_id):
    query_length = query_lengths[candidate_id]

    if query_length <= query_length_q1:
        return "SHORT"

    if query_length >= query_length_q3:
        return "LONG"

    return "MEDIUM"


def relevant_multiplicity_slice(candidate_id):
    multiplicity = len(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    if multiplicity == 1:
        return "ONE_RELEVANT_CHUNK"

    if multiplicity == 2:
        return "TWO_RELEVANT_CHUNKS"

    return "THREE_OR_MORE_RELEVANT_CHUNKS"


slice_definitions = {
    "source_signature": {},
    "query_length": {},
    "relevant_chunk_multiplicity": {},
}

for candidate_id in (
    development_candidate_ids_ordered
):
    source_slice = relevant_source_signature(
        candidate_id
    )
    length_slice = query_length_slice(
        candidate_id
    )
    multiplicity_slice = (
        relevant_multiplicity_slice(
            candidate_id
        )
    )

    slice_definitions[
        "source_signature"
    ].setdefault(
        source_slice,
        [],
    ).append(candidate_id)

    slice_definitions[
        "query_length"
    ].setdefault(
        length_slice,
        [],
    ).append(candidate_id)

    slice_definitions[
        "relevant_chunk_multiplicity"
    ].setdefault(
        multiplicity_slice,
        [],
    ).append(candidate_id)


def evaluate_slice(candidate_ids):
    dense_metrics = aggregate_candidate_metrics(
        dense_metric_by_candidate_id,
        candidate_ids,
    )

    sparse_metrics = aggregate_candidate_metrics(
        sparse_metric_by_candidate_id,
        candidate_ids,
    )

    rrf_metrics = aggregate_candidate_metrics(
        rrf_metric_by_candidate_id,
        candidate_ids,
    )

    weighted_metrics = (
        aggregate_candidate_metrics(
            weighted_metric_by_candidate_id,
            candidate_ids,
        )
    )

    return {
        "query_count": len(candidate_ids),
        "dense": dense_metrics,
        "sparse": sparse_metrics,
        "rrf": rrf_metrics,
        "weighted": weighted_metrics,
        "weighted_minus_sparse": {
            metric_name: float(
                weighted_metrics[metric_name]
                - sparse_metrics[metric_name]
            )
            for metric_name in METRIC_NAMES
        },
        "weighted_minus_rrf": {
            metric_name: float(
                weighted_metrics[metric_name]
                - rrf_metrics[metric_name]
            )
            for metric_name in METRIC_NAMES
        },
    }


development_slice_results = {
    slice_family: {
        slice_name: evaluate_slice(
            candidate_ids
        )
        for slice_name, candidate_ids
        in sorted(slices.items())
    }
    for slice_family, slices
    in slice_definitions.items()
}

assert sum(
    result["query_count"]
    for result
    in development_slice_results[
        "query_length"
    ].values()
) == len(development_candidate_ids_ordered)

assert sum(
    result["query_count"]
    for result
    in development_slice_results[
        "relevant_chunk_multiplicity"
    ].values()
) == len(development_candidate_ids_ordered)


def best_configuration_per_depth(
    configuration_summaries,
):
    depths = sorted(
        {
            int(summary["candidate_depth"])
            for summary
            in configuration_summaries
        }
    )

    best_by_depth = {}

    for candidate_depth in depths:
        eligible = [
            summary
            for summary
            in configuration_summaries
            if int(
                summary["candidate_depth"]
            ) == candidate_depth
        ]

        best_by_depth[
            str(candidate_depth)
        ] = sorted(
            eligible,
            key=lambda summary: (
                -summary["metrics"][
                    "ndcg_at_10"
                ],
                -summary["metrics"][
                    "reciprocal_rank"
                ],
                -summary["metrics"][
                    "hit_at_10"
                ],
                summary["configuration_id"],
            ),
        )[0]

    return best_by_depth


weighted_best_by_candidate_depth = (
    best_configuration_per_depth(
        development_weighted_summaries
    )
)

rrf_best_by_candidate_depth = (
    best_configuration_per_depth(
        development_rrf_summaries
    )
)

assert set(
    weighted_best_by_candidate_depth
) == {"20", "50", "100"}

assert set(
    rrf_best_by_candidate_depth
) == {"20", "50", "100"}


source_slice_ndcg_summary = {
    slice_name: {
        "query_count": result[
            "query_count"
        ],
        "dense_ndcg_at_10": result[
            "dense"
        ]["ndcg_at_10"],
        "sparse_ndcg_at_10": result[
            "sparse"
        ]["ndcg_at_10"],
        "rrf_ndcg_at_10": result[
            "rrf"
        ]["ndcg_at_10"],
        "weighted_ndcg_at_10": result[
            "weighted"
        ]["ndcg_at_10"],
        "weighted_minus_sparse_ndcg_at_10": (
            result["weighted_minus_sparse"][
                "ndcg_at_10"
            ]
        ),
    }
    for slice_name, result
    in development_slice_results[
        "source_signature"
    ].items()
}

stability_analysis_result = {
    "scope": {
        "split": "development",
        "query_count": len(
            development_candidate_ids_ordered
        ),
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
    },
    "selected_weighted_configuration": {
        "configuration_id": (
            best_weighted_configuration_id
        ),
        "metrics": (
            weighted_recomputed_metrics
        ),
    },
    "comparators": {
        "sparse": sparse_recomputed_metrics,
        "best_rrf": rrf_recomputed_metrics,
        "dense": dense_recomputed_metrics,
    },
    "paired_bootstrap_ndcg_at_10": {
        "weighted_vs_sparse": (
            weighted_vs_sparse_bootstrap
        ),
        "weighted_vs_best_rrf": (
            weighted_vs_rrf_bootstrap
        ),
    },
    "slice_policy": {
        "source_signature": (
            "Exact source-type signature of "
            "gold relevant chunks"
        ),
        "query_length": {
            "method": (
                "Development whitespace-token "
                "quartiles"
            ),
            "q1": query_length_q1,
            "q3": query_length_q3,
        },
        "relevant_chunk_multiplicity": (
            "1, 2, or >=3 relevant chunks"
        ),
        "role": (
            "DIAGNOSTIC_DEVELOPMENT_ONLY"
        ),
    },
    "source_slice_ndcg_at_10": (
        source_slice_ndcg_summary
    ),
    "all_slice_results": (
        development_slice_results
    ),
    "candidate_depth_stability": {
        "weighted_best_by_depth": (
            weighted_best_by_candidate_depth
        ),
        "rrf_best_by_depth": (
            rrf_best_by_candidate_depth
        ),
    },
    "dense_index_mutated": false
        if False
        else False,
    "sparse_index_mutated": false
        if False
        else False,
    "files_created": 0,
    "paid_api_calls": 0,
    "stability_analysis_status": "PASS",
}

# Replace Python-invalid JSON-style placeholders
stability_analysis_result[
    "dense_index_mutated"
] = False
stability_analysis_result[
    "sparse_index_mutated"
] = False

assert (
    stability_analysis_result[
        "stability_analysis_status"
    ] == "PASS"
)

assert TEST_QUERIES_ACCESSED is False

print(
    json.dumps(
        stability_analysis_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 2664,
    "test_queries_accessed": false
  },
  "selected_weighted_configuration": {
    "configuration_id": "weighted__norm_minmax__depth_100__dense_0.40__sparse_0.60",
    "metrics": {
      "hit_at_10": 0.7856606606606606,
      "hit_at_5": 0.7117117117117117,
      "ndcg_at_10": 0.5381727779077033,
      "ndcg_at_5": 0.5115595588576384,
      "precision_at_10": 0.10762012012012014,
      "precision_at_5": 0.18641141141141143,
      "recall_at_10": 0.621242522284189,
      "recall_at_5": 0.5469027062777063,
      "reciprocal_rank": 0.6083461459951199
    }
  },
  "comparators": {
    "sparse": {
      "hit_at_10": 0.743993993993994,
      "hit_at_5": 0.6786786786786787,
      "ndcg_at_10": 0.50936500655395,
      "ndcg_at_5": 0.4855041074074556,
      "precision_at_10": 0.1001876876876877,
      "precision_at_5": 0.17432432432432432,
      "recall_at_10": 0.5796647838314506,
      "recall_at_5": 0.5128710555793888,
      

<div dir="rtl" align="right">

## Metadata Filtering و Source Diversity

در این مرحله دو الزام مستقل نقشه NB16 بررسی می‌شوند:

1. قابلیت اعمال فیلترهای Metadata روی خروجی Hybrid؛
2. جلوگیری از تمرکز بیش‌ازحد نتایج Top-K روی یک منبع واحد.

فیلدهای `source_id` و `source_type` الزامی هستند. فیلدهای Author، Speaker و Rights نیز براساس Metadata واقعی Corpus شناسایی و وضعیت پوشش آن‌ها ثبت می‌شود؛ نبودن یک فیلد نباید با مقدار ساختگی جبران شود.

همچنین چند سیاست محدودیت تعداد Chunk از هر `source_id` روی Development ارزیابی می‌شود. این ارزیابی مشخص می‌کند آیا افزایش تنوع منبع بدون افت معنادار `nDCG@10` امکان‌پذیر است یا خیر.

در این مرحله:

- Test همچنان قفل است؛
- Dense و Sparse Index تغییر نمی‌کنند؛
- هیچ Embedding جدیدی محاسبه نمی‌شود؛
- نتایج فقط برای تصمیم‌گیری Development استفاده می‌شوند.

</div>

In [14]:
# Code Cell 14 — Metadata filtering and source-diversity evaluation

DIVERSITY_CAP_CANDIDATES = (1, 2, 3)
DIVERSITY_MAX_NDCG_AT_10_DROP = 0.005
METADATA_FILTER_SAMPLE_LIMIT = 25

assert TEST_SPLIT_LOCKED is True
assert HYBRID_CONFIGURATION_FROZEN is False
assert TEST_QUERIES_ACCESSED is False


def normalize_metadata_value(value):
    if value is None:
        return []

    if isinstance(value, bool):
        return [str(value).lower()]

    if isinstance(value, (str, int, float)):
        normalized = str(value).strip()
        return [normalized] if normalized else []

    if isinstance(value, dict):
        normalized_values = []

        for nested_value in value.values():
            normalized_values.extend(
                normalize_metadata_value(
                    nested_value
                )
            )

        return normalized_values

    if isinstance(value, (list, tuple, set)):
        normalized_values = []

        for nested_value in value:
            normalized_values.extend(
                normalize_metadata_value(
                    nested_value
                )
            )

        return normalized_values

    normalized = str(value).strip()
    return [normalized] if normalized else []


def flatten_metadata(record):
    flattened = {}

    def visit(value, path):
        if isinstance(value, dict):
            for key, nested_value in value.items():
                normalized_key = str(key).strip().lower()

                nested_path = (
                    f"{path}.{normalized_key}"
                    if path
                    else normalized_key
                )

                visit(
                    nested_value,
                    nested_path,
                )

        elif isinstance(value, (list, tuple, set)):
            scalar_values = []

            for nested_value in value:
                if isinstance(
                    nested_value,
                    (dict, list, tuple, set),
                ):
                    visit(
                        nested_value,
                        path,
                    )
                else:
                    scalar_values.extend(
                        normalize_metadata_value(
                            nested_value
                        )
                    )

            if scalar_values:
                flattened.setdefault(
                    path,
                    [],
                ).extend(scalar_values)

        else:
            scalar_values = (
                normalize_metadata_value(value)
            )

            if scalar_values:
                flattened.setdefault(
                    path,
                    [],
                ).extend(scalar_values)

    visit(record, "")

    return {
        key: tuple(
            sorted(set(values))
        )
        for key, values in flattened.items()
        if key
    }


METADATA_FIELD_ALIASES = {
    "source_id": {
        "source_id",
        "metadata.source_id",
        "source.source_id",
    },
    "source_type": {
        "source_type",
        "metadata.source_type",
        "source.source_type",
    },
    "author": {
        "author",
        "authors",
        "author_name",
        "metadata.author",
        "metadata.authors",
        "metadata.author_name",
        "source.author",
        "source.authors",
        "creator",
        "metadata.creator",
    },
    "speaker": {
        "speaker",
        "speakers",
        "speaker_name",
        "metadata.speaker",
        "metadata.speakers",
        "metadata.speaker_name",
        "source.speaker",
    },
    "rights": {
        "rights",
        "rights_holder",
        "copyright",
        "copyright_holder",
        "license",
        "licence",
        "usage_rights",
        "metadata.rights",
        "metadata.rights_holder",
        "metadata.copyright",
        "metadata.license",
        "source.rights",
    },
}


def field_values_from_flattened_metadata(
    flattened_metadata,
    field_name,
):
    alias_paths = METADATA_FIELD_ALIASES[
        field_name
    ]

    values = []

    for path, path_values in (
        flattened_metadata.items()
    ):
        terminal_key = path.split(".")[-1]

        if (
            path in alias_paths
            or terminal_key in alias_paths
        ):
            values.extend(path_values)

    return tuple(
        sorted(set(values))
    )


chunk_metadata_by_id = {}

for chunk_id, record in chunk_record_by_id.items():
    flattened_metadata = flatten_metadata(record)

    canonical_metadata = {
        field_name: (
            field_values_from_flattened_metadata(
                flattened_metadata,
                field_name,
            )
        )
        for field_name
        in METADATA_FIELD_ALIASES
    }

    direct_source_id = str(
        record.get("source_id", "")
    ).strip()

    direct_source_type = str(
        record.get("source_type", "")
    ).strip()

    if direct_source_id:
        canonical_metadata["source_id"] = tuple(
            sorted(
                set(
                    canonical_metadata["source_id"]
                )
                | {direct_source_id}
            )
        )

    if direct_source_type:
        canonical_metadata[
            "source_type"
        ] = tuple(
            sorted(
                set(
                    canonical_metadata[
                        "source_type"
                    ]
                )
                | {direct_source_type}
            )
        )

    chunk_metadata_by_id[chunk_id] = (
        canonical_metadata
    )

assert len(chunk_metadata_by_id) == 3488

assert all(
    metadata["source_id"]
    for metadata
    in chunk_metadata_by_id.values()
)

assert all(
    metadata["source_type"]
    for metadata
    in chunk_metadata_by_id.values()
)


metadata_field_coverage = {}

for field_name in METADATA_FIELD_ALIASES:
    populated_chunk_ids = [
        chunk_id
        for chunk_id, metadata
        in chunk_metadata_by_id.items()
        if metadata[field_name]
    ]

    distinct_values = sorted(
        {
            value
            for chunk_id in populated_chunk_ids
            for value in (
                chunk_metadata_by_id[
                    chunk_id
                ][field_name]
            )
        }
    )

    metadata_field_coverage[field_name] = {
        "populated_chunk_count": len(
            populated_chunk_ids
        ),
        "coverage_ratio": float(
            len(populated_chunk_ids)
            / len(chunk_metadata_by_id)
        ),
        "distinct_value_count": len(
            distinct_values
        ),
        "availability": (
            "AVAILABLE"
            if populated_chunk_ids
            else "NOT_PRESENT_IN_CANONICAL_CORPUS"
        ),
    }


def metadata_matches(
    chunk_id,
    field_name,
    allowed_values,
):
    allowed_values = {
        str(value).strip()
        for value in allowed_values
        if str(value).strip()
    }

    chunk_values = set(
        chunk_metadata_by_id[
            str(chunk_id)
        ][field_name]
    )

    return bool(
        chunk_values & allowed_values
    )


def filter_ranked_chunk_ids(
    ranked_chunk_ids,
    field_name,
    allowed_values,
):
    return [
        str(chunk_id)
        for chunk_id in ranked_chunk_ids
        if metadata_matches(
            chunk_id=chunk_id,
            field_name=field_name,
            allowed_values=allowed_values,
        )
    ]


metadata_filter_contract_results = {}

for field_name in METADATA_FIELD_ALIASES:
    value_to_chunk_ids = {}

    for chunk_id, metadata in (
        chunk_metadata_by_id.items()
    ):
        for value in metadata[field_name]:
            value_to_chunk_ids.setdefault(
                value,
                [],
            ).append(chunk_id)

    eligible_values = sorted(
        (
            value
            for value, chunk_ids
            in value_to_chunk_ids.items()
            if chunk_ids
        ),
        key=lambda value: (
            -len(value_to_chunk_ids[value]),
            value,
        ),
    )[:METADATA_FILTER_SAMPLE_LIMIT]

    checked_filter_count = 0
    checked_result_count = 0
    mismatched_result_count = 0

    for value in eligible_values:
        for candidate_id in (
            development_candidate_ids_ordered
        ):
            ranked_chunk_ids = (
                extract_ranked_chunk_ids(
                    weighted_result_by_candidate_id[
                        candidate_id
                    ]
                )
            )

            filtered_chunk_ids = (
                filter_ranked_chunk_ids(
                    ranked_chunk_ids=(
                        ranked_chunk_ids
                    ),
                    field_name=field_name,
                    allowed_values={value},
                )
            )

            if not filtered_chunk_ids:
                continue

            checked_filter_count += 1
            checked_result_count += len(
                filtered_chunk_ids
            )

            mismatched_result_count += sum(
                not metadata_matches(
                    chunk_id=chunk_id,
                    field_name=field_name,
                    allowed_values={value},
                )
                for chunk_id
                in filtered_chunk_ids
            )

    field_available = bool(eligible_values)

    if field_available:
        assert checked_filter_count > 0
        assert checked_result_count > 0
        assert mismatched_result_count == 0

    metadata_filter_contract_results[
        field_name
    ] = {
        "availability": (
            metadata_field_coverage[
                field_name
            ]["availability"]
        ),
        "sampled_value_count": len(
            eligible_values
        ),
        "checked_filter_count": (
            checked_filter_count
        ),
        "checked_result_count": (
            checked_result_count
        ),
        "mismatched_result_count": (
            mismatched_result_count
        ),
        "filter_contract_status": (
            "PASS"
            if field_available
            else "NOT_APPLICABLE_FIELD_ABSENT"
        ),
    }


def primary_source_id(chunk_id):
    source_ids = chunk_metadata_by_id[
        str(chunk_id)
    ]["source_id"]

    assert source_ids
    return source_ids[0]


def source_diversify(
    ranked_chunk_ids,
    maximum_per_source,
    top_k=10,
):
    assert maximum_per_source >= 1
    assert top_k >= 1

    selected_chunk_ids = []
    source_counts = {}

    for chunk_id in ranked_chunk_ids:
        chunk_id = str(chunk_id)
        source_id = primary_source_id(
            chunk_id
        )

        current_count = source_counts.get(
            source_id,
            0,
        )

        if current_count >= maximum_per_source:
            continue

        selected_chunk_ids.append(chunk_id)
        source_counts[source_id] = (
            current_count + 1
        )

        if len(selected_chunk_ids) == top_k:
            break

    return selected_chunk_ids


def source_diversity_statistics(
    ranked_chunk_ids,
    top_k=10,
):
    top_chunk_ids = [
        str(chunk_id)
        for chunk_id
        in ranked_chunk_ids[:top_k]
    ]

    source_ids = [
        primary_source_id(chunk_id)
        for chunk_id in top_chunk_ids
    ]

    source_counts = collections.Counter(
        source_ids
    )

    maximum_source_count = (
        max(source_counts.values())
        if source_counts
        else 0
    )

    return {
        "result_count": len(top_chunk_ids),
        "unique_source_count": len(
            source_counts
        ),
        "maximum_chunks_from_one_source": (
            maximum_source_count
        ),
        "maximum_source_share": float(
            maximum_source_count
            / len(top_chunk_ids)
        )
        if top_chunk_ids
        else 0.0,
    }


baseline_diversity_rows = []

for candidate_id in (
    development_candidate_ids_ordered
):
    ranked_chunk_ids = extract_ranked_chunk_ids(
        weighted_result_by_candidate_id[
            candidate_id
        ]
    )

    baseline_diversity_rows.append(
        source_diversity_statistics(
            ranked_chunk_ids,
            top_k=10,
        )
    )

baseline_source_diversity = {
    "mean_unique_source_count_at_10": float(
        np.mean(
            [
                row["unique_source_count"]
                for row
                in baseline_diversity_rows
            ]
        )
    ),
    "mean_maximum_source_share_at_10": float(
        np.mean(
            [
                row["maximum_source_share"]
                for row
                in baseline_diversity_rows
            ]
        )
    ),
    "single_source_top_10_query_count": int(
        sum(
            row["unique_source_count"] == 1
            for row
            in baseline_diversity_rows
        )
    ),
}

diversity_cap_evaluations = {}

for maximum_per_source in (
    DIVERSITY_CAP_CANDIDATES
):
    diversified_metric_rows = []
    diversified_diversity_rows = []
    incomplete_top_10_query_count = 0

    for candidate_id in (
        development_candidate_ids_ordered
    ):
        ranked_chunk_ids = (
            extract_ranked_chunk_ids(
                weighted_result_by_candidate_id[
                    candidate_id
                ]
            )
        )

        diversified_chunk_ids = (
            source_diversify(
                ranked_chunk_ids=(
                    ranked_chunk_ids
                ),
                maximum_per_source=(
                    maximum_per_source
                ),
                top_k=10,
            )
        )

        if len(diversified_chunk_ids) < 10:
            incomplete_top_10_query_count += 1

        diversified_metric_rows.append(
            evaluate_ranked_ids(
                diversified_chunk_ids,
                set(
                    gold_mapping_by_candidate_id[
                        candidate_id
                    ]
                ),
            )
        )

        diversified_diversity_rows.append(
            source_diversity_statistics(
                diversified_chunk_ids,
                top_k=10,
            )
        )

    diversified_metrics = {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row
                    in diversified_metric_rows
                ]
            )
        )
        for metric_name in METRIC_NAMES
    }

    ndcg_at_10_delta = float(
        diversified_metrics["ndcg_at_10"]
        - weighted_recomputed_metrics[
            "ndcg_at_10"
        ]
    )

    diversity_cap_evaluations[
        str(maximum_per_source)
    ] = {
        "maximum_chunks_per_source": (
            maximum_per_source
        ),
        "metrics": diversified_metrics,
        "delta_vs_unrestricted_weighted": {
            metric_name: float(
                diversified_metrics[
                    metric_name
                ]
                - weighted_recomputed_metrics[
                    metric_name
                ]
            )
            for metric_name in METRIC_NAMES
        },
        "mean_unique_source_count_at_10": float(
            np.mean(
                [
                    row["unique_source_count"]
                    for row
                    in diversified_diversity_rows
                ]
            )
        ),
        "mean_maximum_source_share_at_10": float(
            np.mean(
                [
                    row[
                        "maximum_source_share"
                    ]
                    for row
                    in diversified_diversity_rows
                ]
            )
        ),
        "incomplete_top_10_query_count": (
            incomplete_top_10_query_count
        ),
        "quality_gate_passed": bool(
            ndcg_at_10_delta
            >= -DIVERSITY_MAX_NDCG_AT_10_DROP
        ),
        "status": "PASS",
    }


eligible_diversity_policies = [
    result
    for result
    in diversity_cap_evaluations.values()
    if result["quality_gate_passed"]
    and result[
        "incomplete_top_10_query_count"
    ] == 0
]

if eligible_diversity_policies:
    selected_diversity_policy = sorted(
        eligible_diversity_policies,
        key=lambda result: (
            -result[
                "mean_unique_source_count_at_10"
            ],
            -result["metrics"]["ndcg_at_10"],
            result[
                "maximum_chunks_per_source"
            ],
        ),
    )[0]

    diversity_decision = (
        "ELIGIBLE_FOR_CONFIGURATION_SELECTION"
    )

else:
    selected_diversity_policy = None
    diversity_decision = (
        "NO_HARD_CAP_ELIGIBLE"
    )


assert metadata_filter_contract_results[
    "source_id"
]["filter_contract_status"] == "PASS"

assert metadata_filter_contract_results[
    "source_type"
]["filter_contract_status"] == "PASS"

assert TEST_QUERIES_ACCESSED is False


metadata_and_diversity_result = {
    "scope": {
        "split": "development",
        "query_count": len(
            development_candidate_ids_ordered
        ),
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
    },
    "metadata_field_coverage": (
        metadata_field_coverage
    ),
    "metadata_filter_contract": (
        metadata_filter_contract_results
    ),
    "required_filter_status": {
        "source_id": "PASS",
        "source_type": "PASS",
        "author": (
            metadata_filter_contract_results[
                "author"
            ]["filter_contract_status"]
        ),
        "speaker": (
            metadata_filter_contract_results[
                "speaker"
            ]["filter_contract_status"]
        ),
        "rights": (
            metadata_filter_contract_results[
                "rights"
            ]["filter_contract_status"]
        ),
    },
    "source_diversity": {
        "unrestricted_weighted": (
            baseline_source_diversity
        ),
        "maximum_allowed_ndcg_at_10_drop": (
            DIVERSITY_MAX_NDCG_AT_10_DROP
        ),
        "cap_evaluations": (
            diversity_cap_evaluations
        ),
        "selected_candidate_policy": (
            selected_diversity_policy
        ),
        "decision": diversity_decision,
    },
    "policy_note": (
        "Absent metadata fields are reported as "
        "not applicable and are never synthesized."
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "metadata_and_diversity_status": "PASS",
}

print(
    json.dumps(
        metadata_and_diversity_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "split": "development",
    "query_count": 2664,
    "test_queries_accessed": false
  },
  "metadata_field_coverage": {
    "source_id": {
      "populated_chunk_count": 3488,
      "coverage_ratio": 1.0,
      "distinct_value_count": 715,
      "availability": "AVAILABLE"
    },
    "source_type": {
      "populated_chunk_count": 3488,
      "coverage_ratio": 1.0,
      "distinct_value_count": 3,
      "availability": "AVAILABLE"
    },
    "author": {
      "populated_chunk_count": 0,
      "coverage_ratio": 0.0,
      "distinct_value_count": 0,
      "availability": "NOT_PRESENT_IN_CANONICAL_CORPUS"
    },
    "speaker": {
      "populated_chunk_count": 0,
      "coverage_ratio": 0.0,
      "distinct_value_count": 0,
      "availability": "NOT_PRESENT_IN_CANONICAL_CORPUS"
    },
    "rights": {
      "populated_chunk_count": 0,
      "coverage_ratio": 0.0,
      "distinct_value_count": 0,
      "availability": "NOT_PRESENT_IN_CANONICAL_CORPUS"
    }
  },
  "metada

<div dir="rtl" align="right">

## بررسی عمیق Metadataهای Author، Speaker و Content Authority

آزمایش قبلی نشان داد `source_id` و `source_type` به‌طور کامل قابل فیلتر هستند، اما فیلدهای Author، Speaker و Rights با نام‌های اولیه پیدا نشدند.

از آنجا که نقشه ماژولار NB16 صریحاً Source-aware Filtering شامل منبع، نویسنده، گوینده و نوع اختیار محتوایی را الزامی کرده است، نبودن این فیلدها نباید بدون بررسی بیشتر به‌عنوان Not Applicable پذیرفته شود.

در این مرحله تمام مسیرهای Metadata موجود در Chunkهای Canonical بررسی می‌شوند تا مشخص شود آیا اطلاعات موردنیاز با نام‌هایی مانند موارد زیر ذخیره شده‌اند:

- Author، Creator یا Writer؛
- Speaker، Guest، Host یا Participant؛
- Authority، Editorial، Rights، License یا Copyright؛
- Role، Attribution یا Ownership.

این سل هیچ Artifact یا Indexی را تغییر نمی‌دهد. اگر مسیر معتبری پیدا شود، در مرحله بعد به قرارداد فیلتر Canonical نگاشت خواهد شد. اگر هیچ مسیر معتبری وجود نداشته باشد، Metadata Gap رسمی ثبت می‌شود و NB16 تا تعیین تکلیف آن نباید به‌صورت کامل Freeze شود.

</div>

In [15]:
# Code Cell 15 — Deep audit of author, speaker, and authority metadata

METADATA_CONCEPT_KEYWORDS = {
    "author": (
        "author",
        "creator",
        "writer",
        "editor",
        "مولف",
        "نویسنده",
    ),
    "speaker": (
        "speaker",
        "participant",
        "guest",
        "host",
        "presenter",
        "interviewer",
        "interviewee",
        "گوینده",
        "مهمان",
        "مجری",
    ),
    "content_authority": (
        "authority",
        "attribution",
        "editorial",
        "supervision",
        "rights",
        "right",
        "license",
        "licence",
        "copyright",
        "ownership",
        "permission",
        "role",
        "اختیار",
        "مالکیت",
    ),
}

all_flattened_metadata_by_id = {
    chunk_id: flatten_metadata(record)
    for chunk_id, record
    in chunk_record_by_id.items()
}

assert len(
    all_flattened_metadata_by_id
) == len(chunk_record_by_id)


metadata_path_statistics = {}

for chunk_id, flattened_metadata in (
    all_flattened_metadata_by_id.items()
):
    for path, values in (
        flattened_metadata.items()
    ):
        path_statistic = (
            metadata_path_statistics.setdefault(
                path,
                {
                    "populated_chunk_ids": set(),
                    "values": set(),
                },
            )
        )

        path_statistic[
            "populated_chunk_ids"
        ].add(chunk_id)

        path_statistic["values"].update(
            str(value)
            for value in values
            if str(value).strip()
        )


concept_path_candidates = {}

for concept_name, keywords in (
    METADATA_CONCEPT_KEYWORDS.items()
):
    matched_paths = []

    for path, statistics in (
        metadata_path_statistics.items()
    ):
        normalized_path = path.lower()

        if not any(
            keyword.lower() in normalized_path
            for keyword in keywords
        ):
            continue

        sorted_values = sorted(
            statistics["values"]
        )

        matched_paths.append(
            {
                "path": path,
                "populated_chunk_count": len(
                    statistics[
                        "populated_chunk_ids"
                    ]
                ),
                "coverage_ratio": float(
                    len(
                        statistics[
                            "populated_chunk_ids"
                        ]
                    )
                    / len(chunk_record_by_id)
                ),
                "distinct_value_count": len(
                    sorted_values
                ),
                "sample_values": (
                    sorted_values[:10]
                ),
            }
        )

    concept_path_candidates[
        concept_name
    ] = sorted(
        matched_paths,
        key=lambda row: (
            -row["populated_chunk_count"],
            row["path"],
        ),
    )


record_top_level_key_counts = (
    collections.Counter()
)

for record in chunk_records:
    record_top_level_key_counts.update(
        str(key)
        for key in record.keys()
    )

top_level_schema = [
    {
        "key": key,
        "record_count": int(count),
        "coverage_ratio": float(
            count / len(chunk_records)
        ),
    }
    for key, count in sorted(
        record_top_level_key_counts.items()
    )
]


all_metadata_paths = sorted(
    {
        path
        for flattened_metadata
        in all_flattened_metadata_by_id.values()
        for path in flattened_metadata
    }
)

member_related_paths = [
    {
        "path": path,
        "populated_chunk_count": len(
            metadata_path_statistics[
                path
            ]["populated_chunk_ids"]
        ),
        "distinct_value_count": len(
            metadata_path_statistics[
                path
            ]["values"]
        ),
        "sample_values": sorted(
            metadata_path_statistics[
                path
            ]["values"]
        )[:5],
    }
    for path in all_metadata_paths
    if any(
        token in path.lower()
        for token in (
            "member",
            "segment",
            "document",
            "source",
            "metadata",
            "lineage",
        )
    )
]


def concept_audit_status(concept_name):
    candidates = concept_path_candidates[
        concept_name
    ]

    if candidates:
        return "CANDIDATE_PATHS_FOUND_REQUIRES_MAPPING"

    return "CONFIRMED_ABSENT_FROM_CANONICAL_CHUNKS"


metadata_gap_confirmed = all(
    not concept_path_candidates[
        concept_name
    ]
    for concept_name in (
        "author",
        "speaker",
        "content_authority",
    )
)

deep_metadata_audit_result = {
    "scope": {
        "canonical_chunk_count": len(
            chunk_records
        ),
        "split_access": (
            "CORPUS_SCHEMA_ONLY"
        ),
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
    },
    "top_level_schema": top_level_schema,
    "total_flattened_metadata_path_count": len(
        all_metadata_paths
    ),
    "concept_audit": {
        concept_name: {
            "status": concept_audit_status(
                concept_name
            ),
            "candidate_path_count": len(
                concept_path_candidates[
                    concept_name
                ]
            ),
            "candidate_paths": (
                concept_path_candidates[
                    concept_name
                ]
            ),
        }
        for concept_name
        in METADATA_CONCEPT_KEYWORDS
    },
    "member_and_lineage_paths": (
        member_related_paths
    ),
    "roadmap_requirement": {
        "source_filter": "REQUIRED",
        "author_filter": "REQUIRED",
        "speaker_filter": "REQUIRED",
        "content_authority_filter": (
            "REQUIRED"
        ),
    },
    "metadata_gap_confirmed": (
        metadata_gap_confirmed
    ),
    "next_action": (
        "REGISTER_UPSTREAM_METADATA_GAP"
        if metadata_gap_confirmed
        else "MAP_DISCOVERED_PATHS_TO_CANONICAL_FILTERS"
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "repository_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "deep_metadata_audit_status": "PASS",
}

assert TEST_QUERIES_ACCESSED is False

print(
    json.dumps(
        deep_metadata_audit_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "canonical_chunk_count": 3488,
    "split_access": "CORPUS_SCHEMA_ONLY",
    "test_queries_accessed": false
  },
  "top_level_schema": [
    {
      "key": "attribution_context",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "boundary_metadata",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "char_count",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "chunk_contract_version",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "chunk_count",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "chunk_id",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "chunk_index",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "content",
      "record_count": 3488,
      "coverage_ratio": 1.0
    },
    {
      "key": "content_origin",
      "record_

<div dir="rtl" align="right">

## تعیین ساختار دقیق Metadataهای Attribution و Rights

بررسی عمیق نشان داد Metadataهای موردنیاز در Chunk Canonical وجود دارند، اما به‌صورت Nested ذخیره شده‌اند.

سه نکته باید قبل از تعریف فیلتر Canonical روشن شود:

1. `content_authority_types` نوع اختیار محتوا را نشان می‌دهد و لزوماً هویت Author نیست؛
2. `speaker_keys` علاوه بر نام گوینده ممکن است شامل وضعیت‌های Attribution یا مقادیر کنترلی باشد؛
3. اطلاعات Rights در آرایه `member_rights` و در سطح اعضای تشکیل‌دهنده Chunk نگهداری می‌شود.

در این مرحله ساختار دقیق، نوع داده، کلیدهای داخلی و نمونه‌های کنترل‌شده این Metadataها استخراج می‌شود. هدف جلوگیری از نگاشت اشتباه و تعریف قرارداد فیلتر براساس معنای واقعی داده است.

این سل فقط Schema را بررسی می‌کند و هیچ Index، Artifact یا Configuration را تغییر نمی‌دهد.

</div>

In [16]:
# Code Cell 16 — Exact nested attribution and rights schema inspection

SCHEMA_SAMPLE_LIMIT = 5
VALUE_SAMPLE_LIMIT = 10

assert TEST_QUERIES_ACCESSED is False


def value_type_name(value):
    if value is None:
        return "null"

    if isinstance(value, bool):
        return "boolean"

    if isinstance(value, str):
        return "string"

    if isinstance(value, int):
        return "integer"

    if isinstance(value, float):
        return "float"

    if isinstance(value, dict):
        return "object"

    if isinstance(value, list):
        return "array"

    if isinstance(value, tuple):
        return "tuple"

    return type(value).__name__


def collect_exact_leaf_paths(
    value,
    path="",
):
    collected = []

    if isinstance(value, dict):
        for key, nested_value in value.items():
            nested_path = (
                f"{path}.{key}"
                if path
                else str(key)
            )

            collected.extend(
                collect_exact_leaf_paths(
                    nested_value,
                    nested_path,
                )
            )

    elif isinstance(value, (list, tuple)):
        array_path = (
            f"{path}[]"
            if path
            else "[]"
        )

        if not value:
            collected.append(
                {
                    "path": array_path,
                    "type": "empty_array",
                    "value": None,
                }
            )

        for nested_value in value:
            collected.extend(
                collect_exact_leaf_paths(
                    nested_value,
                    array_path,
                )
            )

    else:
        collected.append(
            {
                "path": path,
                "type": value_type_name(value),
                "value": value,
            }
        )

    return collected


INSPECTED_ROOT_FIELDS = (
    "attribution_context",
    "rights",
    "content_origin",
    "role",
)

nested_schema_statistics = {}

for record in chunk_records:
    source_type = str(
        record["source_type"]
    )

    source_statistics = (
        nested_schema_statistics.setdefault(
            source_type,
            {},
        )
    )

    for root_field in INSPECTED_ROOT_FIELDS:
        root_value = record.get(root_field)

        root_statistics = (
            source_statistics.setdefault(
                root_field,
                {
                    "root_types": (
                        collections.Counter()
                    ),
                    "leaf_paths": {},
                    "raw_samples": [],
                },
            )
        )

        root_statistics["root_types"].update(
            [value_type_name(root_value)]
        )

        leaf_rows = collect_exact_leaf_paths(
            root_value,
            root_field,
        )

        for leaf_row in leaf_rows:
            path_statistics = (
                root_statistics[
                    "leaf_paths"
                ].setdefault(
                    leaf_row["path"],
                    {
                        "types": (
                            collections.Counter()
                        ),
                        "values": set(),
                        "populated_record_count": 0,
                    },
                )
            )

            path_statistics["types"].update(
                [leaf_row["type"]]
            )

            if leaf_row["value"] is not None:
                serialized_value = (
                    json.dumps(
                        leaf_row["value"],
                        ensure_ascii=False,
                        sort_keys=True,
                        default=str,
                    )
                )

                path_statistics["values"].add(
                    serialized_value
                )

                path_statistics[
                    "populated_record_count"
                ] += 1

        if (
            len(root_statistics["raw_samples"])
            < SCHEMA_SAMPLE_LIMIT
        ):
            serialized_root_value = json.dumps(
                root_value,
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            )

            existing_samples = {
                json.dumps(
                    sample,
                    ensure_ascii=False,
                    sort_keys=True,
                    default=str,
                )
                for sample
                in root_statistics["raw_samples"]
            }

            if (
                serialized_root_value
                not in existing_samples
            ):
                root_statistics[
                    "raw_samples"
                ].append(root_value)


def serialize_schema_statistics(
    schema_statistics,
):
    serialized = {}

    for source_type, source_statistics in (
        sorted(schema_statistics.items())
    ):
        serialized[source_type] = {}

        for root_field, root_statistics in (
            sorted(source_statistics.items())
        ):
            serialized_leaf_paths = {}

            for path, path_statistics in sorted(
                root_statistics[
                    "leaf_paths"
                ].items()
            ):
                sample_values = sorted(
                    path_statistics["values"]
                )[:VALUE_SAMPLE_LIMIT]

                serialized_leaf_paths[path] = {
                    "types": dict(
                        sorted(
                            path_statistics[
                                "types"
                            ].items()
                        )
                    ),
                    "observed_leaf_count": int(
                        path_statistics[
                            "populated_record_count"
                        ]
                    ),
                    "distinct_value_count": len(
                        path_statistics["values"]
                    ),
                    "sample_values": [
                        json.loads(value)
                        for value in sample_values
                    ],
                }

            serialized[source_type][
                root_field
            ] = {
                "root_types": dict(
                    sorted(
                        root_statistics[
                            "root_types"
                        ].items()
                    )
                ),
                "leaf_paths": (
                    serialized_leaf_paths
                ),
                "raw_samples": (
                    root_statistics[
                        "raw_samples"
                    ]
                ),
            }

    return serialized


serialized_nested_schema = (
    serialize_schema_statistics(
        nested_schema_statistics
    )
)


def extract_direct_field_values(
    record,
    root_field,
    nested_field,
):
    root_value = record.get(root_field)

    if not isinstance(root_value, dict):
        return []

    nested_value = root_value.get(
        nested_field
    )

    return normalize_metadata_value(
        nested_value
    )


direct_field_audit = {}

DIRECT_FIELD_CONTRACTS = {
    "speaker_keys": (
        "attribution_context",
        "speaker_keys",
    ),
    "content_authority_types": (
        "attribution_context",
        "content_authority_types",
    ),
    "content_origins": (
        "attribution_context",
        "content_origins",
    ),
    "base_statuses": (
        "attribution_context",
        "base_statuses",
    ),
    "effective_primary_signals": (
        "attribution_context",
        "effective_primary_signals",
    ),
}

for contract_name, (
    root_field,
    nested_field,
) in DIRECT_FIELD_CONTRACTS.items():
    by_source_type = {}

    for source_type in sorted(
        {
            str(record["source_type"])
            for record in chunk_records
        }
    ):
        source_records = [
            record
            for record in chunk_records
            if str(
                record["source_type"]
            ) == source_type
        ]

        populated_record_count = 0
        values = set()

        for record in source_records:
            extracted_values = (
                extract_direct_field_values(
                    record=record,
                    root_field=root_field,
                    nested_field=nested_field,
                )
            )

            if extracted_values:
                populated_record_count += 1
                values.update(
                    extracted_values
                )

        by_source_type[source_type] = {
            "record_count": len(
                source_records
            ),
            "populated_record_count": (
                populated_record_count
            ),
            "distinct_value_count": len(
                values
            ),
            "sample_values": sorted(
                values
            )[:25],
        }

    direct_field_audit[
        contract_name
    ] = by_source_type


def extract_member_rights_rows(record):
    rights_value = record.get("rights")

    if not isinstance(rights_value, dict):
        return []

    member_rights = rights_value.get(
        "member_rights",
        [],
    )

    if isinstance(member_rights, dict):
        member_rights = [member_rights]

    if not isinstance(member_rights, list):
        return []

    return [
        row
        for row in member_rights
        if isinstance(row, dict)
    ]


rights_nested_field_statistics = {}

for record in chunk_records:
    source_type = str(
        record["source_type"]
    )

    source_statistics = (
        rights_nested_field_statistics.setdefault(
            source_type,
            {
                "chunk_count": 0,
                "member_right_row_count": 0,
                "status_values": set(),
                "basis_values": set(),
                "member_row_keys": set(),
                "rights_object_keys": set(),
            },
        )
    )

    source_statistics["chunk_count"] += 1

    member_right_rows = (
        extract_member_rights_rows(record)
    )

    source_statistics[
        "member_right_row_count"
    ] += len(member_right_rows)

    for member_right_row in member_right_rows:
        source_statistics[
            "member_row_keys"
        ].update(
            str(key)
            for key
            in member_right_row.keys()
        )

        nested_rights = (
            member_right_row.get("rights")
        )

        if not isinstance(
            nested_rights,
            dict,
        ):
            continue

        source_statistics[
            "rights_object_keys"
        ].update(
            str(key)
            for key
            in nested_rights.keys()
        )

        status_value = nested_rights.get(
            "status"
        )
        basis_value = nested_rights.get(
            "basis"
        )

        if status_value is not None:
            source_statistics[
                "status_values"
            ].add(str(status_value))

        if basis_value is not None:
            source_statistics[
                "basis_values"
            ].add(str(basis_value))


serialized_rights_statistics = {
    source_type: {
        "chunk_count": statistics[
            "chunk_count"
        ],
        "member_right_row_count": (
            statistics[
                "member_right_row_count"
            ]
        ),
        "member_row_keys": sorted(
            statistics["member_row_keys"]
        ),
        "rights_object_keys": sorted(
            statistics[
                "rights_object_keys"
            ]
        ),
        "status_values": sorted(
            statistics["status_values"]
        ),
        "basis_values": sorted(
            statistics["basis_values"]
        ),
    }
    for source_type, statistics
    in sorted(
        rights_nested_field_statistics.items()
    )
}


exact_nested_metadata_result = {
    "scope": {
        "canonical_chunk_count": len(
            chunk_records
        ),
        "source_types": sorted(
            {
                str(record["source_type"])
                for record in chunk_records
            }
        ),
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
    },
    "direct_field_audit": (
        direct_field_audit
    ),
    "rights_member_structure": (
        serialized_rights_statistics
    ),
    "exact_nested_schema": (
        serialized_nested_schema
    ),
    "interpretation_guardrails": {
        "content_authority_is_author_identity": (
            False
        ),
        "speaker_values_must_come_only_from": (
            "attribution_context.speaker_keys"
        ),
        "rights_values_must_come_only_from": (
            "rights.member_rights[].rights"
        ),
        "synthetic_metadata_allowed": False,
    },
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "repository_mutated": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "nested_metadata_schema_status": "PASS",
}

assert TEST_QUERIES_ACCESSED is False

print(
    json.dumps(
        exact_nested_metadata_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "canonical_chunk_count": 3488,
    "source_types": [
      "book",
      "transcript",
      "website"
    ],
    "test_queries_accessed": false
  },
  "direct_field_audit": {
    "speaker_keys": {
      "book": {
        "record_count": 351,
        "populated_record_count": 0,
        "distinct_value_count": 0,
        "sample_values": []
      },
      "transcript": {
        "record_count": 2041,
        "populated_record_count": 2041,
        "distinct_value_count": 76,
        "sample_values": [
          "Dr. Komeil Roudi",
          "EXPLICIT_SOURCE_MARKER",
          "MANIFEST_ATTRIBUTED",
          "NARRATIVE_CUE",
          "UNRESOLVED",
          "false",
          "true",
          "آقای تهمتن",
          "آقای داودی",
          "آقای دکتر رودی",
          "استاد",
          "افسر موموندی",
          "حامد زحمتکش",
          "خانم تختی‌پور",
          "خانم دکتر جلالی",
          "خانم متولیان",
          "خانم کاظمی",
          "دکتر اخوان",
          "

<div dir="rtl" align="right">

## قرارداد Canonical برای Metadata Filtering

براساس Schema واقعی Corpus، قرارداد فیلتر NB16 به‌شکل زیر تعریف می‌شود:

- فیلتر منبع: `source_id` و `source_type`؛
- فیلتر گوینده: عضو دوم رکوردهای `speaker_keys`؛
- فیلتر نویسندگی Website: `direct_author` یا `editorial_supervision`؛
- فیلتر منشأ محتوا: `primary_author` یا `supervised`؛
- فیلتر وضعیت حقوق: `status`، `basis` و `inferred_release_permission`.

نام شخص Author در Corpus Canonical ذخیره نشده است. بنابراین سیستم نباید نام نویسنده را حدس بزند. فیلتر نویسندگی در این نسخه فقط نوع انتساب محتوا را مشخص می‌کند.

در این مرحله قرارداد Canonical ساخته و با خروجی‌های Hybrid Development آزمایش می‌شود. تمام نتایج فیلترشده باید دقیقاً با شرط درخواستی سازگار باشند.

</div>

In [17]:
# Code Cell 17 — Canonical nested metadata filter contract

SPEAKER_STATUS_VALUES = {
    "EXPLICIT_SOURCE_MARKER",
    "MANIFEST_ATTRIBUTED",
    "NARRATIVE_CUE",
    "UNRESOLVED",
}


def normalized_scalar(value):
    if value is None:
        return None

    if isinstance(value, bool):
        return str(value).lower()

    normalized = str(value).strip()
    return normalized if normalized else None


def extract_speaker_metadata(record):
    attribution_context = record.get(
        "attribution_context"
    )

    if not isinstance(
        attribution_context,
        dict,
    ):
        return {
            "speaker_identities": (),
            "speaker_statuses": (),
            "speaker_primary_signals": (),
        }

    speaker_rows = attribution_context.get(
        "speaker_keys",
        [],
    )

    if not isinstance(speaker_rows, list):
        speaker_rows = []

    speaker_identities = set()
    speaker_statuses = set()
    speaker_primary_signals = set()

    for speaker_row in speaker_rows:
        if not isinstance(
            speaker_row,
            (list, tuple),
        ):
            continue

        speaker_identity = (
            normalized_scalar(
                speaker_row[1]
            )
            if len(speaker_row) > 1
            else None
        )

        speaker_status = (
            normalized_scalar(
                speaker_row[2]
            )
            if len(speaker_row) > 2
            else None
        )

        primary_signal = (
            normalized_scalar(
                speaker_row[3]
            )
            if len(speaker_row) > 3
            else None
        )

        if (
            speaker_identity
            and speaker_identity
            not in SPEAKER_STATUS_VALUES
            and speaker_identity
            not in {"true", "false"}
        ):
            speaker_identities.add(
                speaker_identity
            )

        if speaker_status:
            speaker_statuses.add(
                speaker_status
            )

        if primary_signal:
            speaker_primary_signals.add(
                primary_signal
            )

    return {
        "speaker_identities": tuple(
            sorted(speaker_identities)
        ),
        "speaker_statuses": tuple(
            sorted(speaker_statuses)
        ),
        "speaker_primary_signals": tuple(
            sorted(
                speaker_primary_signals
            )
        ),
    }


def extract_authority_metadata(record):
    attribution_context = record.get(
        "attribution_context"
    )

    if not isinstance(
        attribution_context,
        dict,
    ):
        attribution_context = {}

    content_authority_types = tuple(
        sorted(
            {
                value
                for value in (
                    normalize_metadata_value(
                        attribution_context.get(
                            "content_authority_types"
                        )
                    )
                )
                if value
            }
        )
    )

    content_origins = tuple(
        sorted(
            {
                value
                for value in (
                    normalize_metadata_value(
                        attribution_context.get(
                            "content_origins"
                        )
                    )
                )
                if value
            }
        )
    )

    return {
        "author_attribution_modes": (
            content_authority_types
        ),
        "content_origins": content_origins,
    }


def extract_rights_metadata(record):
    rights_root = record.get("rights")

    if not isinstance(rights_root, dict):
        rights_root = {}

    inferred_permission = (
        normalized_scalar(
            rights_root.get(
                "inferred_release_permission"
            )
        )
    )

    statuses = set()
    bases = set()
    rights_holders = set()
    allowed_uses = set()
    evidence_ids = set()

    member_rights = rights_root.get(
        "member_rights",
        [],
    )

    if isinstance(member_rights, dict):
        member_rights = [member_rights]

    if not isinstance(member_rights, list):
        member_rights = []

    for member_right_row in member_rights:
        if not isinstance(
            member_right_row,
            dict,
        ):
            continue

        rights_object = (
            member_right_row.get("rights")
        )

        if not isinstance(
            rights_object,
            dict,
        ):
            continue

        field_targets = (
            ("status", statuses),
            ("basis", bases),
            ("rights_holder", rights_holders),
            ("allowed_uses", allowed_uses),
            (
                "rights_evidence_id",
                evidence_ids,
            ),
        )

        for field_name, target in field_targets:
            values = normalize_metadata_value(
                rights_object.get(
                    field_name
                )
            )

            target.update(
                value
                for value in values
                if value
            )

    return {
        "rights_statuses": tuple(
            sorted(statuses)
        ),
        "rights_bases": tuple(
            sorted(bases)
        ),
        "rights_holders": tuple(
            sorted(rights_holders)
        ),
        "allowed_uses": tuple(
            sorted(allowed_uses)
        ),
        "rights_evidence_ids": tuple(
            sorted(evidence_ids)
        ),
        "inferred_release_permission": (
            (inferred_permission,)
            if inferred_permission
            is not None
            else ()
        ),
    }


canonical_filter_metadata_by_id = {}

for chunk_id, record in (
    chunk_record_by_id.items()
):
    speaker_metadata = (
        extract_speaker_metadata(record)
    )

    authority_metadata = (
        extract_authority_metadata(record)
    )

    rights_metadata = (
        extract_rights_metadata(record)
    )

    canonical_filter_metadata_by_id[
        chunk_id
    ] = {
        "source_id": (
            str(record["source_id"]),
        ),
        "source_type": (
            str(record["source_type"]),
        ),
        **speaker_metadata,
        **authority_metadata,
        **rights_metadata,
    }

assert len(
    canonical_filter_metadata_by_id
) == 3488


CANONICAL_FILTER_FIELDS = (
    "source_id",
    "source_type",
    "speaker_identities",
    "speaker_statuses",
    "speaker_primary_signals",
    "author_attribution_modes",
    "content_origins",
    "rights_statuses",
    "rights_bases",
    "rights_holders",
    "allowed_uses",
    "rights_evidence_ids",
    "inferred_release_permission",
)


canonical_filter_coverage = {}

for field_name in CANONICAL_FILTER_FIELDS:
    populated_chunk_ids = [
        chunk_id
        for chunk_id, metadata
        in canonical_filter_metadata_by_id.items()
        if metadata[field_name]
    ]

    distinct_values = sorted(
        {
            value
            for chunk_id in populated_chunk_ids
            for value in (
                canonical_filter_metadata_by_id[
                    chunk_id
                ][field_name]
            )
        }
    )

    canonical_filter_coverage[
        field_name
    ] = {
        "populated_chunk_count": len(
            populated_chunk_ids
        ),
        "coverage_ratio": float(
            len(populated_chunk_ids)
            / len(chunk_records)
        ),
        "distinct_value_count": len(
            distinct_values
        ),
        "sample_values": (
            distinct_values[:25]
        ),
        "availability": (
            "AVAILABLE"
            if populated_chunk_ids
            else "ABSENT"
        ),
    }


assert canonical_filter_coverage[
    "source_id"
]["populated_chunk_count"] == 3488

assert canonical_filter_coverage[
    "source_type"
]["populated_chunk_count"] == 3488

assert 0 < canonical_filter_coverage["speaker_identities"]["populated_chunk_count"] <= 2041

assert canonical_filter_coverage[
    "author_attribution_modes"
]["populated_chunk_count"] == 1096

assert canonical_filter_coverage[
    "content_origins"
]["populated_chunk_count"] == 1096

assert canonical_filter_coverage[
    "inferred_release_permission"
]["populated_chunk_count"] == 3488


def canonical_metadata_matches(
    chunk_id,
    field_name,
    allowed_values,
):
    allowed_values = {
        normalized_scalar(value)
        for value in allowed_values
    }

    allowed_values.discard(None)

    chunk_values = set(
        canonical_filter_metadata_by_id[
            str(chunk_id)
        ][field_name]
    )

    return bool(
        chunk_values & allowed_values
    )


def apply_canonical_metadata_filter(
    ranked_chunk_ids,
    field_name,
    allowed_values,
):
    assert field_name in (
        CANONICAL_FILTER_FIELDS
    )

    return [
        str(chunk_id)
        for chunk_id in ranked_chunk_ids
        if canonical_metadata_matches(
            chunk_id=chunk_id,
            field_name=field_name,
            allowed_values=allowed_values,
        )
    ]


canonical_filter_test_results = {}

for field_name in CANONICAL_FILTER_FIELDS:
    available_values = sorted(
        {
            value
            for metadata
            in canonical_filter_metadata_by_id.values()
            for value in metadata[field_name]
        }
    )

    checked_filter_count = 0
    checked_result_count = 0
    mismatch_count = 0

    sampled_values = (
        available_values[:25]
    )

    for value in sampled_values:
        for candidate_id in (
            development_candidate_ids_ordered
        ):
            ranked_chunk_ids = (
                extract_ranked_chunk_ids(
                    weighted_result_by_candidate_id[
                        candidate_id
                    ]
                )
            )

            filtered_chunk_ids = (
                apply_canonical_metadata_filter(
                    ranked_chunk_ids=(
                        ranked_chunk_ids
                    ),
                    field_name=field_name,
                    allowed_values={value},
                )
            )

            if not filtered_chunk_ids:
                continue

            checked_filter_count += 1
            checked_result_count += len(
                filtered_chunk_ids
            )

            mismatch_count += sum(
                not canonical_metadata_matches(
                    chunk_id=chunk_id,
                    field_name=field_name,
                    allowed_values={value},
                )
                for chunk_id
                in filtered_chunk_ids
            )

    if sampled_values:
        assert checked_filter_count > 0
        assert checked_result_count > 0
        assert mismatch_count == 0

    canonical_filter_test_results[
        field_name
    ] = {
        "sampled_value_count": len(
            sampled_values
        ),
        "checked_filter_count": (
            checked_filter_count
        ),
        "checked_result_count": (
            checked_result_count
        ),
        "mismatch_count": mismatch_count,
        "status": (
            "PASS"
            if sampled_values
            else "FIELD_PRESENT_BUT_NO_FILTERABLE_VALUE"
        ),
    }


speaker_values = set(
    canonical_filter_coverage[
        "speaker_identities"
    ]["sample_values"]
)

assert not (
    speaker_values
    & SPEAKER_STATUS_VALUES
)

assert not (
    speaker_values
    & {"true", "false"}
)

assert canonical_filter_test_results[
    "source_id"
]["status"] == "PASS"

assert canonical_filter_test_results[
    "source_type"
]["status"] == "PASS"

assert canonical_filter_test_results[
    "speaker_identities"
]["status"] == "PASS"

assert canonical_filter_test_results[
    "author_attribution_modes"
]["status"] == "PASS"

assert canonical_filter_test_results[
    "rights_statuses"
]["status"] == "PASS"

assert canonical_filter_test_results[
    "inferred_release_permission"
]["status"] == "PASS"

assert TEST_QUERIES_ACCESSED is False


canonical_metadata_filter_result = {
    "scope": {
        "canonical_chunk_count": len(
            chunk_records
        ),
        "development_query_count": len(
            development_candidate_ids_ordered
        ),
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
    },
    "canonical_filter_fields": {
        "source": [
            "source_id",
            "source_type",
        ],
        "speaker": [
            "speaker_identities",
            "speaker_statuses",
            "speaker_primary_signals",
        ],
        "author": {
            "supported_filter": (
                "author_attribution_modes"
            ),
            "values": [
                "direct_author",
                "editorial_supervision",
            ],
            "personal_author_identity": (
                "NOT_PRESENT_AND_NOT_INFERRED"
            ),
        },
        "content_authority": [
            "author_attribution_modes",
            "content_origins",
        ],
        "rights": [
            "rights_statuses",
            "rights_bases",
            "rights_holders",
            "allowed_uses",
            "rights_evidence_ids",
            "inferred_release_permission",
        ],
    },
    "coverage": (
        canonical_filter_coverage
    ),
    "filter_contract_tests": (
        canonical_filter_test_results
    ),
    "roadmap_requirement_status": {
        "source_filter": "PASS",
        "speaker_filter": "PASS",
        "author_attribution_filter": "PASS",
        "content_authority_filter": "PASS",
        "rights_filter": "PASS",
        "personal_author_name_filter": (
            "DOCUMENTED_UPSTREAM_METADATA_LIMITATION"
        ),
    },
    "synthetic_metadata_created": False,
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "files_created": 0,
    "paid_api_calls": 0,
    "canonical_metadata_filter_status": "PASS",
}

print(
    json.dumps(
        canonical_metadata_filter_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "canonical_chunk_count": 3488,
    "development_query_count": 2664,
    "test_queries_accessed": false
  },
  "canonical_filter_fields": {
    "source": [
      "source_id",
      "source_type"
    ],
    "speaker": [
      "speaker_identities",
      "speaker_statuses",
      "speaker_primary_signals"
    ],
    "author": {
      "supported_filter": "author_attribution_modes",
      "values": [
        "direct_author",
        "editorial_supervision"
      ],
      "personal_author_identity": "NOT_PRESENT_AND_NOT_INFERRED"
    },
    "content_authority": [
      "author_attribution_modes",
      "content_origins"
    ],
    "rights": [
      "rights_statuses",
      "rights_bases",
      "rights_holders",
      "allowed_uses",
      "rights_evidence_ids",
      "inferred_release_permission"
    ]
  },
  "coverage": {
    "source_id": {
      "populated_chunk_count": 3488,
      "coverage_ratio": 1.0,
      "distinct_value_count": 715,
      "sample_values": [
      

<div dir="rtl" align="right">

## Ablation نهایی و Freeze تنظیم Hybrid با ADR

در این مرحله Dense، Sparse، RRF، Weighted Fusion و Weighted Fusion همراه با Source Diversity مستقیماً مقایسه می‌شوند.

معیار اصلی انتخاب `nDCG@10` است. معیارهای MRR، Hit، Recall، پایداری Bootstrap، Candidate Depth، Latency و تنوع منبع نیز در تصمیم نهایی لحاظ می‌شوند.

براساس آزمایش‌های Development:

- Weighted Fusion از Dense، Sparse و RRF بهتر است؛
- بهبود `nDCG@10` نسبت به Sparse و RRF دارای بازه اطمینان کاملاً مثبت است؛
- Candidate Depth برابر ۱۰۰ بهترین کیفیت را دارد؛
- سقف سه Chunk از هر منبع قابل‌استفاده است، اما کیفیت Retrieval را اندکی کاهش می‌دهد؛
- خروجی بدون Hard Cap به‌طور طبیعی تنوع منبع بالایی دارد.

بنابراین Hard Cap در تنظیم پیش‌فرض فعال نمی‌شود، اما سیاست سقف سه Chunk به‌عنوان قابلیت اختیاری و Versioned ثبت می‌شود.

پس از اجرای موفق این سل، تنظیم Hybrid و ADR فریز می‌شوند. Test هنوز در همین سل خوانده نمی‌شود و ارزیابی مستقل آن در مرحله بعد انجام خواهد شد.

</div>

In [18]:
# Code Cell 18 — Final ablation, ADR, and development configuration freeze

HYBRID_RELEASE_ID = (
    "rag_finance_phase2_hybrid_v1"
)

HYBRID_CONFIGURATION_SCHEMA_VERSION = "1.0"
HYBRID_TOP_K = 10

HYBRID_CONFIGURATION_PATH = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
    / "hybrid_configuration_v1.json"
)

HYBRID_ADR_PATH = (
    PROJECT_ROOT
    / "reports/phase2"
    / "hybrid_retrieval_adr_v1.json"
)

assert TEST_SPLIT_LOCKED is True
assert HYBRID_CONFIGURATION_FROZEN is False
assert TEST_QUERIES_ACCESSED is False


selected_diversity_candidate = (
    metadata_and_diversity_result[
        "source_diversity"
    ]["selected_candidate_policy"]
)

assert selected_diversity_candidate is not None
assert (
    selected_diversity_candidate[
        "maximum_chunks_per_source"
    ] == 3
)
assert (
    selected_diversity_candidate[
        "quality_gate_passed"
    ] is True
)


ablation_rows = [
    {
        "system": "dense_only",
        "family": "DENSE",
        "configuration_id": (
            "dense_e5_qdrant_cosine_depth_100"
        ),
        "metrics": dense_recomputed_metrics,
    },
    {
        "system": "sparse_only",
        "family": "SPARSE",
        "configuration_id": (
            "fa_join_zwnj_stopwords_v1"
            "__k1_1.20__b_0.30"
            "__depth_100"
        ),
        "metrics": sparse_recomputed_metrics,
    },
    {
        "system": "rrf_hybrid",
        "family": "RRF",
        "configuration_id": (
            best_rrf_configuration_id
        ),
        "metrics": rrf_recomputed_metrics,
    },
    {
        "system": "weighted_hybrid",
        "family": "WEIGHTED_SCORE_FUSION",
        "configuration_id": (
            best_weighted_configuration_id
        ),
        "metrics": (
            weighted_recomputed_metrics
        ),
    },
    {
        "system": (
            "weighted_hybrid_source_cap_3"
        ),
        "family": (
            "WEIGHTED_SCORE_FUSION_WITH_DIVERSITY"
        ),
        "configuration_id": (
            f"{best_weighted_configuration_id}"
            "__source_cap_3"
        ),
        "metrics": (
            selected_diversity_candidate[
                "metrics"
            ]
        ),
    },
]


ablation_rows = sorted(
    ablation_rows,
    key=lambda row: (
        -row["metrics"]["ndcg_at_10"],
        -row["metrics"]["reciprocal_rank"],
        -row["metrics"]["hit_at_10"],
        row["configuration_id"],
    ),
)

assert (
    ablation_rows[0]["system"]
    == "weighted_hybrid"
)


selected_weighted_metrics = (
    weighted_recomputed_metrics
)

selected_delta_vs_dense = {
    metric_name: float(
        selected_weighted_metrics[metric_name]
        - dense_recomputed_metrics[
            metric_name
        ]
    )
    for metric_name in METRIC_NAMES
}

selected_delta_vs_sparse = {
    metric_name: float(
        selected_weighted_metrics[metric_name]
        - sparse_recomputed_metrics[
            metric_name
        ]
    )
    for metric_name in METRIC_NAMES
}

selected_delta_vs_rrf = {
    metric_name: float(
        selected_weighted_metrics[metric_name]
        - rrf_recomputed_metrics[
            metric_name
        ]
    )
    for metric_name in METRIC_NAMES
}

assert (
    selected_delta_vs_dense[
        "ndcg_at_10"
    ] > 0.0
)

assert (
    selected_delta_vs_sparse[
        "ndcg_at_10"
    ] > 0.0
)

assert (
    selected_delta_vs_rrf[
        "ndcg_at_10"
    ] > 0.0
)

assert (
    weighted_vs_sparse_bootstrap[
        "confidence_interval_95"
    ]["lower"] > 0.0
)

assert (
    weighted_vs_rrf_bootstrap[
        "confidence_interval_95"
    ]["lower"] > 0.0
)


selected_weighted_configuration = {
    "fusion_method": (
        "WEIGHTED_SCORE_FUSION"
    ),
    "normalization_method": "minmax",
    "candidate_depth": 100,
    "top_k": HYBRID_TOP_K,
    "dense_weight": 0.4,
    "sparse_weight": 0.6,
    "positive_sparse_scores_only": True,
    "deterministic_tie_break": (
        "CHUNK_ID_ASCENDING"
    ),
}

assert (
    best_weighted_summary[
        "normalization_method"
    ] == selected_weighted_configuration[
        "normalization_method"
    ]
)

assert (
    best_weighted_summary[
        "candidate_depth"
    ] == selected_weighted_configuration[
        "candidate_depth"
    ]
)

assert np.isclose(
    best_weighted_summary["dense_weight"],
    selected_weighted_configuration[
        "dense_weight"
    ],
)

assert np.isclose(
    best_weighted_summary["sparse_weight"],
    selected_weighted_configuration[
        "sparse_weight"
    ],
)


development_cache_path = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
    / "query_embedding_cache_v1"
    / "development_query_embeddings.npy"
)

development_cache_metadata_path = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
    / "query_embedding_cache_v1"
    / "development_query_embeddings.json"
)

assert development_cache_path.exists()
assert (
    development_cache_metadata_path.exists()
)


def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")


def canonical_payload_sha256(value):
    return hashlib.sha256(
        canonical_json_bytes(value)
    ).hexdigest()


def write_json_atomically(path, value):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)

sha256_file = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
hybrid_configuration_payload = {
    "artifact_type": (
        "NB16_HYBRID_CONFIGURATION"
    ),
    "schema_version": (
        HYBRID_CONFIGURATION_SCHEMA_VERSION
    ),
    "release_id": HYBRID_RELEASE_ID,
    "selection_scope": {
        "corpus": (
            "ALL_CANONICAL_CHUNKS"
        ),
        "selection_queries": (
            "DEVELOPMENT_ONLY"
        ),
        "development_query_count": len(
            development_candidate_ids_ordered
        ),
        "test_queries_accessed": False,
        "test_access_policy": (
            "LOCKED_DURING_CONFIGURATION_SELECTION"
        ),
    },
    "canonical_inputs": {
        "chunk_sha256": (
            "3891c4ecad9332ba753cd605bbccd99da"
            "9d063f64934a30ee39e495349cffb50"
        ),
        "golden_dataset_sha256": (
            "b9ca18e710686ddca48a25c637840b7"
            "16adb306717cafd8ec7c96ce9e182e6f2"
        ),
        "gold_mapping_sha256": (
            "42216f67665cfc50697789dffe4eb550"
            "839ee22bcbfc0489accc407519dad4d1"
        ),
        "split_sha256": (
            "68df1196f683b6ab9a9df08c015f001b"
            "0caca056331eb80d0048aec783736d8f"
        ),
        "dense_release_sha256": (
            "706c020caa2bda77c767fda8cd1b574"
            "98818afdbdbbbdd2a72f898b55ca6ca50"
        ),
        "dense_logical_index_sha256": (
            "2fc5e50ebbd1e1e66d59c703eb48f136"
            "7b2d3deadfed62efb9d5a5e6959f7ef4"
        ),
        "sparse_release_sha256": (
            "879f52850a71629f02437c1ab1173e3"
            "f9e6c55ec5441ad4649784be63935c51b"
        ),
        "sparse_configuration_sha256": (
            "96d66676d20bacd764ec9a163995dd482"
            "66c90c2acea1c8a5f77d95d9efb6ecc"
        ),
        "sparse_logical_index_sha256": (
            "ce09bb4749da03c9dfa1a849de18fc8c"
            "490ce6b09eb34a30a96b9fedc478a8eb"
        ),
        "development_query_embedding_sha256": (
            sha256_file(
                development_cache_path
            )
        ),
        "development_query_embedding_metadata_sha256": (
            sha256_file(
                development_cache_metadata_path
            )
        ),
    },
    "dense": {
        "collection_name": (
            DENSE_COLLECTION_NAME
        ),
        "model_id": (
            "intfloat/multilingual-e5-small"
        ),
        "model_revision": (
            "614241f622f53c4eeff9890bdc4f31cfecc418b3"
        ),
        "query_prefix": "query:",
        "dimension": 384,
        "distance": "COSINE",
        "mutation_allowed": False,
        "corpus_embedding_recomputation": (
            False
        ),
    },
    "sparse": {
        "analyzer_id": (
            "fa_join_zwnj_stopwords_v1"
        ),
        "implementation": (
            "local_robertson_bm25_v1"
        ),
        "k1": 1.2,
        "b": 0.3,
        "mutation_allowed": False,
    },
    "hybrid": (
        selected_weighted_configuration
    ),
    "metadata_filters": {
        "source": [
            "source_id",
            "source_type",
        ],
        "speaker": [
            "speaker_identities",
            "speaker_statuses",
            "speaker_primary_signals",
        ],
        "author_attribution": [
            "author_attribution_modes",
            "content_origins",
        ],
        "rights": [
            "rights_statuses",
            "rights_bases",
            "rights_holders",
            "allowed_uses",
            "rights_evidence_ids",
            "inferred_release_permission",
        ],
        "personal_author_identity": (
            "NOT_PRESENT_AND_NOT_INFERRED"
        ),
    },
    "source_diversity": {
        "default_policy": (
            "NO_HARD_CAP"
        ),
        "reason": (
            "Unrestricted weighted retrieval has "
            "high natural source diversity and "
            "higher retrieval quality."
        ),
        "baseline_mean_unique_sources_at_10": (
            baseline_source_diversity[
                "mean_unique_source_count_at_10"
            ]
        ),
        "optional_policy": {
            "policy_id": (
                "maximum_3_chunks_per_source_v1"
            ),
            "maximum_chunks_per_source": 3,
            "development_metrics": (
                selected_diversity_candidate[
                    "metrics"
                ]
            ),
            "ndcg_at_10_delta": (
                selected_diversity_candidate[
                    "delta_vs_unrestricted_weighted"
                ]["ndcg_at_10"]
            ),
            "activation": (
                "EXPLICIT_CONSUMER_REQUEST_ONLY"
            ),
        },
    },
    "development_selection": {
        "primary_metric": "ndcg_at_10",
        "metrics": (
            selected_weighted_metrics
        ),
        "delta_vs_dense": (
            selected_delta_vs_dense
        ),
        "delta_vs_sparse": (
            selected_delta_vs_sparse
        ),
        "delta_vs_best_rrf": (
            selected_delta_vs_rrf
        ),
        "bootstrap": {
            "weighted_vs_sparse": (
                weighted_vs_sparse_bootstrap
            ),
            "weighted_vs_best_rrf": (
                weighted_vs_rrf_bootstrap
            ),
        },
    },
    "freeze_rules": {
        "fusion_mutation_after_test": False,
        "weight_mutation_after_test": False,
        "candidate_depth_mutation_after_test": False,
        "normalization_mutation_after_test": False,
        "metadata_contract_mutation_after_test": False,
        "test_result_may_change_configuration": False,
        "test_result_role": (
            "INDEPENDENT_FINAL_EVALUATION_ONLY"
        ),
    },
    "paid_api_calls": 0,
    "configuration_status": (
        "FROZEN_BEFORE_TEST_ACCESS"
    ),
}

hybrid_configuration_sha256 = (
    canonical_payload_sha256(
        hybrid_configuration_payload
    )
)

hybrid_configuration = {
    **hybrid_configuration_payload,
    "configuration_sha256": (
        hybrid_configuration_sha256
    ),
}

hybrid_adr_payload = {
    "artifact_type": (
        "NB16_HYBRID_RETRIEVAL_ADR"
    ),
    "schema_version": "1.0",
    "decision_id": (
        "ADR_NB16_HYBRID_CONFIGURATION_V1"
    ),
    "status": "ACCEPTED_AND_FROZEN",
    "context": (
        "Select a deterministic hybrid retrieval "
        "configuration using development queries only."
    ),
    "considered_alternatives": (
        ablation_rows
    ),
    "decision": {
        "selected_system": (
            "weighted_hybrid"
        ),
        "configuration_id": (
            best_weighted_configuration_id
        ),
        "configuration_sha256": (
            hybrid_configuration_sha256
        ),
        "fusion": (
            selected_weighted_configuration
        ),
        "default_source_diversity": (
            "NO_HARD_CAP"
        ),
        "optional_source_diversity": (
            "maximum_3_chunks_per_source_v1"
        ),
    },
    "decision_reasons": [
        (
            "Highest development nDCG@10 among "
            "evaluated systems."
        ),
        (
            "Paired bootstrap confidence intervals "
            "against Sparse and RRF are fully positive."
        ),
        (
            "Candidate depth 100 produced the best "
            "development retrieval quality."
        ),
        (
            "Natural Top-10 source diversity is high."
        ),
        (
            "Hard source caps reduce retrieval quality "
            "and therefore are not enabled by default."
        ),
    ],
    "known_limitations": [
        (
            "Personal author identity is not present "
            "in the canonical corpus."
        ),
        (
            "Author filtering is supported through "
            "direct_author versus editorial_supervision."
        ),
        (
            "Rights holder, allowed uses and rights "
            "evidence IDs are currently unpopulated."
        ),
        (
            "No test result may be used to revise "
            "this frozen configuration."
        ),
    ],
    "test_queries_accessed": False,
    "paid_api_calls": 0,
}

hybrid_adr_sha256 = (
    canonical_payload_sha256(
        hybrid_adr_payload
    )
)

hybrid_adr = {
    **hybrid_adr_payload,
    "adr_sha256": hybrid_adr_sha256,
}

write_json_atomically(
    HYBRID_CONFIGURATION_PATH,
    hybrid_configuration,
)

write_json_atomically(
    HYBRID_ADR_PATH,
    hybrid_adr,
)

assert HYBRID_CONFIGURATION_PATH.exists()
assert HYBRID_ADR_PATH.exists()

written_configuration = json.loads(
    HYBRID_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

written_adr = json.loads(
    HYBRID_ADR_PATH.read_text(
        encoding="utf-8"
    )
)

written_configuration_payload = {
    key: value
    for key, value
    in written_configuration.items()
    if key != "configuration_sha256"
}

written_adr_payload = {
    key: value
    for key, value
    in written_adr.items()
    if key != "adr_sha256"
}

assert (
    canonical_payload_sha256(
        written_configuration_payload
    )
    == hybrid_configuration_sha256
)

assert (
    canonical_payload_sha256(
        written_adr_payload
    )
    == hybrid_adr_sha256
)

assert (
    written_configuration[
        "configuration_status"
    ] == "FROZEN_BEFORE_TEST_ACCESS"
)

assert (
    written_adr["status"]
    == "ACCEPTED_AND_FROZEN"
)

HYBRID_CONFIGURATION_FROZEN = True

assert HYBRID_CONFIGURATION_FROZEN is True
assert TEST_QUERIES_ACCESSED is False


hybrid_freeze_result = {
    "release_id": HYBRID_RELEASE_ID,
    "ablation": {
        "evaluated_system_count": len(
            ablation_rows
        ),
        "ranking": ablation_rows,
        "selected_system": (
            "weighted_hybrid"
        ),
    },
    "selected_configuration": (
        selected_weighted_configuration
    ),
    "development_metrics": (
        selected_weighted_metrics
    ),
    "source_diversity_decision": {
        "default": "NO_HARD_CAP",
        "optional": (
            "maximum_3_chunks_per_source_v1"
        ),
    },
    "configuration": {
        "path": str(
            HYBRID_CONFIGURATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": sha256_file(
            HYBRID_CONFIGURATION_PATH
        ),
        "payload_sha256": (
            hybrid_configuration_sha256
        ),
    },
    "adr": {
        "path": str(
            HYBRID_ADR_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": sha256_file(
            HYBRID_ADR_PATH
        ),
        "payload_sha256": (
            hybrid_adr_sha256
        ),
        "status": (
            written_adr["status"]
        ),
    },
    "test_queries_accessed": False,
    "configuration_frozen": (
        HYBRID_CONFIGURATION_FROZEN
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "files_created": 2,
    "paid_api_calls": 0,
    "hybrid_configuration_freeze_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        hybrid_freeze_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_hybrid_v1",
  "ablation": {
    "evaluated_system_count": 5,
    "ranking": [
      {
        "system": "weighted_hybrid",
        "family": "WEIGHTED_SCORE_FUSION",
        "configuration_id": "weighted__norm_minmax__depth_100__dense_0.40__sparse_0.60",
        "metrics": {
          "hit_at_10": 0.7856606606606606,
          "hit_at_5": 0.7117117117117117,
          "ndcg_at_10": 0.5381727779077033,
          "ndcg_at_5": 0.5115595588576384,
          "precision_at_10": 0.10762012012012014,
          "precision_at_5": 0.18641141141141143,
          "recall_at_10": 0.621242522284189,
          "recall_at_5": 0.5469027062777063,
          "reciprocal_rank": 0.6083461459951199
        }
      },
      {
        "system": "weighted_hybrid_source_cap_3",
        "family": "WEIGHTED_SCORE_FUSION_WITH_DIVERSITY",
        "configuration_id": "weighted__norm_minmax__depth_100__dense_0.40__sparse_0.60__source_cap_3",
        "metrics": {
          "hit_at_

<div dir="rtl" align="right">

## ارزیابی مستقل Hybrid Retrieval روی Test

تنظیم Hybrid اکنون براساس Development فریز شده است. در این مرحله برای اولین بار Test باز می‌شود.

روی ۱۱۴۹ Query مستقل Test عملیات زیر انجام می‌شود:

1. ساخت یا بازیابی Cache بردار Queryهای Test؛
2. Dense Retrieval با Candidate Depth برابر ۱۰۰؛
3. Sparse Retrieval با تنظیم فریز‌شده NB15؛
4. Weighted Fusion با وزن‌های فریز‌شده ۰٫۴ و ۰٫۶؛
5. محاسبه معیارهای Dense، Sparse و Hybrid؛
6. محاسبه Slice Metrics براساس نوع منبع؛
7. ثبت Latency و کنترل عدم تغییر Indexها.

نتیجه Test صرفاً ارزیابی مستقل است و اجازه تغییر Fusion، وزن‌ها، Analyzer، Candidate Depth یا Metadata Contract را ندارد.

</div>

In [19]:
# Code Cell 19 — Independent frozen hybrid evaluation on test split

TEST_QUERY_CACHE_DIR = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
    / "query_embedding_cache_v1"
)

TEST_QUERY_EMBEDDING_PATH = (
    TEST_QUERY_CACHE_DIR
    / "test_query_embeddings.npy"
)

TEST_QUERY_METADATA_PATH = (
    TEST_QUERY_CACHE_DIR
    / "test_query_embeddings.json"
)

FROZEN_CANDIDATE_DEPTH = 100
FROZEN_TOP_K = 10
FROZEN_DENSE_WEIGHT = 0.4
FROZEN_SPARSE_WEIGHT = 0.6
FROZEN_NORMALIZATION = "minmax"

assert HYBRID_CONFIGURATION_FROZEN is True
assert HYBRID_CONFIGURATION_PATH.exists()
assert HYBRID_ADR_PATH.exists()


# ------------------------------------------------------------------
# Validate frozen configuration and ADR before test evaluation
# ------------------------------------------------------------------

frozen_configuration = json.loads(
    HYBRID_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

frozen_adr = json.loads(
    HYBRID_ADR_PATH.read_text(
        encoding="utf-8"
    )
)

frozen_configuration_payload = {
    key: value
    for key, value
    in frozen_configuration.items()
    if key != "configuration_sha256"
}

frozen_adr_payload = {
    key: value
    for key, value
    in frozen_adr.items()
    if key != "adr_sha256"
}

assert (
    canonical_payload_sha256(
        frozen_configuration_payload
    )
    == frozen_configuration[
        "configuration_sha256"
    ]
)

assert (
    canonical_payload_sha256(
        frozen_adr_payload
    )
    == frozen_adr["adr_sha256"]
)

assert (
    frozen_configuration[
        "configuration_status"
    ] == "FROZEN_BEFORE_TEST_ACCESS"
)

assert (
    frozen_adr["status"]
    == "ACCEPTED_AND_FROZEN"
)

assert (
    frozen_configuration["hybrid"][
        "fusion_method"
    ] == "WEIGHTED_SCORE_FUSION"
)

assert (
    frozen_configuration["hybrid"][
        "normalization_method"
    ] == FROZEN_NORMALIZATION
)

assert (
    frozen_configuration["hybrid"][
        "candidate_depth"
    ] == FROZEN_CANDIDATE_DEPTH
)

assert (
    frozen_configuration["hybrid"][
        "top_k"
    ] == FROZEN_TOP_K
)

assert np.isclose(
    frozen_configuration["hybrid"][
        "dense_weight"
    ],
    FROZEN_DENSE_WEIGHT,
)

assert np.isclose(
    frozen_configuration["hybrid"][
        "sparse_weight"
    ],
    FROZEN_SPARSE_WEIGHT,
)


# ------------------------------------------------------------------
# Build exact test-query mapping
# ------------------------------------------------------------------

all_question_by_candidate_id = {}

for record in golden_records:
    candidate_id = str(
        record["candidate_id"]
    )

    evaluation_content = record.get(
        "evaluation_content",
        {},
    )

    assert isinstance(
        evaluation_content,
        dict,
    )

    question = str(
        evaluation_content["question"]
    ).strip()

    assert question
    assert (
        candidate_id
        not in all_question_by_candidate_id
    )

    all_question_by_candidate_id[
        candidate_id
    ] = question


test_candidate_ids_ordered = sorted(
    str(candidate_id)
    for candidate_id
    in test_candidate_ids
)

assert len(
    test_candidate_ids_ordered
) == 1149

assert len(
    set(test_candidate_ids_ordered)
) == 1149

assert set(
    test_candidate_ids_ordered
).issubset(
    set(all_question_by_candidate_id)
)

assert not (
    set(test_candidate_ids_ordered)
    & set(development_candidate_ids_ordered)
)


# The configuration was frozen before the first test access.
# This assignment is intentionally idempotent for safe cell reruns.
TEST_SPLIT_LOCKED = False
TEST_QUERIES_ACCESSED = True

test_query_by_candidate_id = {
    candidate_id: (
        all_question_by_candidate_id[
            candidate_id
        ]
    )
    for candidate_id
    in test_candidate_ids_ordered
}

assert len(
    test_query_by_candidate_id
) == 1149


test_query_set_payload = [
    {
        "candidate_id": candidate_id,
        "question": (
            test_query_by_candidate_id[
                candidate_id
            ]
        ),
    }
    for candidate_id
    in test_candidate_ids_ordered
]

test_query_set_sha256 = (
    canonical_payload_sha256(
        test_query_set_payload
    )
)

test_cache_key_payload = {
    "split": "test",
    "query_set_sha256": (
        test_query_set_sha256
    ),
    "model_id": (
        frozen_configuration["dense"][
            "model_id"
        ]
    ),
    "model_revision": (
        frozen_configuration["dense"][
            "model_revision"
        ]
    ),
    "query_prefix": (
        frozen_configuration["dense"][
            "query_prefix"
        ]
    ),
    "normalize_embeddings": True,
}

test_cache_key = canonical_payload_sha256(
    test_cache_key_payload
)


# ------------------------------------------------------------------
# Load or create test query embedding cache
# ------------------------------------------------------------------

test_query_cache_hit = False
test_query_embeddings = None

if (
    TEST_QUERY_EMBEDDING_PATH.exists()
    and TEST_QUERY_METADATA_PATH.exists()
):
    cached_test_metadata = json.loads(
        TEST_QUERY_METADATA_PATH.read_text(
            encoding="utf-8"
        )
    )

    cached_embeddings = np.load(
        TEST_QUERY_EMBEDDING_PATH,
        allow_pickle=False,
    )

    cache_contract_matches = (
        cached_test_metadata.get(
            "cache_key"
        )
        == test_cache_key
        and cached_test_metadata.get(
            "candidate_ids"
        )
        == test_candidate_ids_ordered
        and cached_embeddings.shape
        == (1149, 384)
        and cached_embeddings.dtype
        == np.float32
    )

    if cache_contract_matches:
        test_query_embeddings = (
            cached_embeddings
        )
        test_query_cache_hit = True


if not test_query_cache_hit:
    from sentence_transformers import (
        SentenceTransformer,
    )

    frozen_query_model = SentenceTransformer(
        frozen_configuration["dense"][
            "model_id"
        ],
        revision=frozen_configuration[
            "dense"
        ]["model_revision"],
        local_files_only=True,
    )

    prefixed_test_queries = [
        (
            frozen_configuration[
                "dense"
            ]["query_prefix"]
            + " "
            + test_query_by_candidate_id[
                candidate_id
            ]
        )
        for candidate_id
        in test_candidate_ids_ordered
    ]

    test_query_embeddings = (
        frozen_query_model.encode(
            prefixed_test_queries,
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    ).astype(
        np.float32,
        copy=False,
    )

    assert (
        test_query_embeddings.shape
        == (1149, 384)
    )

    TEST_QUERY_CACHE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.save(
        TEST_QUERY_EMBEDDING_PATH,
        test_query_embeddings,
        allow_pickle=False,
    )

    test_cache_metadata = {
        "artifact_type": (
            "NB16_TEST_QUERY_EMBEDDING_CACHE"
        ),
        "schema_version": "1.0",
        "split": "test",
        "query_count": 1149,
        "candidate_ids": (
            test_candidate_ids_ordered
        ),
        "query_set_sha256": (
            test_query_set_sha256
        ),
        "cache_key": test_cache_key,
        "model_id": (
            frozen_configuration["dense"][
                "model_id"
            ]
        ),
        "model_revision": (
            frozen_configuration["dense"][
                "model_revision"
            ]
        ),
        "query_prefix": (
            frozen_configuration["dense"][
                "query_prefix"
            ]
        ),
        "normalize_embeddings": True,
        "shape": [1149, 384],
        "dtype": "float32",
    }

    write_json_atomically(
        TEST_QUERY_METADATA_PATH,
        test_cache_metadata,
    )


assert test_query_embeddings is not None

assert (
    test_query_embeddings.shape
    == (1149, 384)
)

assert (
    test_query_embeddings.dtype
    == np.float32
)

assert TEST_QUERY_EMBEDDING_PATH.exists()
assert TEST_QUERY_METADATA_PATH.exists()

test_query_embedding_norms = np.linalg.norm(
    test_query_embeddings,
    axis=1,
)

assert np.allclose(
    test_query_embedding_norms,
    1.0,
    rtol=1e-5,
    atol=1e-6,
)


# ------------------------------------------------------------------
# Independent frozen BM25 implementation
# ------------------------------------------------------------------

def frozen_bm25_scores(
    index,
    query_tokens,
    k1,
    b,
):
    assert k1 > 0.0
    assert 0.0 <= b <= 1.0

    document_count = len(
        index.document_ids
    )

    document_lengths = np.asarray(
        index.document_lengths,
        dtype=np.float64,
    )

    average_document_length = float(
        index.average_document_length
    )

    assert document_count == len(
        document_lengths
    )

    assert average_document_length > 0.0

    scores = np.zeros(
        document_count,
        dtype=np.float64,
    )

    query_term_counts = (
        collections.Counter(
            query_tokens
        )
    )

    for token, query_frequency in (
        query_term_counts.items()
    ):
        posting = index.postings.get(
            token
        )

        if posting is None:
            continue

        if (
            isinstance(posting, tuple)
            and len(posting) == 2
        ):
            document_indexes = np.asarray(
                posting[0],
                dtype=np.int64,
            )

            term_frequencies = np.asarray(
                posting[1],
                dtype=np.float64,
            )

        else:
            document_indexes = np.asarray(
                [
                    entry[0]
                    for entry in posting
                ],
                dtype=np.int64,
            )

            term_frequencies = np.asarray(
                [
                    entry[1]
                    for entry in posting
                ],
                dtype=np.float64,
            )

        assert len(document_indexes) == len(
            term_frequencies
        )

        document_frequency = len(
            document_indexes
        )

        if document_frequency == 0:
            continue

        inverse_document_frequency = (
            math.log(
                1.0
                + (
                    document_count
                    - document_frequency
                    + 0.5
                )
                / (
                    document_frequency
                    + 0.5
                )
            )
        )

        length_normalization = (
            1.0
            - b
            + b
            * (
                document_lengths[
                    document_indexes
                ]
                / average_document_length
            )
        )

        denominator = (
            term_frequencies
            + k1
            * length_normalization
        )

        contribution = (
            inverse_document_frequency
            * (
                term_frequencies
                * (k1 + 1.0)
            )
            / denominator
        )

        scores[document_indexes] += (
            contribution
            * float(query_frequency)
        )

    return scores


def frozen_sparse_retrieve(
    question,
    candidate_depth,
):
    query_tokens = (
        analyze_frozen_sparse_text(
            question
        )
    )

    scores = frozen_bm25_scores(
        index=frozen_sparse_index,
        query_tokens=query_tokens,
        k1=1.2,
        b=0.3,
    )

    positive_indexes = np.flatnonzero(
        scores > 0.0
    )

    if positive_indexes.size == 0:
        return {
            "ranked_chunk_ids": [],
            "ranked_scores": [],
        }

    sorted_order = np.lexsort(
        (
            positive_indexes,
            -scores[positive_indexes],
        )
    )

    ranked_indexes = positive_indexes[
        sorted_order
    ][:candidate_depth]

    return {
        "ranked_chunk_ids": [
            str(
                frozen_sparse_index.document_ids[
                    document_index
                ]
            )
            for document_index
            in ranked_indexes
        ],
        "ranked_scores": [
            float(scores[document_index])
            for document_index
            in ranked_indexes
        ],
    }


# ------------------------------------------------------------------
# Frozen Dense retrieval implementation
# ------------------------------------------------------------------

def frozen_dense_retrieve(
    query_vector,
    candidate_depth,
):
    query_response = (
        qdrant_client.query_points(
            collection_name=(
                DENSE_COLLECTION_NAME
            ),
            query=query_vector.tolist(),
            limit=candidate_depth,
            with_payload=True,
            with_vectors=False,
        )
    )

    points = getattr(
        query_response,
        "points",
        query_response,
    )

    ranked_chunk_ids = []
    ranked_scores = []

    for point in points:
        payload = point.payload or {}

        assert "chunk_id" in payload

        ranked_chunk_ids.append(
            str(payload["chunk_id"])
        )

        ranked_scores.append(
            float(point.score)
        )

    assert len(ranked_chunk_ids) <= (
        candidate_depth
    )

    assert len(ranked_chunk_ids) == len(
        ranked_scores
    )

    return {
        "ranked_chunk_ids": (
            ranked_chunk_ids
        ),
        "ranked_scores": ranked_scores,
    }


# ------------------------------------------------------------------
# Frozen weighted fusion implementation
# ------------------------------------------------------------------

def frozen_minmax_scores(scores):
    score_array = np.asarray(
        scores,
        dtype=np.float64,
    )

    if score_array.size == 0:
        return score_array

    minimum_score = float(
        score_array.min()
    )

    maximum_score = float(
        score_array.max()
    )

    if np.isclose(
        maximum_score,
        minimum_score,
    ):
        return np.ones_like(
            score_array,
            dtype=np.float64,
        )

    return (
        score_array - minimum_score
    ) / (
        maximum_score - minimum_score
    )


def frozen_weighted_fusion(
    dense_chunk_ids,
    dense_scores,
    sparse_chunk_ids,
    sparse_scores,
):
    dense_chunk_ids = [
        str(chunk_id)
        for chunk_id
        in dense_chunk_ids[
            :FROZEN_CANDIDATE_DEPTH
        ]
    ]

    sparse_chunk_ids = [
        str(chunk_id)
        for chunk_id
        in sparse_chunk_ids[
            :FROZEN_CANDIDATE_DEPTH
        ]
    ]

    dense_scores = np.asarray(
        dense_scores[
            :FROZEN_CANDIDATE_DEPTH
        ],
        dtype=np.float64,
    )

    sparse_scores = np.asarray(
        sparse_scores[
            :FROZEN_CANDIDATE_DEPTH
        ],
        dtype=np.float64,
    )

    assert len(dense_chunk_ids) == len(
        dense_scores
    )

    assert len(sparse_chunk_ids) == len(
        sparse_scores
    )

    normalized_dense_scores = (
        frozen_minmax_scores(
            dense_scores
        )
    )

    normalized_sparse_scores = (
        frozen_minmax_scores(
            sparse_scores
        )
    )

    fused_scores = (
        collections.defaultdict(float)
    )

    for chunk_id, score in zip(
        dense_chunk_ids,
        normalized_dense_scores,
    ):
        fused_scores[chunk_id] += (
            FROZEN_DENSE_WEIGHT
            * float(score)
        )

    for chunk_id, score in zip(
        sparse_chunk_ids,
        normalized_sparse_scores,
    ):
        fused_scores[chunk_id] += (
            FROZEN_SPARSE_WEIGHT
            * float(score)
        )

    ranked_items = sorted(
        fused_scores.items(),
        key=lambda item: (
            -item[1],
            item[0],
        ),
    )

    return {
        "ranked_chunk_ids": [
            chunk_id
            for chunk_id, score
            in ranked_items
        ],
        "ranked_scores": [
            float(score)
            for chunk_id, score
            in ranked_items
        ],
    }


# ------------------------------------------------------------------
# Execute independent test retrieval
# ------------------------------------------------------------------

canonical_point_count_before_test = (
    qdrant_client.get_collection(
        collection_name=(
            DENSE_COLLECTION_NAME
        )
    ).points_count
)

assert (
    canonical_point_count_before_test
    == 3488
)

test_dense_results = {}
test_sparse_results = {}
test_hybrid_results = {}

test_dense_metric_rows = []
test_sparse_metric_rows = []
test_hybrid_metric_rows = []

test_dense_latencies_ms = []
test_sparse_latencies_ms = []
test_fusion_latencies_ms = []

for query_index, candidate_id in enumerate(
    test_candidate_ids_ordered
):
    question = (
        test_query_by_candidate_id[
            candidate_id
        ]
    )

    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    assert relevant_chunk_ids

    dense_started = time.perf_counter()

    dense_result = frozen_dense_retrieve(
        query_vector=(
            test_query_embeddings[
                query_index
            ]
        ),
        candidate_depth=(
            FROZEN_CANDIDATE_DEPTH
        ),
    )

    test_dense_latencies_ms.append(
        (
            time.perf_counter()
            - dense_started
        )
        * 1000.0
    )

    sparse_started = time.perf_counter()

    sparse_result = (
        frozen_sparse_retrieve(
            question=question,
            candidate_depth=(
                FROZEN_CANDIDATE_DEPTH
            ),
        )
    )

    test_sparse_latencies_ms.append(
        (
            time.perf_counter()
            - sparse_started
        )
        * 1000.0
    )

    fusion_started = time.perf_counter()

    hybrid_result = (
        frozen_weighted_fusion(
            dense_chunk_ids=(
                dense_result[
                    "ranked_chunk_ids"
                ]
            ),
            dense_scores=(
                dense_result[
                    "ranked_scores"
                ]
            ),
            sparse_chunk_ids=(
                sparse_result[
                    "ranked_chunk_ids"
                ]
            ),
            sparse_scores=(
                sparse_result[
                    "ranked_scores"
                ]
            ),
        )
    )

    test_fusion_latencies_ms.append(
        (
            time.perf_counter()
            - fusion_started
        )
        * 1000.0
    )

    dense_metrics = evaluate_ranked_ids(
        dense_result["ranked_chunk_ids"],
        relevant_chunk_ids,
    )

    sparse_metrics = evaluate_ranked_ids(
        sparse_result["ranked_chunk_ids"],
        relevant_chunk_ids,
    )

    hybrid_metrics = evaluate_ranked_ids(
        hybrid_result["ranked_chunk_ids"],
        relevant_chunk_ids,
    )

    test_dense_results[
        candidate_id
    ] = dense_result

    test_sparse_results[
        candidate_id
    ] = sparse_result

    test_hybrid_results[
        candidate_id
    ] = hybrid_result

    test_dense_metric_rows.append(
        {
            "candidate_id": candidate_id,
            **dense_metrics,
        }
    )

    test_sparse_metric_rows.append(
        {
            "candidate_id": candidate_id,
            **sparse_metrics,
        }
    )

    test_hybrid_metric_rows.append(
        {
            "candidate_id": candidate_id,
            **hybrid_metrics,
        }
    )


assert len(test_dense_results) == 1149
assert len(test_sparse_results) == 1149
assert len(test_hybrid_results) == 1149


# ------------------------------------------------------------------
# Aggregate independent test metrics
# ------------------------------------------------------------------

def aggregate_test_metric_rows(rows):
    assert len(rows) == 1149

    return {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row in rows
                ]
            )
        )
        for metric_name in METRIC_NAMES
    }


test_dense_metrics = (
    aggregate_test_metric_rows(
        test_dense_metric_rows
    )
)

test_sparse_metrics = (
    aggregate_test_metric_rows(
        test_sparse_metric_rows
    )
)

test_hybrid_metrics = (
    aggregate_test_metric_rows(
        test_hybrid_metric_rows
    )
)

test_hybrid_delta_vs_dense = {
    metric_name: float(
        test_hybrid_metrics[metric_name]
        - test_dense_metrics[metric_name]
    )
    for metric_name in METRIC_NAMES
}

test_hybrid_delta_vs_sparse = {
    metric_name: float(
        test_hybrid_metrics[metric_name]
        - test_sparse_metrics[metric_name]
    )
    for metric_name in METRIC_NAMES
}


# ------------------------------------------------------------------
# Validate Sparse parity with NB15 independent test
# ------------------------------------------------------------------

EXPECTED_NB15_TEST_TOP_K_METRICS = {
    "hit_at_10": (
        0.7075718015665796
    ),
    "hit_at_5": (
        0.6518711923411662
    ),
    "ndcg_at_10": (
        0.49020522695816876
    ),
    "ndcg_at_5": (
        0.46879829553108543
    ),
    "precision_at_10": (
        0.09451697127937336
    ),
    "precision_at_5": (
        0.16675369886858138
    ),
    "recall_at_10": (
        0.5535776451572797
    ),
    "recall_at_5": (
        0.49356686974180447
    ),
}

sparse_nb15_metric_deltas = {
    metric_name: float(
        test_sparse_metrics[metric_name]
        - expected_value
    )
    for metric_name, expected_value
    in EXPECTED_NB15_TEST_TOP_K_METRICS.items()
}

assert all(
    abs(delta) <= 1e-15
    for delta
    in sparse_nb15_metric_deltas.values()
)


# ------------------------------------------------------------------
# Test source slices
# ------------------------------------------------------------------

test_dense_metric_by_candidate_id = {
    row["candidate_id"]: row
    for row in test_dense_metric_rows
}

test_sparse_metric_by_candidate_id = {
    row["candidate_id"]: row
    for row in test_sparse_metric_rows
}

test_hybrid_metric_by_candidate_id = {
    row["candidate_id"]: row
    for row in test_hybrid_metric_rows
}

test_source_slice_candidate_ids = {}

for candidate_id in (
    test_candidate_ids_ordered
):
    source_signature = (
        relevant_source_signature(
            candidate_id
        )
    )

    test_source_slice_candidate_ids.setdefault(
        source_signature,
        [],
    ).append(candidate_id)


def aggregate_test_slice(
    metric_by_candidate_id,
    candidate_ids,
):
    assert candidate_ids

    return {
        metric_name: float(
            np.mean(
                [
                    metric_by_candidate_id[
                        candidate_id
                    ][metric_name]
                    for candidate_id
                    in candidate_ids
                ]
            )
        )
        for metric_name in METRIC_NAMES
    }


test_source_slice_metrics = {
    source_signature: {
        "query_count": len(
            candidate_ids
        ),
        "dense": aggregate_test_slice(
            test_dense_metric_by_candidate_id,
            candidate_ids,
        ),
        "sparse": aggregate_test_slice(
            test_sparse_metric_by_candidate_id,
            candidate_ids,
        ),
        "hybrid": aggregate_test_slice(
            test_hybrid_metric_by_candidate_id,
            candidate_ids,
        ),
    }
    for source_signature, candidate_ids
    in sorted(
        test_source_slice_candidate_ids.items()
    )
}

assert sum(
    result["query_count"]
    for result
    in test_source_slice_metrics.values()
) == 1149


# ------------------------------------------------------------------
# Paired test bootstrap — reporting only
# ------------------------------------------------------------------

def paired_test_bootstrap(
    challenger_rows,
    reference_rows,
    metric_name,
    sample_count=2000,
    seed=16019,
):
    challenger_by_id = {
        row["candidate_id"]: row
        for row in challenger_rows
    }

    reference_by_id = {
        row["candidate_id"]: row
        for row in reference_rows
    }

    paired_deltas = np.asarray(
        [
            challenger_by_id[
                candidate_id
            ][metric_name]
            - reference_by_id[
                candidate_id
            ][metric_name]
            for candidate_id
            in test_candidate_ids_ordered
        ],
        dtype=np.float64,
    )

    rng = np.random.default_rng(seed)

    bootstrap_means = np.empty(
        sample_count,
        dtype=np.float64,
    )

    batch_size = 100
    query_count = len(paired_deltas)

    for batch_start in range(
        0,
        sample_count,
        batch_size,
    ):
        batch_end = min(
            batch_start + batch_size,
            sample_count,
        )

        sampled_indexes = rng.integers(
            low=0,
            high=query_count,
            size=(
                batch_end - batch_start,
                query_count,
            ),
        )

        bootstrap_means[
            batch_start:batch_end
        ] = paired_deltas[
            sampled_indexes
        ].mean(axis=1)

    confidence_interval = np.quantile(
        bootstrap_means,
        [0.025, 0.975],
    )

    return {
        "metric": metric_name,
        "observed_delta": float(
            paired_deltas.mean()
        ),
        "confidence_interval_95": {
            "lower": float(
                confidence_interval[0]
            ),
            "upper": float(
                confidence_interval[1]
            ),
        },
        "probability_delta_above_zero": float(
            np.mean(
                bootstrap_means > 0.0
            )
        ),
        "bootstrap_sample_count": (
            sample_count
        ),
        "seed": seed,
        "role": (
            "INDEPENDENT_TEST_REPORTING_ONLY"
        ),
    }


test_hybrid_vs_dense_bootstrap = (
    paired_test_bootstrap(
        challenger_rows=(
            test_hybrid_metric_rows
        ),
        reference_rows=(
            test_dense_metric_rows
        ),
        metric_name="ndcg_at_10",
    )
)

test_hybrid_vs_sparse_bootstrap = (
    paired_test_bootstrap(
        challenger_rows=(
            test_hybrid_metric_rows
        ),
        reference_rows=(
            test_sparse_metric_rows
        ),
        metric_name="ndcg_at_10",
    )
)


# ------------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------------

canonical_point_count_after_test = (
    qdrant_client.get_collection(
        collection_name=(
            DENSE_COLLECTION_NAME
        )
    ).points_count
)

assert (
    canonical_point_count_before_test
    == canonical_point_count_after_test
    == 3488
)

assert (
    test_hybrid_metrics["ndcg_at_10"]
    > test_dense_metrics["ndcg_at_10"]
)

assert (
    frozen_configuration[
        "configuration_sha256"
    ] == hybrid_configuration_sha256
)

assert (
    frozen_configuration["hybrid"]
    == selected_weighted_configuration
)

assert TEST_QUERIES_ACCESSED is True
assert HYBRID_CONFIGURATION_FROZEN is True


# ------------------------------------------------------------------
# Final test result
# ------------------------------------------------------------------

independent_test_result = {
    "artifact_type": (
        "NB16_INDEPENDENT_TEST_EVALUATION"
    ),
    "schema_version": "1.0",
    "release_id": HYBRID_RELEASE_ID,
    "scope": {
        "split": "test",
        "query_count": 1149,
        "test_queries_accessed": (
            TEST_QUERIES_ACCESSED
        ),
        "configuration_frozen_before_access": (
            True
        ),
        "test_result_may_change_configuration": (
            False
        ),
    },
    "configuration": {
        "configuration_sha256": (
            hybrid_configuration_sha256
        ),
        **selected_weighted_configuration,
    },
    "query_embedding_cache": {
        "embedding_path": str(
            TEST_QUERY_EMBEDDING_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "metadata_path": str(
            TEST_QUERY_METADATA_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "embedding_sha256": sha256_file(
            TEST_QUERY_EMBEDDING_PATH
        ),
        "metadata_sha256": sha256_file(
            TEST_QUERY_METADATA_PATH
        ),
        "query_set_sha256": (
            test_query_set_sha256
        ),
        "cache_key": test_cache_key,
        "cache_hit": test_query_cache_hit,
        "shape": [1149, 384],
        "dtype": "float32",
        "minimum_norm": float(
            test_query_embedding_norms.min()
        ),
        "maximum_norm": float(
            test_query_embedding_norms.max()
        ),
    },
    "metrics": {
        "dense": test_dense_metrics,
        "sparse": test_sparse_metrics,
        "hybrid": test_hybrid_metrics,
        "hybrid_minus_dense": (
            test_hybrid_delta_vs_dense
        ),
        "hybrid_minus_sparse": (
            test_hybrid_delta_vs_sparse
        ),
    },
    "sparse_nb15_parity": {
        "expected_top_k_metrics": (
            EXPECTED_NB15_TEST_TOP_K_METRICS
        ),
        "metric_deltas": (
            sparse_nb15_metric_deltas
        ),
        "maximum_absolute_delta": float(
            max(
                abs(delta)
                for delta
                in sparse_nb15_metric_deltas.values()
            )
        ),
        "status": "PASS",
    },
    "paired_bootstrap_ndcg_at_10": {
        "hybrid_vs_dense": (
            test_hybrid_vs_dense_bootstrap
        ),
        "hybrid_vs_sparse": (
            test_hybrid_vs_sparse_bootstrap
        ),
    },
    "source_slice_metrics": (
        test_source_slice_metrics
    ),
    "latency_ms": {
        "dense_retrieval": (
            summarize_latency(
                test_dense_latencies_ms,
                (
                    "LOCAL_QDRANT_RETRIEVAL_"
                    "EXCLUDING_QUERY_EMBEDDING"
                ),
            )
        ),
        "sparse_retrieval": (
            summarize_latency(
                test_sparse_latencies_ms,
                "LOCAL_BM25_RETRIEVAL",
            )
        ),
        "fusion": (
            summarize_latency(
                test_fusion_latencies_ms,
                "LOCAL_FUSION_ONLY",
            )
        ),
    },
    "dense_point_count_before": (
        canonical_point_count_before_test
    ),
    "dense_point_count_after": (
        canonical_point_count_after_test
    ),
    "configuration_changed_after_test": (
        False
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "query_embeddings_computed": (
        not test_query_cache_hit
    ),
    "files_created": (
        0 if test_query_cache_hit else 2
    ),
    "paid_api_calls": 0,
    "independent_test_status": "PASS",
}

print(
    json.dumps(
        independent_test_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "artifact_type": "NB16_INDEPENDENT_TEST_EVALUATION",
  "schema_version": "1.0",
  "release_id": "rag_finance_phase2_hybrid_v1",
  "scope": {
    "split": "test",
    "query_count": 1149,
    "test_queries_accessed": true,
    "configuration_frozen_before_access": true,
    "test_result_may_change_configuration": false
  },
  "configuration": {
    "configuration_sha256": "1f618f4870622a6d31e831b54a35d76c96687d6683ab195af171b36214abcbf7",
    "fusion_method": "WEIGHTED_SCORE_FUSION",
    "normalization_method": "minmax",
    "candidate_depth": 100,
    "top_k": 10,
    "dense_weight": 0.4,
    "sparse_weight": 0.6,
    "positive_sparse_scores_only": true,
    "deterministic_tie_break": "CHUNK_ID_ASCENDING"
  },
  "query_embedding_cache": {
    "embedding_path": "artifacts/phase2/hybrid_retrieval/query_embedding_cache_v1/test_query_embeddings.npy",
    "metadata_path": "artifacts/phase2/hybrid_retrieval/query_embedding_cache_v1/test_query_embeddings.json",
    "embedding_sha256": "0e

<div dir="rtl" align="right">

## ساخت Artifactهای نسخه‌دار Hybrid Retrieval

تنظیم Hybrid روی Development انتخاب و پیش از Test فریز شد. ارزیابی مستقل Test نیز بدون تغییر Configuration با موفقیت انجام شد.

در این مرحله خروجی‌های قابل‌تحویل NB16 ساخته می‌شوند:

- Candidateهای Hybrid برای Development و Test؛
- گزارش مستقل Test؛
- Evidence کامل آزمایش‌ها و Ablation؛
- Manifest نسخه‌دار؛
- Release مخصوص مراحل بعدی؛
- Handoff رسمی برای NB17؛
- گزارش Acceptance اولیه.

Candidateهای ذخیره‌شده شامل رتبه، شناسه Chunk و امتیاز Fusion هستند. این Artifact به NB17 اجازه می‌دهد بدون اجرای مجدد Dense و Sparse Retrieval، مرحله Reranking را آغاز کند.

وضعیت Acceptance در پایان این سل هنوز Pending Reload Validation است. Freeze نهایی فقط پس از بارگذاری مستقل Artifactها و اثبات عدم وابستگی به Kernel انجام می‌شود.

</div>

In [20]:
# Code Cell 20 — Materialize versioned hybrid retrieval artifacts

HYBRID_ARTIFACT_ROOT = (
    PROJECT_ROOT
    / "artifacts/phase2/hybrid_retrieval"
)

HYBRID_DATA_ROOT = (
    PROJECT_ROOT
    / "data/phase2/retrieval"
    / "hybrid_retrieval_v1"
)

HYBRID_REPORT_ROOT = (
    PROJECT_ROOT
    / "reports/phase2"
)

HYBRID_CANDIDATES_PATH = (
    HYBRID_DATA_ROOT
    / "hybrid_candidates_v1.jsonl"
)

HYBRID_EVIDENCE_PATH = (
    HYBRID_DATA_ROOT
    / "hybrid_retrieval_evidence_v1.json"
)

HYBRID_MANIFEST_PATH = (
    HYBRID_DATA_ROOT
    / "hybrid_retrieval_manifest.json"
)

HYBRID_TEST_EVALUATION_PATH = (
    HYBRID_REPORT_ROOT
    / "hybrid_retrieval_test_evaluation_v1.json"
)

HYBRID_RELEASE_PATH = (
    HYBRID_ARTIFACT_ROOT
    / "hybrid_retrieval_release_v1.json"
)

NB17_HANDOFF_PATH = (
    HYBRID_ARTIFACT_ROOT
    / "nb17_hybrid_retrieval_handoff_v1.json"
)

HYBRID_ACCEPTANCE_PATH = (
    HYBRID_REPORT_ROOT
    / "hybrid_retrieval_acceptance_v1.json"
)

assert HYBRID_CONFIGURATION_FROZEN is True
assert TEST_QUERIES_ACCESSED is True

assert (
    independent_test_result[
        "independent_test_status"
    ] == "PASS"
)

assert (
    frozen_configuration[
        "configuration_sha256"
    ] == hybrid_configuration_sha256
)


def extract_ranked_scores(result):
    if not isinstance(result, dict):
        return None

    candidate_keys = (
        "ranked_scores",
        "scores",
        "fused_scores",
        "hybrid_scores",
    )

    for key in candidate_keys:
        if key not in result:
            continue

        values = result[key]

        if isinstance(
            values,
            (list, tuple, np.ndarray),
        ):
            return [
                float(value)
                for value in values
            ]

    return None


def development_hybrid_result(
    candidate_id,
):
    stored_result = (
        weighted_result_by_candidate_id[
            candidate_id
        ]
    )

    ranked_chunk_ids = (
        extract_ranked_chunk_ids(
            stored_result
        )
    )

    ranked_scores = extract_ranked_scores(
        stored_result
    )

    if (
        ranked_scores is not None
        and len(ranked_scores)
        >= len(ranked_chunk_ids)
    ):
        return {
            "ranked_chunk_ids": (
                ranked_chunk_ids
            ),
            "ranked_scores": (
                ranked_scores[
                    :len(ranked_chunk_ids)
                ]
            ),
        }

    dense_result = (
        dense_result_by_candidate_id[
            candidate_id
        ]
    )

    sparse_result = (
        sparse_result_by_candidate_id[
            candidate_id
        ]
    )

    dense_chunk_ids = (
        extract_ranked_chunk_ids(
            dense_result
        )
    )

    sparse_chunk_ids = (
        extract_ranked_chunk_ids(
            sparse_result
        )
    )

    dense_scores = extract_ranked_scores(
        dense_result
    )

    sparse_scores = extract_ranked_scores(
        sparse_result
    )

    assert dense_scores is not None
    assert sparse_scores is not None

    rebuilt_result = frozen_weighted_fusion(
        dense_chunk_ids=dense_chunk_ids,
        dense_scores=dense_scores,
        sparse_chunk_ids=sparse_chunk_ids,
        sparse_scores=sparse_scores,
    )

    assert (
        rebuilt_result[
            "ranked_chunk_ids"
        ][:FROZEN_TOP_K]
        == ranked_chunk_ids[
            :FROZEN_TOP_K
        ]
    )

    return rebuilt_result


def write_jsonl_atomically(
    path,
    records,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        for record in records:
            file_handle.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                )
            )
            file_handle.write("\n")

    temporary_path.replace(path)


hybrid_candidate_records = []

for split_name, candidate_ids in (
    (
        "development",
        development_candidate_ids_ordered,
    ),
    (
        "test",
        test_candidate_ids_ordered,
    ),
):
    for candidate_id in candidate_ids:
        if split_name == "development":
            retrieval_result = (
                development_hybrid_result(
                    candidate_id
                )
            )
        else:
            retrieval_result = (
                test_hybrid_results[
                    candidate_id
                ]
            )

        ranked_chunk_ids = [
            str(chunk_id)
            for chunk_id
            in retrieval_result[
                "ranked_chunk_ids"
            ][:FROZEN_CANDIDATE_DEPTH]
        ]

        ranked_scores = [
            float(score)
            for score
            in retrieval_result[
                "ranked_scores"
            ][:FROZEN_CANDIDATE_DEPTH]
        ]

        assert len(ranked_chunk_ids) == len(
            ranked_scores
        )

        assert len(ranked_chunk_ids) > 0
        assert len(ranked_chunk_ids) <= 100

        assert len(ranked_chunk_ids) == len(
            set(ranked_chunk_ids)
        )

        assert set(ranked_chunk_ids).issubset(
            set(chunk_record_by_id)
        )

        candidate_rows = [
            {
                "rank": rank,
                "chunk_id": chunk_id,
                "hybrid_score": score,
            }
            for rank, (chunk_id, score)
            in enumerate(
                zip(
                    ranked_chunk_ids,
                    ranked_scores,
                ),
                start=1,
            )
        ]

        hybrid_candidate_records.append(
            {
                "candidate_id": candidate_id,
                "split": split_name,
                "query_sha256": (
                    hashlib.sha256(
                        all_question_by_candidate_id[
                            candidate_id
                        ].encode("utf-8")
                    ).hexdigest()
                ),
                "candidate_depth": (
                    FROZEN_CANDIDATE_DEPTH
                ),
                "candidate_count": len(
                    candidate_rows
                ),
                "candidates": candidate_rows,
            }
        )


assert len(
    hybrid_candidate_records
) == 3813

assert len(
    {
        record["candidate_id"]
        for record
        in hybrid_candidate_records
    }
) == 3813

assert collections.Counter(
    record["split"]
    for record
    in hybrid_candidate_records
) == {
    "development": 2664,
    "test": 1149,
}


hybrid_candidate_logical_sha256 = (
    hashlib.sha256()
)

for record in hybrid_candidate_records:
    hybrid_candidate_logical_sha256.update(
        canonical_json_bytes(record)
    )
    hybrid_candidate_logical_sha256.update(
        b"\n"
    )

hybrid_candidate_logical_sha256 = (
    hybrid_candidate_logical_sha256.hexdigest()
)


write_jsonl_atomically(
    HYBRID_CANDIDATES_PATH,
    hybrid_candidate_records,
)

write_json_atomically(
    HYBRID_TEST_EVALUATION_PATH,
    independent_test_result,
)

assert HYBRID_CANDIDATES_PATH.exists()
assert HYBRID_TEST_EVALUATION_PATH.exists()


hybrid_evidence = {
    "artifact_type": (
        "NB16_HYBRID_RETRIEVAL_EVIDENCE"
    ),
    "schema_version": "1.0",
    "release_id": HYBRID_RELEASE_ID,
    "configuration": {
        "path": str(
            HYBRID_CONFIGURATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": sha256_file(
            HYBRID_CONFIGURATION_PATH
        ),
        "payload_sha256": (
            hybrid_configuration_sha256
        ),
        "status": (
            "FROZEN_BEFORE_TEST_ACCESS"
        ),
    },
    "adr": {
        "path": str(
            HYBRID_ADR_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": sha256_file(
            HYBRID_ADR_PATH
        ),
        "payload_sha256": (
            hybrid_adr_sha256
        ),
        "status": "ACCEPTED_AND_FROZEN",
    },
    "development": {
        "query_count": 2664,
        "dense_metrics": (
            dense_recomputed_metrics
        ),
        "sparse_metrics": (
            sparse_recomputed_metrics
        ),
        "best_rrf_metrics": (
            rrf_recomputed_metrics
        ),
        "selected_hybrid_metrics": (
            weighted_recomputed_metrics
        ),
        "bootstrap": {
            "hybrid_vs_sparse": (
                weighted_vs_sparse_bootstrap
            ),
            "hybrid_vs_rrf": (
                weighted_vs_rrf_bootstrap
            ),
        },
        "slice_results": (
            development_slice_results
        ),
    },
    "independent_test": {
        "query_count": 1149,
        "metrics": (
            independent_test_result[
                "metrics"
            ]
        ),
        "bootstrap": (
            independent_test_result[
                "paired_bootstrap_ndcg_at_10"
            ]
        ),
        "source_slice_metrics": (
            independent_test_result[
                "source_slice_metrics"
            ]
        ),
        "sparse_nb15_parity": (
            independent_test_result[
                "sparse_nb15_parity"
            ]
        ),
        "status": "PASS",
    },
    "metadata_filtering": {
        "contract": (
            canonical_metadata_filter_result[
                "canonical_filter_fields"
            ]
        ),
        "coverage": (
            canonical_metadata_filter_result[
                "coverage"
            ]
        ),
        "tests": (
            canonical_metadata_filter_result[
                "filter_contract_tests"
            ]
        ),
        "personal_author_identity": (
            "NOT_PRESENT_AND_NOT_INFERRED"
        ),
        "status": "PASS",
    },
    "source_diversity": (
        metadata_and_diversity_result[
            "source_diversity"
        ]
    ),
    "ablation": (
        hybrid_freeze_result["ablation"]
    ),
    "candidate_artifact": {
        "path": str(
            HYBRID_CANDIDATES_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "record_count": 3813,
        "development_record_count": 2664,
        "test_record_count": 1149,
        "candidate_depth": 100,
        "logical_sha256": (
            hybrid_candidate_logical_sha256
        ),
        "file_sha256": sha256_file(
            HYBRID_CANDIDATES_PATH
        ),
    },
    "configuration_changed_after_test": False,
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "paid_api_calls": 0,
    "evidence_status": "PASS",
}

write_json_atomically(
    HYBRID_EVIDENCE_PATH,
    hybrid_evidence,
)

assert HYBRID_EVIDENCE_PATH.exists()


hybrid_manifest = {
    "artifact_type": (
        "NB16_HYBRID_RETRIEVAL_MANIFEST"
    ),
    "schema_version": "1.0",
    "release_id": HYBRID_RELEASE_ID,
    "configuration_sha256": (
        hybrid_configuration_sha256
    ),
    "logical_retrieval_sha256": (
        hybrid_candidate_logical_sha256
    ),
    "inputs": {
        "chunk": {
            "path": (
                "data/phase2/chunks/"
                "nb11_benchmark_v1/"
                "baseline_phase1_char_safe_v1.jsonl"
            ),
            "sha256": (
                "3891c4ecad9332ba753cd605bbccd99da"
                "9d063f64934a30ee39e495349cffb50"
            ),
            "record_count": 3488,
        },
        "golden_dataset": {
            "path": (
                "artifacts/phase2/"
                "golden_dataset_v2/"
                "golden_dataset_v2.jsonl"
            ),
            "sha256": (
                "b9ca18e710686ddca48a25c637840b7"
                "16adb306717cafd8ec7c96ce9e182e6f2"
            ),
            "record_count": 3813,
        },
        "dense_release": {
            "path": (
                "artifacts/phase2/dense_index/"
                "dense_index_release_v1.json"
            ),
            "sha256": (
                "706c020caa2bda77c767fda8cd1b574"
                "98818afdbdbbbdd2a72f898b55ca6ca50"
            ),
        },
        "sparse_release": {
            "path": (
                "artifacts/phase2/sparse_index/"
                "sparse_index_release_v1.json"
            ),
            "sha256": (
                "879f52850a71629f02437c1ab1173e3"
                "f9e6c55ec5441ad4649784be63935c51b"
            ),
        },
    },
    "artifacts": {
        "configuration": {
            "path": str(
                HYBRID_CONFIGURATION_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": sha256_file(
                HYBRID_CONFIGURATION_PATH
            ),
        },
        "adr": {
            "path": str(
                HYBRID_ADR_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": sha256_file(
                HYBRID_ADR_PATH
            ),
        },
        "candidates": {
            "path": str(
                HYBRID_CANDIDATES_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": sha256_file(
                HYBRID_CANDIDATES_PATH
            ),
            "record_count": 3813,
        },
        "test_evaluation": {
            "path": str(
                HYBRID_TEST_EVALUATION_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": sha256_file(
                HYBRID_TEST_EVALUATION_PATH
            ),
        },
        "evidence": {
            "path": str(
                HYBRID_EVIDENCE_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": sha256_file(
                HYBRID_EVIDENCE_PATH
            ),
        },
        "development_query_embeddings": {
            "path": str(
                development_cache_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": sha256_file(
                development_cache_path
            ),
        },
        "test_query_embeddings": {
            "path": str(
                TEST_QUERY_EMBEDDING_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256": sha256_file(
                TEST_QUERY_EMBEDDING_PATH
            ),
        },
    },
    "metrics": {
        "development": (
            weighted_recomputed_metrics
        ),
        "test": test_hybrid_metrics,
    },
    "constraints": {
        "configuration_mutation_allowed": False,
        "dense_index_mutation_allowed": False,
        "sparse_index_mutation_allowed": False,
        "corpus_embedding_recomputation_required": False,
        "test_result_may_change_configuration": False,
    },
    "manifest_status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
}

write_json_atomically(
    HYBRID_MANIFEST_PATH,
    hybrid_manifest,
)

assert HYBRID_MANIFEST_PATH.exists()


hybrid_release = {
    "artifact_type": (
        "NB16_HYBRID_RETRIEVAL_RELEASE"
    ),
    "schema_version": "1.0",
    "release_id": HYBRID_RELEASE_ID,
    "configuration": {
        "path": str(
            HYBRID_CONFIGURATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_CONFIGURATION_PATH
        ),
        "payload_sha256": (
            hybrid_configuration_sha256
        ),
    },
    "manifest": {
        "path": str(
            HYBRID_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_MANIFEST_PATH
        ),
    },
    "candidates": {
        "path": str(
            HYBRID_CANDIDATES_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_CANDIDATES_PATH
        ),
        "logical_sha256": (
            hybrid_candidate_logical_sha256
        ),
        "record_count": 3813,
        "candidate_depth": 100,
    },
    "test_evaluation": {
        "path": str(
            HYBRID_TEST_EVALUATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_TEST_EVALUATION_PATH
        ),
        "query_count": 1149,
        "metrics": test_hybrid_metrics,
    },
    "fusion": (
        selected_weighted_configuration
    ),
    "metadata_filter_status": "PASS",
    "source_diversity_default": (
        "NO_HARD_CAP"
    ),
    "final_status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
    "status": (
        "FROZEN_FOR_RERANKING_PENDING_RELOAD"
    ),
}

write_json_atomically(
    HYBRID_RELEASE_PATH,
    hybrid_release,
)

assert HYBRID_RELEASE_PATH.exists()


nb17_handoff = {
    "artifact_type": (
        "NB17_HYBRID_RETRIEVAL_HANDOFF"
    ),
    "schema_version": "1.0",
    "release_id": HYBRID_RELEASE_ID,
    "hybrid_release": {
        "path": str(
            HYBRID_RELEASE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_RELEASE_PATH
        ),
    },
    "configuration_sha256": (
        hybrid_configuration_sha256
    ),
    "logical_retrieval_sha256": (
        hybrid_candidate_logical_sha256
    ),
    "candidate_path": str(
        HYBRID_CANDIDATES_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "candidate_sha256": sha256_file(
        HYBRID_CANDIDATES_PATH
    ),
    "candidate_record_count": 3813,
    "development_query_count": 2664,
    "test_query_count": 1149,
    "candidate_depth": 100,
    "reranking_scope": {
        "input": (
            "FROZEN_HYBRID_CANDIDATES"
        ),
        "maximum_candidate_depth": 100,
        "hybrid_configuration_mutation_allowed": (
            False
        ),
        "dense_index_mutation_allowed": False,
        "sparse_index_mutation_allowed": False,
        "corpus_embedding_recomputation_allowed": (
            False
        ),
    },
    "status": (
        "READY_FOR_NB17_PENDING_RELOAD"
    ),
}

write_json_atomically(
    NB17_HANDOFF_PATH,
    nb17_handoff,
)

assert NB17_HANDOFF_PATH.exists()


hybrid_acceptance = {
    "artifact_type": (
        "NB16_HYBRID_RETRIEVAL_ACCEPTANCE"
    ),
    "schema_version": "1.0",
    "release_id": HYBRID_RELEASE_ID,
    "gates": {
        "canonical_inputs": "PASS",
        "dense_baseline": "PASS",
        "sparse_baseline": "PASS",
        "rrf_fusion": "PASS",
        "weighted_fusion": "PASS",
        "metadata_filtering": "PASS",
        "source_diversity": "PASS",
        "candidate_depth": "PASS",
        "slice_metrics": "PASS",
        "ablation": "PASS",
        "adr_and_configuration_freeze": "PASS",
        "independent_test": "PASS",
        "sparse_nb15_parity": "PASS",
        "artifact_materialization": "PASS",
        "reload_validation": (
            "PENDING"
        ),
    },
    "test_metrics": test_hybrid_metrics,
    "configuration_sha256": (
        hybrid_configuration_sha256
    ),
    "logical_retrieval_sha256": (
        hybrid_candidate_logical_sha256
    ),
    "manifest_sha256": sha256_file(
        HYBRID_MANIFEST_PATH
    ),
    "release_sha256": sha256_file(
        HYBRID_RELEASE_PATH
    ),
    "nb17_handoff_sha256": sha256_file(
        NB17_HANDOFF_PATH
    ),
    "known_limitations": (
        frozen_adr["known_limitations"]
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "paid_api_calls": 0,
    "acceptance_status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
}

write_json_atomically(
    HYBRID_ACCEPTANCE_PATH,
    hybrid_acceptance,
)

assert HYBRID_ACCEPTANCE_PATH.exists()


artifact_write_result = {
    "release_id": HYBRID_RELEASE_ID,
    "candidate_records": {
        "total": 3813,
        "development": 2664,
        "test": 1149,
        "candidate_depth": 100,
        "logical_sha256": (
            hybrid_candidate_logical_sha256
        ),
        "path": str(
            HYBRID_CANDIDATES_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "file_sha256": sha256_file(
            HYBRID_CANDIDATES_PATH
        ),
    },
    "configuration": {
        "path": str(
            HYBRID_CONFIGURATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_CONFIGURATION_PATH
        ),
    },
    "adr": {
        "path": str(
            HYBRID_ADR_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_ADR_PATH
        ),
    },
    "evidence": {
        "path": str(
            HYBRID_EVIDENCE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_EVIDENCE_PATH
        ),
    },
    "manifest": {
        "path": str(
            HYBRID_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_MANIFEST_PATH
        ),
    },
    "release": {
        "path": str(
            HYBRID_RELEASE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_RELEASE_PATH
        ),
        "status": (
            "FROZEN_FOR_RERANKING_PENDING_RELOAD"
        ),
    },
    "nb17_handoff": {
        "path": str(
            NB17_HANDOFF_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            NB17_HANDOFF_PATH
        ),
        "status": (
            "READY_FOR_NB17_PENDING_RELOAD"
        ),
    },
    "acceptance": {
        "path": str(
            HYBRID_ACCEPTANCE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_ACCEPTANCE_PATH
        ),
        "status": (
            "PASS_PENDING_RELOAD_VALIDATION"
        ),
    },
    "configuration_changed_after_test": False,
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "paid_api_calls": 0,
    "artifact_write_status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
}

print(
    json.dumps(
        artifact_write_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_hybrid_v1",
  "candidate_records": {
    "total": 3813,
    "development": 2664,
    "test": 1149,
    "candidate_depth": 100,
    "logical_sha256": "f54277680f3d20681b1585915f54bb4485ecb1fcccb15c27df0061ff0725d19a",
    "path": "data/phase2/retrieval/hybrid_retrieval_v1/hybrid_candidates_v1.jsonl",
    "file_sha256": "f54277680f3d20681b1585915f54bb4485ecb1fcccb15c27df0061ff0725d19a"
  },
  "configuration": {
    "path": "artifacts/phase2/hybrid_retrieval/hybrid_configuration_v1.json",
    "sha256": "e7dc28836b0e6b2f72cf19aa61507af41a76df4980b056f39af015ed91ba26d5"
  },
  "adr": {
    "path": "reports/phase2/hybrid_retrieval_adr_v1.json",
    "sha256": "daea54cb618ab5cfc25934b785cf7e960650e92e8a6440fabda3b496265f8418"
  },
  "evidence": {
    "path": "data/phase2/retrieval/hybrid_retrieval_v1/hybrid_retrieval_evidence_v1.json",
    "sha256": "1ec23c3f678a30326d1a25350fecaf9188024ea8a6c1f1f5a01ee4d85219feca"
  },
  "manifest": {
    "path": "data/ph

<div dir="rtl" align="right">

## Reload Validation و Finalization Artifactهای Hybrid

Artifactهای Hybrid نباید فقط در Kernel فعلی معتبر باشند. در این مرحله فایل Candidateها، Configuration، Manifest و گزارش Test مستقیماً از Disk بارگذاری و دوباره بررسی می‌شوند.

کنترل‌های این مرحله شامل موارد زیر است:

1. اعتبار تمام SHAهای ثبت‌شده؛
2. بارگذاری ۳۸۱۳ Candidate Record؛
3. برابری کامل شناسه Queryها و Splitها؛
4. کنترل Rank، Score، Chunk ID و Candidate Depth؛
5. بازسازی Metrics از Artifact ذخیره‌شده؛
6. برابری Metrics بازسازی‌شده با Development و Test؛
7. تأیید عدم تغییر Configuration؛
8. نهایی‌سازی Manifest، Release، Acceptance و Handoff برای NB17.

پس از عبور این مرحله، Hybrid Retrieval به‌صورت نهایی Frozen و آماده NB17 خواهد بود.

</div>

In [21]:
# Code Cell 21 — Independent reload validation and finalization

HYBRID_RELOAD_VALIDATION_PATH = (
    HYBRID_DATA_ROOT
    / "hybrid_reload_validation_v1.json"
)

TOP_K_PARITY_METRICS = (
    "hit_at_10",
    "hit_at_5",
    "ndcg_at_10",
    "ndcg_at_5",
    "precision_at_10",
    "precision_at_5",
    "recall_at_10",
    "recall_at_5",
)

assert HYBRID_CONFIGURATION_FROZEN is True
assert TEST_QUERIES_ACCESSED is True


# ------------------------------------------------------------------
# Reload primary artifacts
# ------------------------------------------------------------------

reloaded_configuration = json.loads(
    HYBRID_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_adr = json.loads(
    HYBRID_ADR_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_manifest = json.loads(
    HYBRID_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_release = json.loads(
    HYBRID_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_handoff = json.loads(
    NB17_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_acceptance = json.loads(
    HYBRID_ACCEPTANCE_PATH.read_text(
        encoding="utf-8"
    )
)

reloaded_test_evaluation = json.loads(
    HYBRID_TEST_EVALUATION_PATH.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------------
# Validate configuration and ADR
# ------------------------------------------------------------------

configuration_payload = {
    key: value
    for key, value
    in reloaded_configuration.items()
    if key != "configuration_sha256"
}

adr_payload = {
    key: value
    for key, value
    in reloaded_adr.items()
    if key != "adr_sha256"
}

assert (
    canonical_payload_sha256(
        configuration_payload
    )
    == reloaded_configuration[
        "configuration_sha256"
    ]
    == hybrid_configuration_sha256
)

assert (
    canonical_payload_sha256(
        adr_payload
    )
    == reloaded_adr["adr_sha256"]
    == hybrid_adr_sha256
)

assert (
    reloaded_configuration[
        "configuration_status"
    ] == "FROZEN_BEFORE_TEST_ACCESS"
)

assert (
    reloaded_adr["status"]
    == "ACCEPTED_AND_FROZEN"
)


# ------------------------------------------------------------------
# Validate current manifest inventory
# ------------------------------------------------------------------

for artifact_name, artifact_contract in (
    reloaded_manifest["artifacts"].items()
):
    artifact_path = (
        PROJECT_ROOT
        / artifact_contract["path"]
    )

    assert artifact_path.exists(), (
        artifact_name,
        artifact_path,
    )

    assert (
        sha256_file(artifact_path)
        == artifact_contract["sha256"]
    ), artifact_name


# ------------------------------------------------------------------
# Reload candidate JSONL
# ------------------------------------------------------------------

reloaded_candidate_records = []

with HYBRID_CANDIDATES_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    for line_number, line in enumerate(
        file_handle,
        start=1,
    ):
        stripped_line = line.strip()

        assert stripped_line, line_number

        reloaded_candidate_records.append(
            json.loads(stripped_line)
        )


assert len(
    reloaded_candidate_records
) == 3813

candidate_split = {
    **{
        candidate_id: "development"
        for candidate_id
        in development_candidate_ids_ordered
    },
    **{
        candidate_id: "test"
        for candidate_id
        in test_candidate_ids_ordered
    },
}

expected_candidate_id_set = set(
    candidate_split
)

reloaded_candidate_id_set = {
    str(record["candidate_id"])
    for record
    in reloaded_candidate_records
}

assert (
    reloaded_candidate_id_set
    == expected_candidate_id_set
)

assert len(
    reloaded_candidate_id_set
) == 3813

reloaded_split_counts = (
    collections.Counter(
        record["split"]
        for record
        in reloaded_candidate_records
    )
)

assert reloaded_split_counts == {
    "development": 2664,
    "test": 1149,
}


# ------------------------------------------------------------------
# Validate every candidate record
# ------------------------------------------------------------------

candidate_validation_failures = []
candidate_counts = []

known_chunk_id_set = set(
    chunk_record_by_id
)

for record in reloaded_candidate_records:
    candidate_id = str(
        record["candidate_id"]
    )

    split_name = str(
        record["split"]
    )

    candidates = record["candidates"]

    if (
        split_name
        != candidate_split[candidate_id]
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "SPLIT_MISMATCH",
            )
        )
        continue

    if (
        record["candidate_depth"]
        != FROZEN_CANDIDATE_DEPTH
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "DEPTH_MISMATCH",
            )
        )
        continue

    if (
        record["candidate_count"]
        != len(candidates)
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "COUNT_MISMATCH",
            )
        )
        continue

    if not (
        1
        <= len(candidates)
        <= FROZEN_CANDIDATE_DEPTH
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "INVALID_COUNT",
            )
        )
        continue

    ranks = [
        candidate["rank"]
        for candidate in candidates
    ]

    if ranks != list(
        range(
            1,
            len(candidates) + 1,
        )
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "RANK_MISMATCH",
            )
        )
        continue

    chunk_ids = [
        str(candidate["chunk_id"])
        for candidate in candidates
    ]

    if len(chunk_ids) != len(
        set(chunk_ids)
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "DUPLICATE_CHUNK_ID",
            )
        )
        continue

    if not set(chunk_ids).issubset(
        known_chunk_id_set
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "UNKNOWN_CHUNK_ID",
            )
        )
        continue

    scores = np.asarray(
        [
            candidate["hybrid_score"]
            for candidate in candidates
        ],
        dtype=np.float64,
    )

    if not np.all(
        np.isfinite(scores)
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "NON_FINITE_SCORE",
            )
        )
        continue

    if (
        len(scores) > 1
        and not np.all(
            scores[:-1] >= scores[1:]
        )
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "SCORE_ORDER_MISMATCH",
            )
        )
        continue

    expected_query_sha256 = (
        hashlib.sha256(
            all_question_by_candidate_id[
                candidate_id
            ].encode("utf-8")
        ).hexdigest()
    )

    if (
        record["query_sha256"]
        != expected_query_sha256
    ):
        candidate_validation_failures.append(
            (
                candidate_id,
                "QUERY_HASH_MISMATCH",
            )
        )
        continue

    candidate_counts.append(
        len(candidates)
    )


assert not candidate_validation_failures
assert len(candidate_counts) == 3813


# ------------------------------------------------------------------
# Recompute logical candidate hash
# ------------------------------------------------------------------

logical_hasher = hashlib.sha256()

for record in reloaded_candidate_records:
    logical_hasher.update(
        canonical_json_bytes(record)
    )
    logical_hasher.update(b"\n")

reloaded_logical_sha256 = (
    logical_hasher.hexdigest()
)

assert (
    reloaded_logical_sha256
    == hybrid_candidate_logical_sha256
)

assert (
    reloaded_logical_sha256
    == reloaded_manifest[
        "logical_retrieval_sha256"
    ]
)

assert (
    reloaded_logical_sha256
    == reloaded_release[
        "candidates"
    ]["logical_sha256"]
)

assert (
    reloaded_logical_sha256
    == reloaded_handoff[
        "logical_retrieval_sha256"
    ]
)


# ------------------------------------------------------------------
# Recompute metrics from persisted Top-100 candidates
# ------------------------------------------------------------------

reloaded_metric_rows = {
    "development": [],
    "test": [],
}

for record in reloaded_candidate_records:
    candidate_id = str(
        record["candidate_id"]
    )

    ranked_chunk_ids = [
        str(candidate["chunk_id"])
        for candidate
        in record["candidates"]
    ]

    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    metrics = evaluate_ranked_ids(
        ranked_chunk_ids,
        relevant_chunk_ids,
    )

    reloaded_metric_rows[
        record["split"]
    ].append(
        {
            "candidate_id": candidate_id,
            **metrics,
        }
    )


def aggregate_reloaded_metrics(rows):
    return {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row in rows
                ]
            )
        )
        for metric_name in METRIC_NAMES
    }


reloaded_development_metrics = (
    aggregate_reloaded_metrics(
        reloaded_metric_rows[
            "development"
        ]
    )
)

reloaded_test_metrics = (
    aggregate_reloaded_metrics(
        reloaded_metric_rows["test"]
    )
)


# ------------------------------------------------------------------
# Exact parity applies to Top-5 and Top-10 metrics
# MRR scope differs because persisted candidates stop at rank 100
# ------------------------------------------------------------------

development_top_k_deltas = {
    metric_name: float(
        reloaded_development_metrics[
            metric_name
        ]
        - reloaded_manifest["metrics"][
            "development"
        ][metric_name]
    )
    for metric_name
    in TOP_K_PARITY_METRICS
}

test_top_k_deltas = {
    metric_name: float(
        reloaded_test_metrics[
            metric_name
        ]
        - reloaded_manifest["metrics"][
            "test"
        ][metric_name]
    )
    for metric_name
    in TOP_K_PARITY_METRICS
}

assert all(
    abs(delta) <= 1e-15
    for delta
    in development_top_k_deltas.values()
)

assert all(
    abs(delta) <= 1e-15
    for delta
    in test_top_k_deltas.values()
)

for metric_name in TOP_K_PARITY_METRICS:
    assert np.isclose(
        reloaded_test_metrics[
            metric_name
        ],
        reloaded_test_evaluation[
            "metrics"
        ]["hybrid"][metric_name],
        rtol=0.0,
        atol=1e-15,
    )


development_full_ranking_mrr = float(
    reloaded_manifest["metrics"][
        "development"
    ]["reciprocal_rank"]
)

test_full_ranking_mrr = float(
    reloaded_manifest["metrics"][
        "test"
    ]["reciprocal_rank"]
)

development_candidate_100_mrr = float(
    reloaded_development_metrics[
        "reciprocal_rank"
    ]
)

test_candidate_100_mrr = float(
    reloaded_test_metrics[
        "reciprocal_rank"
    ]
)

development_mrr_scope_delta = float(
    development_candidate_100_mrr
    - development_full_ranking_mrr
)

test_mrr_scope_delta = float(
    test_candidate_100_mrr
    - test_full_ranking_mrr
)

assert development_candidate_100_mrr <= (
    development_full_ranking_mrr
    + 1e-15
)

assert test_candidate_100_mrr <= (
    test_full_ranking_mrr
    + 1e-15
)


# ------------------------------------------------------------------
# Reload validation evidence
# ------------------------------------------------------------------

hybrid_reload_validation = {
    "artifact_type": (
        "NB16_HYBRID_RELOAD_VALIDATION"
    ),
    "schema_version": "1.0",
    "release_id": HYBRID_RELEASE_ID,
    "configuration_sha256": (
        hybrid_configuration_sha256
    ),
    "logical_retrieval_sha256": (
        reloaded_logical_sha256
    ),
    "file_hash_validation": "PASS",
    "candidate_validation": {
        "record_count": 3813,
        "development_record_count": 2664,
        "test_record_count": 1149,
        "candidate_depth": 100,
        "minimum_candidate_count": int(
            min(candidate_counts)
        ),
        "maximum_candidate_count": int(
            max(candidate_counts)
        ),
        "id_uniqueness": "PASS",
        "split_parity": "PASS",
        "rank_validation": "PASS",
        "score_validation": "PASS",
        "chunk_id_validation": "PASS",
        "query_hash_validation": "PASS",
    },
    "metric_validation": {
        "top_k_metric_scope": (
            "EXACT_TOP_5_AND_TOP_10_PARITY"
        ),
        "development": {
            "reloaded_metrics": (
                reloaded_development_metrics
            ),
            "top_k_metric_deltas": (
                development_top_k_deltas
            ),
            "maximum_top_k_absolute_delta": (
                float(
                    max(
                        abs(delta)
                        for delta
                        in development_top_k_deltas.values()
                    )
                )
            ),
            "status": "PASS",
        },
        "test": {
            "reloaded_metrics": (
                reloaded_test_metrics
            ),
            "top_k_metric_deltas": (
                test_top_k_deltas
            ),
            "maximum_top_k_absolute_delta": (
                float(
                    max(
                        abs(delta)
                        for delta
                        in test_top_k_deltas.values()
                    )
                )
            ),
            "status": "PASS",
        },
        "mrr_scope": {
            "selection_evaluation": (
                "FULL_POSITIVE_FUSED_RANKING"
            ),
            "persisted_candidate_evaluation": (
                "CANDIDATE_DEPTH_100"
            ),
            "development_full_ranking_mrr": (
                development_full_ranking_mrr
            ),
            "development_candidate_100_mrr": (
                development_candidate_100_mrr
            ),
            "development_delta": (
                development_mrr_scope_delta
            ),
            "test_full_ranking_mrr": (
                test_full_ranking_mrr
            ),
            "test_candidate_100_mrr": (
                test_candidate_100_mrr
            ),
            "test_delta": (
                test_mrr_scope_delta
            ),
            "reason": (
                "Persisted NB17 handoff contains only "
                "the frozen Top-100 candidate pool."
            ),
            "status": "PASS_DOCUMENTED_SCOPE",
        },
    },
    "configuration_changed": False,
    "candidate_artifact_changed": False,
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "paid_api_calls": 0,
    "reload_validation_status": "PASS",
}

write_json_atomically(
    HYBRID_RELOAD_VALIDATION_PATH,
    hybrid_reload_validation,
)

assert (
    HYBRID_RELOAD_VALIDATION_PATH.exists()
)


# ------------------------------------------------------------------
# Finalize manifest
# ------------------------------------------------------------------

reloaded_manifest[
    "artifacts"
]["reload_validation"] = {
    "path": str(
        HYBRID_RELOAD_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "sha256": sha256_file(
        HYBRID_RELOAD_VALIDATION_PATH
    ),
}

reloaded_manifest[
    "metric_scope"
] = {
    "top_k_metrics": (
        "EXACTLY_REPRODUCIBLE_FROM_PERSISTED_TOP_100"
    ),
    "reciprocal_rank_selection_scope": (
        "FULL_POSITIVE_FUSED_RANKING"
    ),
    "reciprocal_rank_handoff_scope": (
        "PERSISTED_TOP_100"
    ),
}

reloaded_manifest[
    "manifest_status"
] = "PASS_COMPLETE_FROZEN"

write_json_atomically(
    HYBRID_MANIFEST_PATH,
    reloaded_manifest,
)

final_manifest_sha256 = sha256_file(
    HYBRID_MANIFEST_PATH
)


# ------------------------------------------------------------------
# Finalize release
# ------------------------------------------------------------------

reloaded_release["manifest"] = {
    "path": str(
        HYBRID_MANIFEST_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "sha256": final_manifest_sha256,
}

reloaded_release[
    "reload_validation"
] = {
    "path": str(
        HYBRID_RELOAD_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "sha256": sha256_file(
        HYBRID_RELOAD_VALIDATION_PATH
    ),
    "status": "PASS",
}

reloaded_release[
    "metric_scope"
] = reloaded_manifest[
    "metric_scope"
]

reloaded_release[
    "final_status"
] = "PASS_COMPLETE_FROZEN"

reloaded_release[
    "status"
] = "FROZEN_FOR_RERANKING"

write_json_atomically(
    HYBRID_RELEASE_PATH,
    reloaded_release,
)

final_release_sha256 = sha256_file(
    HYBRID_RELEASE_PATH
)


# ------------------------------------------------------------------
# Finalize NB17 handoff
# ------------------------------------------------------------------

reloaded_handoff["hybrid_release"] = {
    "path": str(
        HYBRID_RELEASE_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "sha256": final_release_sha256,
}

reloaded_handoff[
    "reload_validation"
] = {
    "path": str(
        HYBRID_RELOAD_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "sha256": sha256_file(
        HYBRID_RELOAD_VALIDATION_PATH
    ),
    "status": "PASS",
}

reloaded_handoff[
    "metric_scope"
] = {
    "candidate_pool": "TOP_100",
    "reranking_input": (
        "PERSISTED_FROZEN_TOP_100"
    ),
}

reloaded_handoff[
    "status"
] = "READY_FOR_NB17"

write_json_atomically(
    NB17_HANDOFF_PATH,
    reloaded_handoff,
)

final_handoff_sha256 = sha256_file(
    NB17_HANDOFF_PATH
)


# ------------------------------------------------------------------
# Finalize acceptance
# ------------------------------------------------------------------

reloaded_acceptance[
    "gates"
]["reload_validation"] = "PASS"

reloaded_acceptance[
    "manifest_sha256"
] = final_manifest_sha256

reloaded_acceptance[
    "release_sha256"
] = final_release_sha256

reloaded_acceptance[
    "nb17_handoff_sha256"
] = final_handoff_sha256

reloaded_acceptance[
    "reload_validation"
] = {
    "path": str(
        HYBRID_RELOAD_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "sha256": sha256_file(
        HYBRID_RELOAD_VALIDATION_PATH
    ),
}

reloaded_acceptance[
    "metric_scope"
] = reloaded_manifest[
    "metric_scope"
]

reloaded_acceptance[
    "acceptance_status"
] = "PASS_COMPLETE_FROZEN"

write_json_atomically(
    HYBRID_ACCEPTANCE_PATH,
    reloaded_acceptance,
)

final_acceptance_sha256 = sha256_file(
    HYBRID_ACCEPTANCE_PATH
)


# ------------------------------------------------------------------
# Final cross-artifact validation
# ------------------------------------------------------------------

final_manifest = json.loads(
    HYBRID_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

final_release = json.loads(
    HYBRID_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

final_handoff = json.loads(
    NB17_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

final_acceptance = json.loads(
    HYBRID_ACCEPTANCE_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    final_manifest["manifest_status"]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    final_release["status"]
    == "FROZEN_FOR_RERANKING"
)

assert (
    final_release["final_status"]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    final_release["manifest"]["sha256"]
    == final_manifest_sha256
)

assert (
    final_handoff["status"]
    == "READY_FOR_NB17"
)

assert (
    final_handoff[
        "hybrid_release"
    ]["sha256"]
    == final_release_sha256
)

assert (
    final_acceptance[
        "acceptance_status"
    ] == "PASS_COMPLETE_FROZEN"
)

assert (
    final_acceptance[
        "manifest_sha256"
    ] == final_manifest_sha256
)

assert (
    final_acceptance[
        "release_sha256"
    ] == final_release_sha256
)

assert (
    final_acceptance[
        "nb17_handoff_sha256"
    ] == final_handoff_sha256
)


reload_finalization_result = {
    "release_id": HYBRID_RELEASE_ID,
    "configuration_sha256": (
        hybrid_configuration_sha256
    ),
    "logical_retrieval_sha256": (
        reloaded_logical_sha256
    ),
    "candidate_record_count": 3813,
    "top_k_metric_parity": "PASS",
    "mrr_scope": {
        "development_full_ranking": (
            development_full_ranking_mrr
        ),
        "development_top_100": (
            development_candidate_100_mrr
        ),
        "test_full_ranking": (
            test_full_ranking_mrr
        ),
        "test_top_100": (
            test_candidate_100_mrr
        ),
        "status": "PASS_DOCUMENTED_SCOPE",
    },
    "reload_validation": {
        "path": str(
            HYBRID_RELOAD_VALIDATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": sha256_file(
            HYBRID_RELOAD_VALIDATION_PATH
        ),
        "status": "PASS",
    },
    "manifest": {
        "path": str(
            HYBRID_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": final_manifest_sha256,
        "status": "PASS_COMPLETE_FROZEN",
    },
    "release": {
        "path": str(
            HYBRID_RELEASE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": final_release_sha256,
        "status": "FROZEN_FOR_RERANKING",
    },
    "nb17_handoff": {
        "path": str(
            NB17_HANDOFF_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": final_handoff_sha256,
        "status": "READY_FOR_NB17",
    },
    "acceptance": {
        "path": str(
            HYBRID_ACCEPTANCE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": final_acceptance_sha256,
        "status": "PASS_COMPLETE_FROZEN",
    },
    "configuration_changed": False,
    "candidate_artifact_changed": False,
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "paid_api_calls": 0,
    "finalization_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

print(
    json.dumps(
        reload_finalization_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_hybrid_v1",
  "configuration_sha256": "1f618f4870622a6d31e831b54a35d76c96687d6683ab195af171b36214abcbf7",
  "logical_retrieval_sha256": "f54277680f3d20681b1585915f54bb4485ecb1fcccb15c27df0061ff0725d19a",
  "candidate_record_count": 3813,
  "top_k_metric_parity": "PASS",
  "mrr_scope": {
    "development_full_ranking": 0.6083461459951199,
    "development_top_100": 0.6082169141249909,
    "test_full_ranking": 0.5900985040571087,
    "test_top_100": 0.5899166127502804,
    "status": "PASS_DOCUMENTED_SCOPE"
  },
  "reload_validation": {
    "path": "data/phase2/retrieval/hybrid_retrieval_v1/hybrid_reload_validation_v1.json",
    "sha256": "1d85c12d0fed3b6b217741bb92690f8a1fd6238efe33e9d5f8312676631ee9ba",
    "status": "PASS"
  },
  "manifest": {
    "path": "data/phase2/retrieval/hybrid_retrieval_v1/hybrid_retrieval_manifest.json",
    "sha256": "362b9cbfd33e2f8e307ffc6cc7dcffc48d2cc308fe74f9bb0ba896d1e080032c",
    "status": "PASS_COMPLETE_FROZEN"
 

<div dir="rtl" align="right">

## Final Health Check نوت‌بوک NB16

در آخرین کنترل اجرایی، ساختار خود Notebook و تمام Artifactهای نهایی بررسی می‌شوند.

این کنترل شامل موارد زیر است:

- ترتیب کامل Code Cellهای ۰۱ تا ۲۲؛
- نبود Syntax Error؛
- نبود سل کد اجرا‌نشده؛
- نبود Error Output باقی‌مانده؛
- صحت قالب RTL تمام Markdownها؛
- صحت SHA و وضعیت Configuration، Manifest، Release، Acceptance و Handoff؛
- تکمیل تمام ده مأموریت نقشه ماژولار NB16؛
- عدم تغییر Dense و Sparse Index؛
- عدم محاسبه مجدد Corpus Embedding؛
- آماده‌بودن ورودی نسخه‌دار NB17.

این سل باید پس از Save Notebook اجرا شود. سپس برای تأیید نهایی قابلیت بازتولید، Notebook با `Restart Kernel → Run All` اجرا خواهد شد.

</div>

In [22]:
# Code Cell 22 — NB16 final health check

import ast
import re

NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "notebooks/phase2"
    / "16_hybrid_retrieval.ipynb"
)

FINAL_HEALTH_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/phase2"
    / "hybrid_retrieval_final_health_v1.json"
)

assert NOTEBOOK_PATH.exists()

notebook_payload = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)

notebook_cells = notebook_payload[
    "cells"
]

code_cells = [
    cell
    for cell in notebook_cells
    if cell.get("cell_type") == "code"
]

markdown_cells = [
    cell
    for cell in notebook_cells
    if cell.get("cell_type") == "markdown"
]

assert len(code_cells) == 22


# ------------------------------------------------------------------
# Validate Code Cell numbering
# ------------------------------------------------------------------

code_cell_numbers = []
missing_code_cell_headers = []

for physical_index, cell in enumerate(
    code_cells,
    start=1,
):
    source = "".join(
        cell.get("source", [])
    )

    match = re.search(
        r"^\s*#\s*Code Cell\s+(\d+)\b",
        source,
        flags=re.MULTILINE,
    )

    if match is None:
        missing_code_cell_headers.append(
            physical_index
        )
        continue

    code_cell_numbers.append(
        int(match.group(1))
    )

assert not missing_code_cell_headers

assert code_cell_numbers == list(
    range(1, 23)
)

assert len(code_cell_numbers) == len(
    set(code_cell_numbers)
)


# ------------------------------------------------------------------
# Validate syntax
# ------------------------------------------------------------------

syntax_errors = []

for code_number, cell in zip(
    code_cell_numbers,
    code_cells,
):
    source = "".join(
        cell.get("source", [])
    )

    try:
        ast.parse(source)
    except SyntaxError as error:
        syntax_errors.append(
            {
                "code_cell": code_number,
                "line": error.lineno,
                "message": error.msg,
            }
        )

assert not syntax_errors, syntax_errors


# ------------------------------------------------------------------
# Validate execution state and error outputs
# ------------------------------------------------------------------

unexecuted_code_cells = []
error_outputs = []

for code_number, cell in zip(
    code_cell_numbers,
    code_cells,
):
    execution_count = cell.get(
        "execution_count"
    )

    if (
        execution_count is None
        and code_number != 22
    ):
        unexecuted_code_cells.append(
            code_number
        )

    for output in cell.get(
        "outputs",
        [],
    ):
        if output.get("output_type") == "error" and code_number != 22:
            error_outputs.append(
                {
                    "code_cell": code_number,
                    "ename": output.get(
                        "ename"
                    ),
                    "evalue": output.get(
                        "evalue"
                    ),
                }
            )

assert not unexecuted_code_cells, (
    unexecuted_code_cells
)

assert not error_outputs, error_outputs


# ------------------------------------------------------------------
# Validate RTL Markdown contract
# ------------------------------------------------------------------

invalid_rtl_markdown_cells = []

for markdown_index, cell in enumerate(
    markdown_cells,
    start=1,
):
    source = "".join(
        cell.get("source", [])
    ).strip()

    if not source:
        invalid_rtl_markdown_cells.append(
            {
                "markdown_cell": (
                    markdown_index
                ),
                "reason": "EMPTY_MARKDOWN",
            }
        )
        continue

    if not source.startswith(
        '<div dir="rtl" align="right">'
    ):
        invalid_rtl_markdown_cells.append(
            {
                "markdown_cell": (
                    markdown_index
                ),
                "reason": (
                    "MISSING_RTL_OPENING"
                ),
            }
        )

    if not source.endswith("</div>"):
        invalid_rtl_markdown_cells.append(
            {
                "markdown_cell": (
                    markdown_index
                ),
                "reason": (
                    "MISSING_RTL_CLOSING"
                ),
            }
        )

assert not invalid_rtl_markdown_cells, (
    invalid_rtl_markdown_cells
)


# ------------------------------------------------------------------
# Reload finalized artifacts
# ------------------------------------------------------------------

final_configuration = json.loads(
    HYBRID_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

final_manifest = json.loads(
    HYBRID_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

final_release = json.loads(
    HYBRID_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

final_handoff = json.loads(
    NB17_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

final_acceptance = json.loads(
    HYBRID_ACCEPTANCE_PATH.read_text(
        encoding="utf-8"
    )
)

final_reload_validation = json.loads(
    HYBRID_RELOAD_VALIDATION_PATH.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------------
# Validate configuration integrity
# ------------------------------------------------------------------

final_configuration_payload = {
    key: value
    for key, value
    in final_configuration.items()
    if key != "configuration_sha256"
}

assert (
    canonical_payload_sha256(
        final_configuration_payload
    )
    == final_configuration[
        "configuration_sha256"
    ]
    == hybrid_configuration_sha256
)

assert (
    final_configuration[
        "configuration_status"
    ] == "FROZEN_BEFORE_TEST_ACCESS"
)


# ------------------------------------------------------------------
# Validate finalized statuses
# ------------------------------------------------------------------

assert (
    final_manifest["manifest_status"]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    final_release["status"]
    == "FROZEN_FOR_RERANKING"
)

assert (
    final_release["final_status"]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    final_handoff["status"]
    == "READY_FOR_NB17"
)

assert (
    final_acceptance[
        "acceptance_status"
    ] == "PASS_COMPLETE_FROZEN"
)

assert (
    final_reload_validation[
        "reload_validation_status"
    ] == "PASS"
)


# ------------------------------------------------------------------
# Validate finalized hashes and cross-references
# ------------------------------------------------------------------

final_manifest_sha256 = sha256_file(
    HYBRID_MANIFEST_PATH
)

final_release_sha256 = sha256_file(
    HYBRID_RELEASE_PATH
)

final_handoff_sha256 = sha256_file(
    NB17_HANDOFF_PATH
)

final_acceptance_sha256 = sha256_file(
    HYBRID_ACCEPTANCE_PATH
)

final_reload_validation_sha256 = (
    sha256_file(
        HYBRID_RELOAD_VALIDATION_PATH
    )
)

final_candidate_sha256 = sha256_file(
    HYBRID_CANDIDATES_PATH
)

assert (
    final_release["manifest"]["sha256"]
    == final_manifest_sha256
)

assert (
    final_release["candidates"]["sha256"]
    == final_candidate_sha256
)

assert (
    final_release["candidates"][
        "logical_sha256"
    ] == hybrid_candidate_logical_sha256
)

assert (
    final_handoff[
        "hybrid_release"
    ]["sha256"]
    == final_release_sha256
)

assert (
    final_handoff[
        "candidate_sha256"
    ] == final_candidate_sha256
)

assert (
    final_handoff[
        "logical_retrieval_sha256"
    ] == hybrid_candidate_logical_sha256
)

assert (
    final_acceptance[
        "manifest_sha256"
    ] == final_manifest_sha256
)

assert (
    final_acceptance[
        "release_sha256"
    ] == final_release_sha256
)

assert (
    final_acceptance[
        "nb17_handoff_sha256"
    ] == final_handoff_sha256
)

assert (
    final_acceptance[
        "reload_validation"
    ]["sha256"]
    == final_reload_validation_sha256
)


# ------------------------------------------------------------------
# Validate all modular NB16 deliverables
# ------------------------------------------------------------------

required_nb16_gates = {
    "dense_baseline",
    "sparse_baseline",
    "rrf_fusion",
    "weighted_fusion",
    "metadata_filtering",
    "source_diversity",
    "candidate_depth",
    "slice_metrics",
    "ablation",
    "adr_and_configuration_freeze",
    "independent_test",
    "reload_validation",
}

acceptance_gates = (
    final_acceptance["gates"]
)

missing_nb16_gates = sorted(
    required_nb16_gates
    - set(acceptance_gates)
)

failed_nb16_gates = sorted(
    gate_name
    for gate_name
    in required_nb16_gates
    if acceptance_gates.get(
        gate_name
    ) != "PASS"
)

assert not missing_nb16_gates
assert not failed_nb16_gates


# ------------------------------------------------------------------
# Validate immutable upstream systems
# ------------------------------------------------------------------

final_dense_point_count = (
    qdrant_client.get_collection(
        collection_name=(
            DENSE_COLLECTION_NAME
        )
    ).points_count
)

assert final_dense_point_count == 3488

assert (
    final_acceptance[
        "dense_index_mutated"
    ] is False
)

assert (
    final_acceptance[
        "sparse_index_mutated"
    ] is False
)

assert (
    final_acceptance[
        "corpus_embeddings_recomputed"
    ] is False
)

assert (
    final_acceptance[
        "paid_api_calls"
    ] == 0
)


# ------------------------------------------------------------------
# Write final health report
# ------------------------------------------------------------------

nb16_final_health = {
    "notebook": (
        "16_hybrid_retrieval.ipynb"
    ),
    "nb16_objective": "PASS",
    "modular_nb16_deliverables": (
        "PASS"
    ),
    "execution_health": "PASS",
    "syntax_health": "PASS",
    "markdown_rtl_health": "PASS",
    "code_cell_count": len(code_cells),
    "code_cell_numbering": (
        "01_TO_22_PASS"
    ),
    "canonical_input_hashes": "PASS",
    "dense_baseline_gate": "PASS",
    "sparse_baseline_gate": "PASS",
    "rrf_gate": "PASS",
    "weighted_fusion_gate": "PASS",
    "metadata_filter_gate": "PASS",
    "source_diversity_gate": "PASS",
    "candidate_depth_gate": "PASS",
    "slice_metrics_gate": "PASS",
    "ablation_gate": "PASS",
    "configuration_freeze_gate": "PASS",
    "independent_test_gate": "PASS",
    "reload_validation_gate": "PASS",
    "candidate_record_count": 3813,
    "development_query_count": 2664,
    "test_query_count": 1149,
    "candidate_depth": 100,
    "selected_configuration": (
        final_configuration["hybrid"]
    ),
    "development_metrics": (
        final_manifest["metrics"][
            "development"
        ]
    ),
    "test_metrics": (
        final_manifest["metrics"]["test"]
    ),
    "configuration_sha256": (
        final_configuration[
            "configuration_sha256"
        ]
    ),
    "logical_retrieval_sha256": (
        hybrid_candidate_logical_sha256
    ),
    "manifest": {
        "path": str(
            HYBRID_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": final_manifest_sha256,
        "status": (
            final_manifest[
                "manifest_status"
            ]
        ),
    },
    "release": {
        "path": str(
            HYBRID_RELEASE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": final_release_sha256,
        "status": (
            final_release["status"]
        ),
    },
    "nb17_handoff": {
        "path": str(
            NB17_HANDOFF_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": final_handoff_sha256,
        "status": (
            final_handoff["status"]
        ),
    },
    "acceptance": {
        "path": str(
            HYBRID_ACCEPTANCE_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": (
            final_acceptance_sha256
        ),
        "status": (
            final_acceptance[
                "acceptance_status"
            ]
        ),
    },
    "reload_validation": {
        "path": str(
            HYBRID_RELOAD_VALIDATION_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "sha256": (
            final_reload_validation_sha256
        ),
        "status": "PASS",
    },
    "personal_author_identity": (
        "DOCUMENTED_UPSTREAM_METADATA_LIMITATION"
    ),
    "configuration_changed_after_test": (
        False
    ),
    "dense_index_mutated": False,
    "sparse_index_mutated": False,
    "corpus_embeddings_recomputed": False,
    "paid_api_calls": 0,
    "next_execution_notebook": (
        "17_reranking.ipynb"
    ),
    "final_status": (
        "PASS_COMPLETE_RELOAD_VALIDATED_FROZEN"
    ),
}

write_json_atomically(
    FINAL_HEALTH_REPORT_PATH,
    nb16_final_health,
)

assert FINAL_HEALTH_REPORT_PATH.exists()

print(
    json.dumps(
        nb16_final_health,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "\nNB16 FINAL HEALTH: "
    "PASS — COMPLETE / RELOAD-VALIDATED / "
    "FROZEN / READY FOR NB17"
)

{
  "notebook": "16_hybrid_retrieval.ipynb",
  "nb16_objective": "PASS",
  "modular_nb16_deliverables": "PASS",
  "execution_health": "PASS",
  "syntax_health": "PASS",
  "markdown_rtl_health": "PASS",
  "code_cell_count": 22,
  "code_cell_numbering": "01_TO_22_PASS",
  "canonical_input_hashes": "PASS",
  "dense_baseline_gate": "PASS",
  "sparse_baseline_gate": "PASS",
  "rrf_gate": "PASS",
  "weighted_fusion_gate": "PASS",
  "metadata_filter_gate": "PASS",
  "source_diversity_gate": "PASS",
  "candidate_depth_gate": "PASS",
  "slice_metrics_gate": "PASS",
  "ablation_gate": "PASS",
  "configuration_freeze_gate": "PASS",
  "independent_test_gate": "PASS",
  "reload_validation_gate": "PASS",
  "candidate_record_count": 3813,
  "development_query_count": 2664,
  "test_query_count": 1149,
  "candidate_depth": 100,
  "selected_configuration": {
    "candidate_depth": 100,
    "dense_weight": 0.4,
    "deterministic_tie_break": "CHUNK_ID_ASCENDING",
    "fusion_method": "WEIGHTED_SCORE_FUSI

<div dir="rtl" align="right">

# نتیجه نهایی NB16 — Hybrid Retrieval V1

هدف Notebook 16 صرفاً ترکیب خروجی Dense و Sparse نبود؛ هدف، طراحی و انتخاب یک سیستم Hybrid Retrieval نسخه‌دار، قابل‌ارزیابی، قابل‌بازتولید و آماده تحویل به مرحله Reranking بود.

## وضعیت فعلی پروژه

تا این مرحله زنجیره Retrieval پروژه به وضعیت زیر رسیده است:

- `NB14 — Dense Index`: تکمیل، اعتبارسنجی و فریز شده؛
- `NB15 — Sparse Index`: تکمیل، Reload-Validated و فریز شده؛
- `NB16 — Hybrid Retrieval`: تکمیل، Reload-Validated و فریز شده؛
- مرحله بعد: `NB17 — Reranking`.

## ورودی‌های Canonical

NB16 ورودی‌های نسخه‌دار مراحل قبل را بارگذاری و اعتبارسنجی کرد:

- Dense Index شامل ۳۴۸۸ Chunk؛
- Sparse Index شامل ۳۴۸۸ Document؛
- Golden Dataset شامل ۳۸۱۳ Query؛
- Development Split شامل ۲۶۶۴ Query؛
- Test Split مستقل شامل ۱۱۴۹ Query؛
- نگاشت کامل Golden Evidence به Chunkها؛
- برابری کامل شناسه‌های Document میان Dense، Sparse و Corpus Canonical.

هیچ Index قبلی تغییر نکرد و هیچ Corpus Embedding جدیدی محاسبه نشد.

## آزمایش‌های انجام‌شده

تمام مأموریت‌های رسمی نقشه ماژولار NB16 انجام شدند:

1. Dense Baseline اجرا شد.
2. Sparse Baseline اجرا و برابری آن با NB15 تأیید شد.
3. Grid کامل RRF Fusion بررسی شد.
4. Grid کامل Weighted Fusion بررسی شد.
5. Candidate Depthهای ۲۰، ۵۰ و ۱۰۰ مقایسه شدند.
6. Slice Metrics براساس نوع منبع، طول Query و تعداد Chunkهای مرتبط محاسبه شدند.
7. Metadata Filtering بررسی و قرارداد Canonical آن ساخته شد.
8. Source Diversity و محدودیت تعداد Chunk از هر منبع آزمایش شد.
9. Ablation میان Dense، Sparse، RRF و Weighted Hybrid انجام شد.
10. Configuration نهایی با ADR و پیش از دسترسی به Test فریز شد.
11. Test مستقل بدون تغییر Configuration اجرا شد.
12. Candidateهای Hybrid برای NB17 ذخیره شدند.
13. تمام Artifactها از Disk مجدداً Load و اعتبارسنجی شدند.
14. Final Health Check تمام Gateهای Notebook را تأیید کرد.

## تنظیم نهایی انتخاب‌شده

تنظیم فریز‌شده Hybrid Retrieval:

- Fusion Method: `WEIGHTED_SCORE_FUSION`
- Score Normalization: `minmax`
- Candidate Depth: `100`
- Top-K Evaluation: `10`
- Dense Weight: `0.40`
- Sparse Weight: `0.60`
- Sparse Score Policy: فقط امتیازهای مثبت
- Tie Break: مرتب‌سازی قطعی براساس `chunk_id`
- Source Diversity پیش‌فرض: بدون Hard Cap
- Source Diversity اختیاری: حداکثر سه Chunk از هر `source_id`

Hard Cap به‌صورت پیش‌فرض فعال نشد؛ زیرا Retrieval بدون محدودیت، به‌طور طبیعی تنوع منبع بالایی داشت و کیفیت بیشتری ارائه می‌کرد.

## نتیجه Development

نتایج تنظیم منتخب روی ۲۶۶۴ Query توسعه:

- `Hit@10`: 0.78566
- `Hit@5`: 0.71171
- `nDCG@10`: 0.53817
- `nDCG@5`: 0.51156
- `Recall@10`: 0.62124
- `Recall@5`: 0.54690
- `MRR`: 0.60835

Weighted Hybrid در معیار اصلی `nDCG@10` از Dense، Sparse و بهترین RRF بهتر بود.

Bootstrap زوجی نیز نشان داد بهبود نسبت به Sparse و RRF از نظر آماری معتبر است و بازه اطمینان ۹۵ درصد اختلاف، کاملاً بالاتر از صفر قرار دارد.

## نتیجه مستقل Test

پس از Freeze کامل Configuration، ارزیابی روی ۱۱۴۹ Query مستقل Test انجام شد:

- `Hit@10`: 0.75544
- `Hit@5`: 0.69626
- `nDCG@10`: 0.52167
- `nDCG@5`: 0.49834
- `Recall@10`: 0.59986
- `Recall@5`: 0.53366
- `MRR`: 0.59010

بهبود Hybrid نسبت به Sparse روی Test:

- `Hit@10`: +0.04787
- `Hit@5`: +0.04439
- `nDCG@10`: +0.03146
- `nDCG@5`: +0.02954
- `Recall@10`: +0.04628
- `MRR`: +0.02641

بازه اطمینان ۹۵ درصد اختلاف `nDCG@10` میان Hybrid و Sparse کاملاً مثبت بود. بنابراین بهبود مشاهده‌شده فقط محدود به Development نیست و روی Test مستقل نیز تأیید شده است.

## نتیجه Sliceهای منبع

Hybrid Retrieval در هر سه Source Type نسبت به Sparse بهبود داشت:

- Book: `nDCG@10 = 0.63027`
- Transcript: `nDCG@10 = 0.43926`
- Website: `nDCG@10 = 0.61917`

بنابراین بهبود Hybrid فقط ناشی از یک نوع منبع خاص نیست.

## Metadata Filtering

فیلترهای زیر با موفقیت ساخته و آزمایش شدند:

- `source_id`
- `source_type`
- Speaker Identity
- Speaker Attribution Status
- Primary Speaker Signal
- Author Attribution Mode
- Content Origin
- Rights Status
- Rights Basis
- Inferred Release Permission

تمام Filter Contract Testها بدون Mismatch عبور کردند.

### محدودیت ثبت‌شده Metadata

نام شخص Author در Corpus Canonical موجود نیست و هیچ مقدار ساختگی یا استنباطی تولید نشد.

برای Website، فیلتر نویسندگی براساس نوع انتساب محتوا پشتیبانی می‌شود:

- `direct_author`
- `editorial_supervision`

همچنین فیلدهای زیر در Schema وجود دارند، اما در نسخه فعلی مقدار قابل‌فیلتر ندارند:

- Rights Holder
- Allowed Uses
- Rights Evidence ID

این موارد به‌عنوان محدودیت Upstream ثبت شده‌اند و شکست Hybrid Retrieval محسوب نمی‌شوند.

## Candidate Artifact برای NB17

برای تمام ۳۸۱۳ Query، Candidateهای Hybrid نسخه‌دار ساخته شدند:

- Development: ۲۶۶۴ رکورد
- Test: ۱۱۴۹ رکورد
- Candidate Depth: حداکثر ۱۰۰
- Rankها: معتبر
- Scoreها: مرتب و متناهی
- Chunk IDها: معتبر و بدون Duplicate در هر Query
- Query Hashها: معتبر
- Logical Retrieval SHA256:

`f54277680f3d20681b1585915f54bb4485ecb1fcccb15c27df0061ff0725d19a`

این Candidateها ورودی مستقیم NB17 هستند و NB17 نباید Hybrid Configuration را تغییر دهد.

## توضیح Scope معیار MRR

معیارهای Top-5 و Top-10 از Candidate Artifact ذخیره‌شده به‌صورت دقیق بازسازی شدند.

MRR زمان انتخاب Configuration روی Ranking کامل Fusion محاسبه شده است، درحالی‌که Artifact تحویلی NB17 عمداً به Top-100 محدود است. این اختلاف Scope به‌طور صریح در Manifest و Reload Validation ثبت شده و هیچ اثری روی برابری معیارهای Top-5 و Top-10 یا انتخاب Configuration ندارد.

## Artifactهای نهایی

### Manifest

- مسیر: `data/phase2/retrieval/hybrid_retrieval_v1/hybrid_retrieval_manifest.json`
- SHA256: `8f9260f52836d1168a16e1237dd0f6c5bf3b1f51afa905cb40d0f4d6b2ff4947`
- وضعیت: `PASS_COMPLETE_FROZEN`

### Release

- مسیر: `artifacts/phase2/hybrid_retrieval/hybrid_retrieval_release_v1.json`
- SHA256: `7675baca475467ed27368711b3ed539f212a4b5a1dec86ebb8b479c4d5a24bcb`
- وضعیت: `FROZEN_FOR_RERANKING`

### NB17 Handoff

- مسیر: `artifacts/phase2/hybrid_retrieval/nb17_hybrid_retrieval_handoff_v1.json`
- SHA256: `cc3ceb1bae7628ce67d6d74dcc6e093cfa238b040fd291cef1735d906ee83259`
- وضعیت: `READY_FOR_NB17`

### Acceptance

- مسیر: `reports/phase2/hybrid_retrieval_acceptance_v1.json`
- SHA256: `8f2742f28b4d0858c91fcbe3297e0d469d1b6d0cd3ab64e30e503b39526f68c0`
- وضعیت: `PASS_COMPLETE_FROZEN`

### Reload Validation

- مسیر: `data/phase2/retrieval/hybrid_retrieval_v1/hybrid_reload_validation_v1.json`
- SHA256: `1d85c12d0fed3b6b217741bb92690f8a1fd6238efe33e9d5f8312676631ee9ba`
- وضعیت: `PASS`

## کنترل‌های ایمنی و یکپارچگی

- Dense Index تغییر نکرد.
- Sparse Index تغییر نکرد.
- Corpus Embedding مجدداً محاسبه نشد.
- Configuration پس از مشاهده Test تغییر نکرد.
- Candidate Artifact پس از Freeze تغییر نکرد.
- تعداد API Call پولی: صفر
- Syntax Health: PASS
- Execution Health: PASS
- Markdown RTL Health: PASS
- Code Cell Numbering: `01_TO_22_PASS`
- Reload Validation: PASS

## وضعیت نهایی NB16

Notebook 16 به هدف رسمی نقشه ماژولار و الزامات Hybrid Retrieval رسیده است.

وضعیت نهایی:

`PASS_COMPLETE_RELOAD_VALIDATED_FROZEN`

NB16 از این مرحله به بعد Frozen است. مراحل بعدی نباید وزن‌ها، Fusion Method، Candidate Depth، Analyzer، Dense Index، Sparse Index یا Candidate Artifact آن را تغییر دهند.

## مرحله بعد

مرحله بعدی پروژه:

`NB17 — 17_reranking.ipynb`

NB17 باید Candidateهای فریز‌شده Top-100 را از Handoff رسمی دریافت کند، Rerankerهای کاندید را روی Development مقایسه کند، تنظیم منتخب را پیش از Test فریز کند و سپس ارزیابی مستقل Test را انجام دهد.

</div>